In [ ]:
# Block 01 — Environment, Paths, Harness & Session Guard
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : none (first block of every session)
#
# WHAT CHANGED IN v7 (driven by the v6 results)
#   v6 (5 seeds each): C0 base 38.59 ± 0.17, C1 person 38.72 ± 0.13 — above
#   Xenos et al. (38.52) but below EmotiCon+CCIM (39.13) as single models;
#   5-seed ensembles reached 39.16 / 39.22. APC lost 4.2 mAP (dropped) and the
#   C2 regularisers lost 0.22 (dropped). Every configuration saturated at
#   test-like val ≈ 42.5. v7 attacks the two things an ensemble fixes and a
#   single model does not — information and label noise — and adds soups:
#     * a SigLIP-2 so400m person-crop stream (new information: the person at
#       384 px from a second, differently trained frozen encoder)
#     * cross-fitted self-distillation: out-of-fold teacher probabilities as
#       soft targets for the single-annotator train labels
#     * shared initialisation across seeds, so seeds can be weight-averaged
#       ("model soup": one network, reported as its own row)
#     * length-bucketed batches (same data, less padding → shorter epochs)
#   Runs live in /kaggle/working/runs_v7/. Downloads go to the scratch disk.
#
# EPOCHS (unchanged from v6): cosine LR over 8 epochs, early stop after 3 flat
# epochs (never before epoch 5). The epoch count is part of a run's identity.
# ══════════════════════════════════════════════════════════════════════════════

import os, sys, gc, json, glob, time, math, random, shutil, warnings, subprocess, logging
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Model downloads (InstructBLIP shards ~10 GB, SigLIP-2 ~4.5 GB) go to the large
# scratch disk, not the container's root filesystem.
if Path("/kaggle/temp").is_dir() and os.environ.get("EMOTIC_TEST_MODE") != "1":
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf_home")
for _lg in ("transformers", "accelerate", "torch", "huggingface_hub", "filelock",
            "PIL", "urllib3", "h5py", "timm", "datasets", "matplotlib"):
    logging.getLogger(_lg).setLevel(logging.ERROR)

import numpy as np
import torch

_G, _R, _Y, _B, _D, _0 = ("\033[92m", "\033[91m", "\033[93m",
                          "\033[94m", "\033[90m", "\033[0m")

# ──────────────────────────────────────────────────────────────────────────────
# 1.0  SESSION CLOCK
# ──────────────────────────────────────────────────────────────────────────────
SESSION_BUDGET_H = float(os.environ.get("EMOTIC_SESSION_BUDGET_H", "11.5"))
EVAL_RESERVE_MIN = 30.0          # kept free after training for Blocks 09-10
_MARKER = Path("/tmp/emotic_session_start.txt")


def _session_start():
    now = time.time()
    cands = {"now": now}
    try:
        import psutil
        cands["kernel process"] = psutil.Process(os.getpid()).create_time()
    except Exception:
        pass
    try:
        cands["marker (/tmp)"] = float(_MARKER.read_text().strip())
    except Exception:
        pass
    valid = {k: v for k, v in cands.items() if now - 13 * 3600 < v <= now}
    start = min(valid.values())
    try:
        if not _MARKER.exists():
            _MARKER.write_text(f"{start:.3f}")
    except Exception:
        pass
    return start, cands


SESSION_START, _SESSION_CANDIDATES = _session_start()


class SessionGuard:
    """Wall-clock budget for the whole Kaggle session."""

    def __init__(self, start, budget_h, reserve_min):
        self.start, self.budget_s, self.reserve_s = start, budget_h * 3600.0, reserve_min * 60.0

    def elapsed_s(self):
        return time.time() - self.start

    def remaining_s(self):
        return max(0.0, self.budget_s - self.elapsed_s())

    def remaining_h(self):
        return self.remaining_s() / 3600.0

    def training_deadline(self):
        """Absolute unix time by which training workers must have paused."""
        return self.start + self.budget_s - self.reserve_s

    def training_remaining_s(self):
        return max(0.0, self.training_deadline() - time.time())

    def footer(self):
        return (f"session left {self.remaining_h():.2f} h | "
                f"elapsed {self.elapsed_s() / 3600:.2f} h")


GUARD = SessionGuard(SESSION_START, SESSION_BUDGET_H, EVAL_RESERVE_MIN)


# ──────────────────────────────────────────────────────────────────────────────
# VERDICT HARNESS (used by every block)
# ──────────────────────────────────────────────────────────────────────────────
def rule(ch="─", w=76):
    print(ch * w)


def header(title, w=76):
    print("\n" + "═" * w)
    print(f"  {title}")
    print("═" * w + "\n")


class BlockVerdict:
    """Per-block check collector. Green = as expected, red = not, yellow = warning."""

    def __init__(self, number, name):
        self.number, self.name, self.rows, self.t0 = number, name, [], time.time()

    def check(self, label, ok, got="", expected="", fatal=True):
        ok = bool(ok)
        self.rows.append((label, ok, fatal))
        mark = f"{_G}✓{_0}" if ok else (f"{_R}✗{_0}" if fatal else f"{_Y}!{_0}")
        detail = (f"{_D}got {got}, expected {expected}{_0}"
                  if (got != "" or expected != "") else "")
        print(f"  {mark} {label:<50s} {detail}")
        return ok

    def note(self, label, value=""):
        print(f"  {_B}·{_0} {label}" + (f": {value}" if value != "" else ""))

    def finish(self):
        header(f"{self.number}.9  VERDICT")
        for label, ok, fatal in self.rows:
            mark = f"{_G}✓{_0}" if ok else (f"{_R}✗{_0}" if fatal else f"{_Y}!{_0}")
            print(f"  {mark} {label}")
        hard = [r for r in self.rows if not r[1] and r[2]]
        n, bar = len(self.rows), "█" * 76
        print()
        tag = f"{self.number:02d}" if isinstance(self.number, int) else str(self.number)
        if hard:
            msg = f"█  BLOCK {tag}: {len(hard)} OF {n} CHECKS FAILED"
            print(f"{_R}{bar}\n{msg}{' ' * max(1, 75 - len(msg))}█\n{bar}{_0}")
        else:
            msg = f"█  BLOCK {tag}: ALL {n} CHECKS PASSED"
            print(f"{_G}{bar}\n{msg}{' ' * max(1, 75 - len(msg))}█\n{bar}{_0}")
        allv = {}
        if VERDICTS_PATH.exists():
            try:
                allv = json.loads(VERDICTS_PATH.read_text())
            except Exception:
                allv = {}
        allv[str(self.number)] = {"name": self.name, "status": "FAIL" if hard else "PASS",
                                  "checks": n, "failed": len(hard),
                                  "ts": time.strftime("%Y-%m-%d %H:%M:%S")}
        try:
            VERDICTS_PATH.write_text(json.dumps(allv, indent=2))
        except Exception:
            pass
        rule()

        def _k(kv):
            s = str(kv[0])
            return (int("".join(ch for ch in s if ch.isdigit()) or 0), s)

        seq = "  ".join(f"{k}:{_G if v['status'] == 'PASS' else _R}{v['status']}{_0}"
                        for k, v in sorted(allv.items(), key=_k))
        print(f"  VERDICT SUMMARY   {seq}")
        rule()
        print(f"  {GUARD.footer()} | block {time.time() - self.t0:.1f} s")
        return not hard


# ──────────────────────────────────────────────────────────────────────────────
# 1.1  PATHS
# ──────────────────────────────────────────────────────────────────────────────
WORK_DIR = Path(os.environ.get("EMOTIC_WORK_DIR", "/kaggle/working"))
WORK_DIR.mkdir(parents=True, exist_ok=True)
VERDICTS_PATH = WORK_DIR / "_verdicts.json"
if VERDICTS_PATH.exists():
    VERDICTS_PATH.unlink()          # fresh summary for this session

header("BLOCK 01 — Environment, Paths, Harness & Session Guard  [v7]")
V = BlockVerdict(1, "Setup")

print("1.0  SESSION CLOCK\n")
for _k, _v in _SESSION_CANDIDATES.items():
    V.note(f"  {_k}", time.strftime("%H:%M:%S", time.localtime(_v)))
V.note("session start used", time.strftime("%H:%M:%S", time.localtime(SESSION_START)))
V.note("budget", f"{SESSION_BUDGET_H:.1f} h total; training must pause by "
                 f"{time.strftime('%H:%M:%S', time.localtime(GUARD.training_deadline()))} "
                 f"(reserving {EVAL_RESERVE_MIN:.0f} min for evaluation)")
V.check("budget within Kaggle's 12 h limit", SESSION_BUDGET_H <= 11.75,
        f"{SESSION_BUDGET_H} h", "<= 11.75 h")
V.check("session has time left", GUARD.remaining_h() > 0.5,
        f"{GUARD.remaining_h():.2f} h", "> 0.5 h")

print("\n1.1  DEPENDENCIES\n")
import scipy, transformers, torchvision          # noqa: E401
V.note("python", sys.version.split()[0])
V.note("torch", torch.__version__)
V.note("torchvision", torchvision.__version__)
V.note("transformers", transformers.__version__)
V.note("model downloads (HF_HOME)", os.environ.get("HF_HOME", "default (~/.cache/huggingface)"))
_tfv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
V.check("transformers >= 4.37", _tfv >= (4, 37), transformers.__version__, ">=4.37")
try:
    import safetensors, psutil                   # noqa: F401
    _have_deps = True
except Exception:
    _have_deps = False
V.check("safetensors + psutil importable", _have_deps, _have_deps, True)

print("\n1.2  DATASET PATHS\n")
DATA_ROOT = Path(os.environ.get("EMOTIC_DATA_ROOT",
                                "/kaggle/input/datasets/jasodhiyayadav/pami-version"))
if not DATA_ROOT.is_dir():
    for cand in sorted(glob.glob("/kaggle/input/**/emotic/emotic", recursive=True)):
        DATA_ROOT = Path(cand).parent.parent
        break
ANN_MAT = DATA_ROOT / "Annotations" / "Annotations" / "Annotations.mat"
IMG_BASE = DATA_ROOT / "emotic" / "emotic"
SOURCE_FOLDERS = ["ade20k", "emodb_small", "framesdb", "mscoco"]
DESC_PATHS = {s: DATA_ROOT / f"{s}_descriptions.json" for s in ("train", "val", "test")}

V.check("data root exists", DATA_ROOT.is_dir(), str(DATA_ROOT), "directory")
V.check("Annotations.mat exists", ANN_MAT.is_file(),
        f"{ANN_MAT.stat().st_size / 1e6:.2f} MB" if ANN_MAT.is_file() else "missing", ">1 MB")
V.check("image base exists", IMG_BASE.is_dir(), str(IMG_BASE), "directory")
_counts = {s: len(list((IMG_BASE / s / "images").glob("*.jpg")))
           for s in SOURCE_FOLDERS if (IMG_BASE / s / "images").is_dir()}
V.check("all 4 source folders present", len(_counts) == 4, len(_counts), 4)
_n_img = sum(_counts.values())
V.check("image count plausible",
        20000 <= _n_img <= 25000 or os.environ.get("EMOTIC_TEST_MODE") == "1",
        f"{_n_img:,}", "~23,185")
for s in SOURCE_FOLDERS:
    V.note(f"  {s}", f"{_counts.get(s, 0):,} images")
V.check("cached description JSONs present", all(p.is_file() for p in DESC_PATHS.values()),
        f"{sum(p.is_file() for p in DESC_PATHS.values())}/3", "3/3")

# ──────────────────────────────────────────────────────────────────────────────
# 1.3  LABEL SPACE
# ──────────────────────────────────────────────────────────────────────────────
EMOTION_LABELS = [
    "Affection", "Anger", "Annoyance", "Anticipation", "Aversion",
    "Confidence", "Disapproval", "Disconnection", "Disquietment",
    "Doubt/Confusion", "Embarrassment", "Engagement", "Esteem",
    "Excitement", "Fatigue", "Fear", "Happiness", "Pain", "Peace",
    "Pleasure", "Sadness", "Sensitivity", "Suffering", "Surprise",
    "Sympathy", "Yearning",
]
NUM_CLASSES = len(EMOTION_LABELS)
EMO2IDX = {e: i for i, e in enumerate(EMOTION_LABELS)}
VAD_DIMS = ["valence", "arousal", "dominance"]

# ──────────────────────────────────────────────────────────────────────────────
# 1.4  DIRECTORIES & GLOBAL CONFIG
# ──────────────────────────────────────────────────────────────────────────────
print("\n1.4  CONFIG\n")
CACHE_DIR = WORK_DIR / "cache"
ARRAY_DIR = CACHE_DIR / "arrays"
RUNS_DIR = WORK_DIR / "runs_v7"           # v7 runs
OLD_RUNS_DIRS = [WORK_DIR / "runs", WORK_DIR / "runs_long", WORK_DIR / "runs_v6"]   # never touched
LOG_DIR = WORK_DIR / "logs"
VIZ_DIR = WORK_DIR / "figures"
CONTROL_DIR = WORK_DIR / "control"
CORE_PATH = WORK_DIR / "emotic_core.py"
for _d in (CACHE_DIR, ARRAY_DIR, RUNS_DIR, LOG_DIR, VIZ_DIR, CONTROL_DIR):
    _d.mkdir(parents=True, exist_ok=True)


# ── EPOCHS & EARLY STOPPING ────────────────────────────────────────────────────
LR_SCHEDULE = "cosine"  # "cosine": decays to 0 over MAX_EPOCHS | "plateau": v5 behaviour
MAX_EPOCHS = 8          # cosine length; v5 runs peaked at epoch 5–7 and then only overfit
PATIENCE = 3            # stop after this many epochs without a val gain > MIN_DELTA
MIN_EPOCHS = 5          # never stop before this epoch
MIN_DELTA = 0.01        # a "gain" is > 0.01 test-like val mAP (below 2-decimal rounding)


class CFG:
    # -- data -----------------------------------------------------------------
    data_root, ann_mat, img_base, desc_paths = DATA_ROOT, ANN_MAT, IMG_BASE, DESC_PATHS
    work_dir, cache_dir, array_dir, runs_dir = WORK_DIR, CACHE_DIR, ARRAY_DIR, RUNS_DIR
    log_dir, viz_dir, control_dir, core_path = LOG_DIR, VIZ_DIR, CONTROL_DIR, CORE_PATH
    seed = 42

    # -- backbone (Xenos et al. footnote: instructblip-flan-t5-xl) -------------
    backbone_ckpt = os.environ.get("EMOTIC_BACKBONE", "Salesforce/instructblip-flan-t5-xl")
    vis_dim, vis_tokens, qformer_dim, num_query_tokens = 1408, 257, 768, 32
    # person-crop view (Block 07): square crop, side = 1.2 x max(box w, h),
    # encoded at 224 px by the same frozen ViT, 2x2-pooled -> 8x8 + CLS tokens
    crop_margin, crop_pool = 1.2, 2
    crop_tokens = (16 // crop_pool) ** 2 + 1

    # -- v7: second frozen encoder for the person crop -------------------------
    # SigLIP-2 so400m/14 at 384 px (27x27 patches) -> pooled token + 9x9 grid.
    # SigLIP (v1) so400m is the fallback if SigLIP-2 cannot be downloaded.
    siglip_repos = [r for r in os.environ.get("EMOTIC_SIGLIP", "").split(",") if r] or \
        ["google/siglip2-so400m-patch14-384", "google/siglip-so400m-patch14-384"]
    sig_grid = 9
    sig_tokens, sig_dim = sig_grid ** 2 + 1, 1152

    # -- inputs ---------------------------------------------------------------
    # Descriptions are kept whole: 224 BERT tokens cut ~20% of them, and the cut
    # removes the closing sentence where LLaVa names the emotions. 320 keeps
    # >97% intact; batches are padded only to their own longest description.
    max_text_len = 320
    augment = False                   # Phase A: no image augmentation anywhere

    # -- v7: cross-fitted self-distillation ----------------------------------
    # Teachers: KD_FOLDS folds x KD_REPEATS repeats, folds grouped by IMAGE (all
    # persons of a photo in the same fold). Soft target for a student:
    #     (1 - KD_ALPHA) * single-annotator label + KD_ALPHA * teacher probability
    kd_folds, kd_repeats, kd_alpha, kd_fold_seed = 5, 2, 0.5, 2026

    # -- training defaults handed to every worker (see emotic_core.DEFAULTS) ---
    train = dict(
        batch_size=32, grad_accum=2, eval_batch_size=64,          # 64 effective (paper)
        lr=1e-4, backbone_lr_mult=0.1, grad_clip=1.0,               # paper
        weight_decay=0.05,                                          # matrices only
        qformer_dropout=0.4, head_dropout=0.1,                    # paper-parity run
        lr_schedule=LR_SCHEDULE, max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
        min_delta=MIN_DELTA, warmup_epochs=1.0,
        plateau_factor=0.5, plateau_patience=4, plateau_min_scale=0.03125, keep_extendable=True,
        floor_check_epoch=2, floor_min_lift=1.5,
        ema_decay=0.998, amp=True, num_workers=2, log_every=185, max_oom_per_epoch=20,
        # v7 recipe additions
        init_seed=0,                    # shared initial weights for every seed (soups)
        bucket=True, bucket_chunk=50,   # length-bucketed batches
        kd_alpha=kd_alpha,
        # components (switched on per configuration in Block 08)
        use_roi=False, use_geo=False, use_crop=False, use_sig=False, use_vad=False,
        vad_weight=0.2, vis_token_drop=0.0, txt_token_drop=0.0, roi_grid=7,
    )

    # -- published EMOTIC test mAP --------------------------------------------
    xenos_test_map = 38.52
    ccim_test_map = 39.13
    # v6 measured test ≈ 0.906 x test-like val (±0.007); used only to colour the
    # live val monitor in Block 08, never as a result
    val_to_test = 0.906


V.note("backbone", CFG.backbone_ckpt)
V.note("text length", f"{CFG.max_text_len} tokens (dynamic padding)")
V.note("training", f"bs {CFG.train['batch_size']} x {CFG.train['grad_accum']} = 64, "
                   f"lr {CFG.train['lr']:g} (Q-Former x{CFG.train['backbone_lr_mult']}), EMA {CFG.train['ema_decay']}, "
                   f"length-bucketed batches, shared init seed {CFG.train['init_seed']}")
V.note("SigLIP stream", f"{CFG.siglip_repos[0]} (fallback {', '.join(CFG.siglip_repos[1:]) or 'none'}), "
                        f"{CFG.sig_tokens} tokens x {CFG.sig_dim}")
V.note("self-distillation", f"{CFG.kd_folds} folds x {CFG.kd_repeats} repeats (grouped by image), "
                            f"soft-target weight a = {CFG.kd_alpha}")
V.note("epochs", f"{LR_SCHEDULE} LR over {MAX_EPOCHS} epochs (1 warm-up), early stop after {PATIENCE} "
                 f"epochs without a gain > {MIN_DELTA} (never before epoch {MIN_EPOCHS})")
V.check("LR schedule is known", LR_SCHEDULE in ("cosine", "plateau"), LR_SCHEDULE, "cosine | plateau")
V.check("epoch budget is valid", MAX_EPOCHS >= 1 and PATIENCE >= 1 and 1 <= MIN_EPOCHS <= MAX_EPOCHS
        and MIN_DELTA >= 0, f"max {MAX_EPOCHS}, patience {PATIENCE}, min {MIN_EPOCHS}", "consistent")
V.note("targets", f"Xenos {CFG.xenos_test_map}  |  CCIM {CFG.ccim_test_map}")
V.check("Phase A: augmentation disabled", CFG.augment is False, CFG.augment, False)
V.check("self-distillation settings are valid", CFG.kd_folds >= 2 and CFG.kd_repeats >= 1
        and 0.0 < CFG.kd_alpha < 1.0, f"{CFG.kd_folds} x {CFG.kd_repeats}, a {CFG.kd_alpha}", ">= 2 folds, 0 < a < 1")
V.check("effective batch matches paper",
        CFG.train["batch_size"] * CFG.train["grad_accum"] == 64,
        CFG.train["batch_size"] * CFG.train["grad_accum"], 64)


# ──────────────────────────────────────────────────────────────────────────────
# 1.5  UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def seed_everything(seed=CFG.seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def host_mem():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f"{(vm.total - vm.available) / 1e9:.1f}/{vm.total / 1e9:.1f}GB"
    except Exception:
        return "n/a"


def disk_free_gb(p):
    try:
        return shutil.disk_usage(str(p)).free / 1e9
    except Exception:
        return 0.0


def dir_size_gb(p):
    p = Path(p)
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9


seed_everything()
N_GPUS = torch.cuda.device_count()

# ──────────────────────────────────────────────────────────────────────────────
# 1.6  HARDWARE & DISK
# ──────────────────────────────────────────────────────────────────────────────
print("\n1.6  HARDWARE & DISK\n")
_vram = 0.0
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    _vram += p.total_memory / 1e9
    V.note(f"  GPU {i}", f"{p.name}  {p.total_memory / 1e9:.1f} GB  sm_{p.major}{p.minor}")
V.check("2 GPUs available (two training workers in parallel)", N_GPUS >= 2, N_GPUS, 2,
        fatal=os.environ.get("EMOTIC_TEST_MODE") != "1")
V.note("host RAM in use", host_mem())
V.note("/kaggle/working free", f"{disk_free_gb(WORK_DIR):.1f} GB")
for _cand in ("/kaggle/temp", "/tmp"):
    if Path(_cand).exists():
        V.note(f"{_cand} free", f"{disk_free_gb(_cand):.1f} GB")

for _od in OLD_RUNS_DIRS:
    if _od.exists():
        V.note("earlier runs (kept, not used)", f"{_od}  {dir_size_gb(_od):.2f} GB")
_legacy = [d for d in (WORK_DIR / "checkpoints",) if d.exists()]
_legacy_gb = sum(dir_size_gb(d) for d in _legacy)
CLEAN_LEGACY = False              # set True to delete old-layout checkpoints
if _legacy:
    V.note("legacy checkpoint dirs", f"{[str(d) for d in _legacy]}  {_legacy_gb:.2f} GB")
    if CLEAN_LEGACY:
        for d in _legacy:
            shutil.rmtree(d, ignore_errors=True)
        V.note("legacy dirs", "deleted")
V.check("/kaggle/working has >= 8 GB free for checkpoints", disk_free_gb(WORK_DIR) >= 8.0,
        f"{disk_free_gb(WORK_DIR):.1f} GB", ">= 8 GB")

(CACHE_DIR / "config.json").write_text(json.dumps(
    {k: str(v) for k, v in vars(CFG).items() if not k.startswith("__")}, indent=2))
V.note("config saved", str(CACHE_DIR / "config.json"))

BLOCK_01_OK = V.finish()


In [ ]:
# Block 02 — Annotation Forensics & Parsing
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Block 01
#
# Unchanged logic from the version that ran all-green (v3).
#
#   EMOTIC's `combined_categories` is the UNION over annotators. Val has 5
#   annotators, test 3, train 1, so labels/person is 1.96 / 6.16 / 4.42 by
#   construction. Every published EMOTIC mAP is computed on these labels.
#
#   2.7 builds TEST-LIKE VAL labels: the union over a seeded 3-annotator subset
#   of each val person's own annotators (test's protocol). Model selection runs
#   on these; the per-annotator sets (`ann_sets`) are kept for Block 03, which
#   measures how often a single annotator misses a label that a 3-annotator
#   union contains.
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio

header("BLOCK 02 — Annotation Forensics & Parsing  [v3]")
V = BlockVerdict(2, "Annotations")

BENCHMARK_DENSITY = {"train": 1.96, "val": 6.16, "test": 4.42}

# ──────────────────────────────────────────────────────────────────────────────
# 2.1  LOAD
# ──────────────────────────────────────────────────────────────────────────────
print("2.1  LOAD\n")
_t0 = time.time()
MAT = sio.loadmat(str(CFG.ann_mat), squeeze_me=True, struct_as_record=False)
_present = [k for k in MAT if not k.startswith("_")]
V.check("all three splits present",
        all(s in MAT for s in ("train", "val", "test")), sorted(_present),
        "train/val/test")
V.note("load time", f"{time.time() - _t0:.1f}s")


# ──────────────────────────────────────────────────────────────────────────────
# 2.2  PRIMITIVES
# ──────────────────────────────────────────────────────────────────────────────
def fields_of(o):
    return list(getattr(o, "_fieldnames", []) or [])


def as_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        if x.dtype == object or x.ndim > 0:
            f = x.flatten()
            if f.size == 0:
                return []
            return [f[0]] if f.size == 1 else list(f)
        return [x.item()]
    return [x]


def to_str(x):
    if isinstance(x, bytes):
        return x.decode("utf-8", "ignore").strip()
    if isinstance(x, (str, np.str_)):
        return str(x).strip()
    if isinstance(x, np.ndarray):
        x = x.squeeze()
        if x.ndim == 0:
            return to_str(x.item())
        if x.size == 1:
            return to_str(x.flat[0])
        return " ".join(to_str(i) for i in x.flatten())
    return str(x).strip()


def to_float(x, default=5.0):
    try:
        a = np.asarray(x, dtype=float).flatten()
        return float(a[0]) if a.size else default
    except Exception:
        return default


def to_bbox(x):
    try:
        a = np.asarray(x, dtype=float).flatten()
        if a.size >= 4:
            x1, y1, x2, y2 = int(a[0]), int(a[1]), int(a[2]), int(a[3])
            return [max(0, x1), max(0, y1), max(x1 + 1, x2), max(y1 + 1, y2)]
    except Exception:
        pass
    return None


def image_hw(entry):
    """(width, height) from entry.image_size {n_row, n_col}."""
    for n in as_list(getattr(entry, "image_size", None)):
        if n is None:
            continue
        if hasattr(n, "_fieldnames"):
            h = int(to_float(getattr(n, "n_row", 0), 0))
            w = int(to_float(getattr(n, "n_col", 0), 0))
            if w > 0 and h > 0:
                return w, h
        else:
            a = np.asarray(n, dtype=float).flatten()
            if a.size >= 2 and a[0] > 0 and a[1] > 0:
                return int(a[1]), int(a[0])
    return 0, 0


def category_strings(node):
    out = []
    for item in as_list(node):
        if item is None:
            continue
        if hasattr(item, "_fieldnames") and "categories" in fields_of(item):
            out += category_strings(getattr(item, "categories"))
            continue
        s = to_str(item)
        if s in EMO2IDX:
            out.append(s)
        elif " " in s:
            out += [p for p in s.split() if p in EMO2IDX]
    return out


def annotator_sets(person):
    """Per-annotator label index sets from `annotations_categories`."""
    if "annotations_categories" not in fields_of(person):
        return []
    sets = []
    for rec in as_list(getattr(person, "annotations_categories")):
        cats = category_strings(rec)
        if cats:
            sets.append(sorted({EMO2IDX[c] for c in cats}))
    return sets


def extract_categories(person):
    """Official benchmark labels. Returns (label names, source tag)."""
    pf = fields_of(person)
    if "combined_categories" in pf:
        cats = category_strings(getattr(person, "combined_categories"))
        if cats:
            return sorted(set(cats)), "combined"
    if "annotations_categories" in pf:
        cats = category_strings(getattr(person, "annotations_categories"))
        if cats:
            return sorted(set(cats)), "single"
    return [], "empty"


def extract_vad(person):
    pf = fields_of(person)
    for fname in ("combined_continuous", "annotations_continuous"):
        if fname not in pf:
            continue
        vals = []
        for r in as_list(getattr(person, fname)):
            if r is not None and hasattr(r, "_fieldnames"):
                vals.append([to_float(getattr(r, d, 5.0)) for d in VAD_DIMS])
        if vals:
            return np.mean(np.asarray(vals, np.float32), 0), fname
    return np.array([5.0, 5.0, 5.0], np.float32), "default"


# ──────────────────────────────────────────────────────────────────────────────
# 2.3  IMAGE INDEX & RESOLVER
# ──────────────────────────────────────────────────────────────────────────────
print("\n2.3  IMAGE INDEX\n")
IMG_BY_SRC, IMG_BY_NAME = {}, {}
for src in SOURCE_FOLDERS:
    d = CFG.img_base / src / "images"
    if not d.is_dir():
        d = CFG.img_base / src
    if d.is_dir():
        for p in d.glob("*.jpg"):
            IMG_BY_SRC[(src, p.name)] = str(p)
            IMG_BY_NAME.setdefault(p.name, str(p))
V.check("image index populated",
        len(IMG_BY_SRC) > 20000 or os.environ.get("EMOTIC_TEST_MODE") == "1",
        f"{len(IMG_BY_SRC):,}", "~23,185")


def resolve_image(folder_raw, fname):
    """EMOTIC `folder` is a relative path such as 'mscoco/images'."""
    folder = folder_raw.replace("\\", "/").strip("/")
    parts = [q for q in folder.split("/") if q and q != "emotic"]
    src = parts[0] if parts else ""
    c = CFG.img_base / folder / fname
    if c.is_file():
        return str(c), src
    hit = IMG_BY_SRC.get((src, fname))
    if hit:
        return hit, src
    for c in (CFG.img_base / src / "images" / fname, CFG.img_base / src / fname):
        if c.is_file():
            return str(c), src
    return IMG_BY_NAME.get(fname), src


# ──────────────────────────────────────────────────────────────────────────────
# 2.4  PARSE
# ──────────────────────────────────────────────────────────────────────────────
print("\n2.4  PARSE\n")
records, source_tally, skipped = [], Counter(), Counter()
ann_counts = {s: Counter() for s in ("train", "val", "test")}
_t0 = time.time()

for split in ("train", "val", "test"):
    n_ok = 0
    for entry in as_list(MAT[split]):
        fname = to_str(getattr(entry, "filename", ""))
        folder_raw = to_str(getattr(entry, "folder", ""))
        if not fname or not folder_raw:
            skipped["no_name"] += 1
            continue
        img_path, src = resolve_image(folder_raw, fname)
        if img_path is None:
            skipped["no_image"] += 1
            continue
        W, H = image_hw(entry)

        for person in as_list(getattr(entry, "person", None)):
            if person is None:
                continue
            bbox = to_bbox(getattr(person, "body_bbox", None))
            if bbox is None:
                skipped["no_bbox"] += 1
                continue
            cats, csrc = extract_categories(person)
            source_tally[f"{split}:{csrc}"] += 1
            if not cats:
                skipped["no_label"] += 1
                continue

            vec = np.zeros(NUM_CLASSES, np.float32)
            for c in cats:
                vec[EMO2IDX[c]] = 1.0
            asets = annotator_sets(person)
            ann_counts[split][len(asets)] += 1
            vad, vsrc = extract_vad(person)
            vad = np.clip(vad / 10.0, 0.0, 1.0)

            records.append({
                "split": split, "img_path": img_path, "folder": src,
                "filename": fname, "img_w": W, "img_h": H,
                "xmin": bbox[0], "ymin": bbox[1], "xmax": bbox[2], "ymax": bbox[3],
                "label_vec": vec, "n_labels": int(vec.sum()),
                "ann_sets": asets, "n_annotators": len(asets),
                "valence": float(vad[0]), "arousal": float(vad[1]),
                "dominance": float(vad[2]),
                "label_source": csrc, "vad_source": vsrc,
            })
            n_ok += 1
    V.note(f"[{split}] persons parsed", f"{n_ok:,}")

DF = pd.DataFrame(records)
V.note("parse time", f"{time.time() - _t0:.1f}s")
V.check("records parsed",
        len(DF) > 30000 or os.environ.get("EMOTIC_TEST_MODE") == "1",
        f"{len(DF):,}", "~34,319")
V.note("skipped", ", ".join(f"{k}={v}" for k, v in sorted(skipped.items())) or "none")

for s in ("val", "test"):
    n_c = source_tally.get(f"{s}:combined", 0)
    n_t = sum(v for k, v in source_tally.items() if k.startswith(f"{s}:"))
    V.check(f"[{s}] uses the consensus field", n_t > 0 and n_c / n_t > 0.95,
            f"{100 * n_c / max(n_t, 1):.1f}%", "> 95%")


# ──────────────────────────────────────────────────────────────────────────────
# 2.5  ANNOTATOR STRUCTURE
# ──────────────────────────────────────────────────────────────────────────────
print("\n2.5  ANNOTATOR STRUCTURE\n")
ANN_MEDIAN = {}
for s in ("train", "val", "test"):
    dist = ann_counts[s]
    tot = sum(dist.values())
    ANN_MEDIAN[s] = int(np.median(np.repeat(list(dist.keys()),
                                            list(dist.values())))) if tot else 0
    V.note(f"[{s}] annotators/person",
           ", ".join(f"{k}:{100 * v / tot:.0f}%" for k, v in sorted(dist.items())) +
           f"   median={ANN_MEDIAN[s]}")

V.check("train is single-annotator", ANN_MEDIAN["train"] == 1, ANN_MEDIAN["train"], 1)
V.check("val has more annotators than test",
        ANN_MEDIAN["val"] > ANN_MEDIAN["test"],
        f"val {ANN_MEDIAN['val']} vs test {ANN_MEDIAN['test']}", "val > test")
K_TESTLIKE = max(1, ANN_MEDIAN["test"])
V.note("test-like k", f"{K_TESTLIKE} annotators (val will be recut to match)")


# ──────────────────────────────────────────────────────────────────────────────
# 2.6  LABEL DENSITY — matched against the benchmark
# ──────────────────────────────────────────────────────────────────────────────
print("\n2.6  LABEL DENSITY\n")
DENSITY = {s: float(DF.loc[DF["split"] == s, "n_labels"].mean())
           for s in ("train", "val", "test")}
for s in ("train", "val", "test"):
    V.note(f"[{s}] labels/person",
           f"{DENSITY[s]:.2f}   (benchmark {BENCHMARK_DENSITY[s]:.2f})")

_tol = 0.15 if os.environ.get("EMOTIC_TEST_MODE") != "1" else 99.0
for s in ("train", "val", "test"):
    V.check(f"[{s}] density matches benchmark",
            abs(DENSITY[s] - BENCHMARK_DENSITY[s]) < _tol,
            f"{DENSITY[s]:.2f}", f"{BENCHMARK_DENSITY[s]:.2f} +/- 0.15")

print()
print(f"  {_B}The asymmetry is the benchmark, not a bug.{_0}")
print("  combined_categories is the UNION over annotators, so density scales")
print("  with annotator count (1 / 5 / 3). Raw mAP is therefore not comparable")
print("  across splits, because each split has a different prevalence floor.")


# ──────────────────────────────────────────────────────────────────────────────
# 2.7  TEST-LIKE VAL LABELS
# ──────────────────────────────────────────────────────────────────────────────
print("\n2.7  TEST-LIKE VAL LABELS\n")
_rng = np.random.RandomState(CFG.seed)


def testlike_vector(ann_sets, k):
    """Union over a seeded k-subset of this person's annotators."""
    if not ann_sets:
        return None
    idx = np.arange(len(ann_sets))
    if len(idx) > k:
        idx = _rng.choice(idx, size=k, replace=False)
    v = np.zeros(NUM_CLASSES, np.float32)
    for i in idx:
        for c in ann_sets[i]:
            v[c] = 1.0
    return v


tl, n_built = [], 0
for r in DF.itertuples(index=False):
    if r.split == "val":
        v = testlike_vector(r.ann_sets, K_TESTLIKE)
        if v is not None and v.sum() > 0:
            tl.append(v); n_built += 1
            continue
    tl.append(r.label_vec.copy())
DF["label_vec_testlike"] = tl
DF["n_labels_testlike"] = [int(v.sum()) for v in tl]

_d_tl = float(DF.loc[DF["split"] == "val", "n_labels_testlike"].mean())
V.note("val test-like density", f"{_d_tl:.2f}  (official val {DENSITY['val']:.2f}, "
                               f"test {DENSITY['test']:.2f})")
V.check("test-like val rebuilt for most persons",
        n_built / max((DF["split"] == "val").sum(), 1) > 0.95,
        f"{n_built:,}", f"{int((DF['split'] == 'val').sum()):,}")
V.check("test-like val density close to test",
        abs(_d_tl - DENSITY["test"]) < 1.2,
        f"{_d_tl:.2f}", f"{DENSITY['test']:.2f} +/- 1.2", fatal=False)


# ──────────────────────────────────────────────────────────────────────────────
# 2.8  SANITY & PERSIST
# ──────────────────────────────────────────────────────────────────────────────
print("\n2.8  SANITY & PERSIST\n")
SIZES = {s: int((DF["split"] == s).sum()) for s in ("train", "val", "test")}
EXPECTED = {"train": 23706, "val": 3332, "test": 7280}
for s in ("train", "val", "test"):
    d = abs(SIZES[s] - EXPECTED[s]) / EXPECTED[s]
    V.check(f"[{s}] size within 5% of published", d < 0.05,
            f"{SIZES[s]:,}", f"~{EXPECTED[s]:,}", fatal=False)

TEST_AUDIT = {"ours": SIZES["test"], "official": 7280, "xenos_2024": 7203,
              "no_label_skipped": int(skipped.get("no_label", 0)),
              "test_annotators": {int(k): int(v) for k, v in ann_counts["test"].items()}}
V.note("test-set audit", f"{SIZES['test']:,} persons = official 7,280 minus {skipped.get('no_label', 0)} "
                         f"with no label; Xenos et al. report 7,203 ({SIZES['test'] - 7203:+d} vs ours). Their "
                         f"exclusion list is not published, so published numbers are compared as reported.")
_t_ann = ann_counts["test"]
V.note("test annotators per person", ", ".join(f"{k}: {100 * v / max(sum(_t_ann.values()), 1):.0f}%"
                                               for k, v in sorted(_t_ann.items())))
V.check("every person has >= 1 label", int(DF["n_labels"].min()) >= 1,
        int(DF["n_labels"].min()), ">= 1")
V.check("boxes well formed",
        bool(((DF["xmax"] > DF["xmin"]) & (DF["ymax"] > DF["ymin"])).all()),
        "valid", "valid")
V.check("image sizes recovered",
        float(((DF["img_w"] > 0) & (DF["img_h"] > 0)).mean()) > 0.99,
        f"{100 * float(((DF['img_w'] > 0) & (DF['img_h'] > 0)).mean()):.1f}%", "> 99%")
V.check("VAD within [0,1]",
        bool(DF[VAD_DIMS].min().min() >= 0 and DF[VAD_DIMS].max().max() <= 1),
        "in range", "[0,1]")

_out = CFG.cache_dir / "annotations.pkl"
DF.to_pickle(_out)
V.check("dataframe persisted", _out.is_file(),
        f"{_out.stat().st_size / 1e6:.1f} MB", "written")

(CFG.cache_dir / "annotation_report.json").write_text(json.dumps({
    "density": DENSITY, "benchmark_density": BENCHMARK_DENSITY,
    "val_testlike_density": _d_tl, "k_testlike": K_TESTLIKE,
    "ann_median": ANN_MEDIAN, "sizes": SIZES, "test_audit": TEST_AUDIT}, indent=2))

del MAT
free_memory()
BLOCK_02_OK = V.finish()


In [ ]:
# Block 03 — Splits, Prevalence Floors, Geometry & Protocol Gap
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–02
#
# 3.1–3.4: split frames, prevalence floors (lift = mAP / floor), per-class
# statistics, 9-D person geometry.
#
# 3.5 THE PROTOCOL GAP (the measurement behind the paper's analysis). A train
#   label comes from ONE annotator; a test label is the UNION of THREE. So a
#   train "negative" is often a test-protocol positive. Measured on val only
#   (4–5 annotators per person): one annotator is the "single" labeller, the
#   union of 3 OTHER annotators the reference, and per class
#
#       rho_c  = P( c in 3-annotator union | single annotator did not mark c )
#       beta_c = P( single annotator ticks c | 3-annotator union ticks c )
#
#   Only val annotation records are used — no test data.
#
#   v7 note. Under the annotator model checked here (a latent label, each
#   annotator ticking it independently with probability beta_c), a person's
#   single-annotator tick probability and 3-annotator union probability are
#   both proportional to the same latent probability, so they RANK persons
#   identically within a class — and mAP only depends on that ranking. So
#   re-calibrating the targets towards the test protocol cannot improve mAP
#   in principle (v6 APC: −4.2 mAP in practice), while reducing the NOISE of
#   the single-annotator targets can: that is what v7's cross-fitted
#   self-distillation (Block 08) is for.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

header("BLOCK 03 — Splits, Floors, Geometry & Protocol Gap  [v7]")
V = BlockVerdict(3, "Splits & Protocol")
TEST_MODE = os.environ.get("EMOTIC_TEST_MODE") == "1"

# ──────────────────────────────────────────────────────────────────────────────
# 3.1  SPLIT FRAMES
# ──────────────────────────────────────────────────────────────────────────────
print("3.1  SPLIT FRAMES\n")
df_train = DF[DF["split"] == "train"].reset_index(drop=True).copy()
df_val = DF[DF["split"] == "val"].reset_index(drop=True).copy()
df_test = DF[DF["split"] == "test"].reset_index(drop=True).copy()
SPLIT_DF = {"train": df_train, "val": df_val, "test": df_test}

for s, d in SPLIT_DF.items():
    V.note(f"[{s}]", f"{len(d):,} persons, {d['filename'].nunique():,} images")
V.check("splits non-empty", all(len(d) > 0 for d in SPLIT_DF.values()),
        [len(d) for d in SPLIT_DF.values()], "all > 0")

_img = {s: set(d["filename"]) for s, d in SPLIT_DF.items()}
_ov = {f"{a}|{b}": len(_img[a] & _img[b])
       for a, b in (("train", "val"), ("train", "test"), ("val", "test"))}
V.note("image overlap", _ov)
V.check("train|test image overlap is zero", _ov["train|test"] == 0, _ov["train|test"], 0)
V.check("val|test image overlap is zero", _ov["val|test"] == 0, _ov["val|test"], 0)
V.check("train|val overlap (official split artefact)", _ov["train|val"] < 600,
        _ov["train|val"], "< 600 (person-level split)", fatal=False)


def label_matrix(d, col="label_vec"):
    return np.stack(d[col].values).astype(np.float32)


Y = {s: label_matrix(d) for s, d in SPLIT_DF.items()}
Y_TESTLIKE_VAL = label_matrix(df_val, "label_vec_testlike")
TRAIN_LABELS, VAL_LABELS, TEST_LABELS = Y["train"], Y["val"], Y["test"]
V.check("label matrix width", TRAIN_LABELS.shape[1] == NUM_CLASSES,
        TRAIN_LABELS.shape[1], NUM_CLASSES)

# ──────────────────────────────────────────────────────────────────────────────
# 3.2  PREVALENCE FLOORS
# ──────────────────────────────────────────────────────────────────────────────
print("\n3.2  PREVALENCE FLOORS (random-ranker mAP)\n")
PREVALENCE = {s: Y[s].mean(0) for s in Y}
PREVALENCE["val_testlike"] = Y_TESTLIKE_VAL.mean(0)
MAP_FLOOR = {s: float(p.mean() * 100.0) for s, p in PREVALENCE.items()}

for s in ("train", "val", "val_testlike", "test"):
    n = (Y_TESTLIKE_VAL if s == "val_testlike" else Y[s]).sum(1).mean()
    V.note(f"[{s}] floor", f"{MAP_FLOOR[s]:6.2f}%   (labels/person {n:.2f})")

V.check("test-like val floor matches test floor",
        abs(MAP_FLOOR["val_testlike"] - MAP_FLOOR["test"]) < 3.0,
        f"{abs(MAP_FLOOR['val_testlike'] - MAP_FLOOR['test']):.2f}", "< 3.0", fatal=False)

XENOS_TEST_MAP, CCIM_TEST_MAP = CFG.xenos_test_map, CFG.ccim_test_map
TARGET_LIFT = CCIM_TEST_MAP / MAP_FLOOR["test"]
print()
print(f"  {_B}Reference lifts on our test floor of {MAP_FLOOR['test']:.2f}{_0}")
for nm, m in (("Xenos et al. 2024", XENOS_TEST_MAP), ("EmotiCon+CCIM (SOTA)", CCIM_TEST_MAP),
              ("our paper-parity run", 38.99)):
    print(f"    {nm:<24s} {m:5.2f} mAP  ->  lift {m / MAP_FLOOR['test']:.3f}x")

# ──────────────────────────────────────────────────────────────────────────────
# 3.3  PER-CLASS STATISTICS
# ──────────────────────────────────────────────────────────────────────────────
print("\n3.3  PER-CLASS STATISTICS\n")
stats = pd.DataFrame({
    "emotion": EMOTION_LABELS,
    "train_n": TRAIN_LABELS.sum(0).astype(int),
    "val_n": VAL_LABELS.sum(0).astype(int),
    "test_n": TEST_LABELS.sum(0).astype(int),
    "train_prev": PREVALENCE["train"],
    "test_prev": PREVALENCE["test"],
}).sort_values("train_n", ascending=False).reset_index(drop=True)
print(stats.to_string(index=False, float_format="{:.4f}".format))
V.note("imbalance", f"{stats['train_n'].max() / max(stats['train_n'].min(), 1):.1f}x")
V.check("all classes have train positives", stats["train_n"].min() > 0,
        int(stats["train_n"].min()), "> 0")
V.check("all classes have test positives", stats["test_n"].min() > 0,
        int(stats["test_n"].min()), "> 0")
stats.to_csv(CFG.cache_dir / "class_stats.csv", index=False)

# ──────────────────────────────────────────────────────────────────────────────
# 3.4  GEOMETRY FEATURES
# ──────────────────────────────────────────────────────────────────────────────
print("\n3.4  GEOMETRY FEATURES\n")


def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)
    return float(inter / max(ua, 1e-6))


def add_geometry(d):
    """9-D geometry from annotation image sizes: centre, size, area, aspect,
    person count, max/mean IoU with the other people in the photo."""
    d = d.copy()
    # np.array(...) copies: pandas >= 3 hands out READ-ONLY views from to_numpy()
    W = np.array(d["img_w"].to_numpy(), dtype=np.float32)
    H = np.array(d["img_h"].to_numpy(), dtype=np.float32)
    bad = (W <= 0) | (H <= 0)
    W[bad], H[bad] = 640.0, 480.0
    box = np.array(d[["xmin", "ymin", "xmax", "ymax"]].to_numpy(), dtype=np.float32)
    box[:, 0] = np.clip(box[:, 0], 0, W - 2)
    box[:, 1] = np.clip(box[:, 1], 0, H - 2)
    box[:, 2] = np.clip(box[:, 2], box[:, 0] + 1, W)
    box[:, 3] = np.clip(box[:, 3], box[:, 1] + 1, H)
    by_img = {}
    for i, k in enumerate(d["filename"].to_numpy()):
        by_img.setdefault(k, []).append(i)
    geo = np.zeros((len(d), 9), np.float32)
    for idxs in by_img.values():
        n = len(idxs)
        for i in idxs:
            x1, y1, x2, y2 = box[i]
            bw, bh = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
            ious = [_iou(box[i], box[j]) for j in idxs if j != i]
            geo[i] = [(x1 + x2) / 2 / W[i], (y1 + y2) / 2 / H[i], bw / W[i], bh / H[i],
                      (bw * bh) / (W[i] * H[i]), min(bw / bh, 10.0) / 10.0, min(n, 10) / 10.0,
                      max(ious) if ious else 0.0, float(np.mean(ious)) if ious else 0.0]
    d["geometry"] = list(np.clip(geo, 0.0, 1.0))
    return d


_t0 = time.time()
for s in SPLIT_DF:
    SPLIT_DF[s] = add_geometry(SPLIT_DF[s])
df_train, df_val, df_test = SPLIT_DF["train"], SPLIT_DF["val"], SPLIT_DF["test"]
V.note("build time", f"{time.time() - _t0:.1f}s")
_g = np.stack(df_train["geometry"].values)
GEOMETRY_DIM = int(_g.shape[1])
V.check("geometry dimensionality", GEOMETRY_DIM == 9, GEOMETRY_DIM, 9)
V.check("geometry finite and in [0,1]",
        bool(np.isfinite(_g).all() and (_g >= 0).all() and (_g <= 1).all()),
        f"[{_g.min():.3f}, {_g.max():.3f}]", "[0, 1]")
V.note("multi-person images", f"{100 * (_g[:, 6] > 0.1).mean():.1f}% of train persons")

# ──────────────────────────────────────────────────────────────────────────────
# 3.5  PROTOCOL GAP  (val annotator records only)
# ──────────────────────────────────────────────────────────────────────────────
print("\n3.5  PROTOCOL GAP — single annotator vs 3-annotator union\n")
K_REF = int(K_TESTLIKE)
_rs = np.random.RandomState(CFG.seed + 11)
_neg = np.zeros(NUM_CLASSES)
_fn = np.zeros(NUM_CLASSES)
_single_pos = np.zeros(NUM_CLASSES)
_union_pos = np.zeros(NUM_CLASSES)
_both_pos = np.zeros(NUM_CLASSES)
_n_single, _dens = 0, []
for sets in df_val["ann_sets"]:
    m = len(sets)
    if m < K_REF + 1:
        continue
    for a in range(m):
        others = [j for j in range(m) if j != a]
        pick = _rs.choice(others, size=K_REF, replace=False)
        ref = np.zeros(NUM_CLASSES, bool)
        for j in pick:
            ref[list(sets[j])] = True
        s_ = np.zeros(NUM_CLASSES, bool)
        s_[list(sets[a])] = True
        _neg += ~s_
        _fn += (~s_) & ref
        _single_pos += s_
        _union_pos += ref
        _both_pos += s_ & ref
        _n_single += 1
        _dens.append(int(s_.sum()))

PROTOCOL_RHO = (_fn / np.maximum(_neg, 1.0)).astype(np.float32)
PROTOCOL_BETA = np.clip(np.where(_union_pos > 0, _both_pos / np.maximum(_union_pos, 1.0), 0.5),
                        0.05, 1.0).astype(np.float32)
_p_single = _single_pos / max(_n_single, 1)
_p_implied = _p_single + (1.0 - _p_single) * PROTOCOL_RHO
# annotator model: P(Y=1) = p_single / beta; rho = P(Y)(1-beta)(1-(1-beta)^K) / (1 - p_single)
_pY = np.clip(_p_single / PROTOCOL_BETA, 0.0, 0.999)
_rho_model = _pY * (1 - PROTOCOL_BETA) * (1 - (1 - PROTOCOL_BETA) ** K_REF) / np.maximum(1 - _p_single, 1e-6)
gap = pd.DataFrame({
    "emotion": EMOTION_LABELS,
    "train_prev (1 ann)": PREVALENCE["train"],
    "val single-ann prev": _p_single,
    "rho (missed by 1 of 3)": PROTOCOL_RHO,
    "beta (tick rate)": PROTOCOL_BETA,
    "rho from beta model": _rho_model,
    "implied union prev": _p_implied,
    "val-TL prev (measured)": PREVALENCE["val_testlike"],
}).sort_values("rho (missed by 1 of 3)", ascending=False).reset_index(drop=True)
print(gap.to_string(index=False, float_format="{:.3f}".format))
gap.to_csv(CFG.cache_dir / "protocol_gap.csv", index=False)

_single_density = float(np.mean(_dens)) if _dens else 0.0
_train_density = float(TRAIN_LABELS.sum(1).mean())
print()
V.note("single-annotator samples used", f"{_n_single:,} (val persons with >= {K_REF + 1} annotators)")
V.note("labels per single annotator", f"val {_single_density:.2f}  vs  train {_train_density:.2f}")
V.check("enough single-vs-union comparisons", _n_single >= (100 if TEST_MODE else 5000),
        f"{_n_single:,}", ">= 5,000")
V.check("val single annotators label like the train annotator",
        abs(_single_density - _train_density) / max(_train_density, 1e-9) < 0.35,
        f"{_single_density:.2f} vs {_train_density:.2f}", "within 35%", fatal=False)
V.check("rho finite and in [0, 0.9]",
        bool(np.isfinite(PROTOCOL_RHO).all() and PROTOCOL_RHO.min() >= 0 and PROTOCOL_RHO.max() <= 0.9),
        f"[{PROTOCOL_RHO.min():.3f}, {PROTOCOL_RHO.max():.3f}]", "[0, 0.9]")
V.check("beta finite and in [0.05, 1]", bool(np.isfinite(PROTOCOL_BETA).all()), 
        f"[{PROTOCOL_BETA.min():.3f}, {PROTOCOL_BETA.max():.3f}]", "[0.05, 1]")
_mae_model = float(np.abs(_rho_model - PROTOCOL_RHO).mean())
V.note("annotator model", f"beta mean {PROTOCOL_BETA.mean():.3f}; rho predicted from beta vs measured: "
                          f"mean abs diff {_mae_model:.4f} (mean rho {PROTOCOL_RHO.mean():.3f})")
V.check("annotator model reproduces the measured rho", _mae_model < 0.03,
        f"mean abs diff {_mae_model:.4f}", "< 0.03", fatal=False)
_mae = float(np.abs(_p_implied - PREVALENCE["val_testlike"]).mean())
V.check("single+rho reproduces measured 3-annotator prevalence", _mae < 0.05,
        f"mean abs diff {_mae:.3f}", "< 0.05", fatal=False)
_common = np.argsort(-PREVALENCE["train"])[:6]
_rare = np.argsort(PREVALENCE["train"])[:6]
V.note("mean rho, 6 most common classes", f"{PROTOCOL_RHO[_common].mean():.3f}")
V.note("mean rho, 6 rarest classes", f"{PROTOCOL_RHO[_rare].mean():.3f}")
print(f"\n  {_B}Reading this table{_0}")
print("    A train negative for a class near the top is often a positive under the")
print("    test protocol. Plain BCE pushes those samples towards 0 anyway, which is")
print("    label noise concentrated exactly where EMOTIC models score worst in lift.")

# ──────────────────────────────────────────────────────────────────────────────
# 3.6  PERSIST
# ──────────────────────────────────────────────────────────────────────────────
print("\n3.6  PERSIST\n")
np.save(CFG.array_dir / "protocol_rho.npy", PROTOCOL_RHO)
np.save(CFG.array_dir / "protocol_beta.npy", PROTOCOL_BETA)
FLOORS = {"map_floor": MAP_FLOOR, "density": {s: float(Y[s].sum(1).mean()) for s in Y},
          "protocol_k": K_REF, "beta_mean": float(PROTOCOL_BETA.mean()), "rho_mean": float(PROTOCOL_RHO.mean()),
          "sizes": {s: int(len(d)) for s, d in SPLIT_DF.items()},
          "xenos_test_map": XENOS_TEST_MAP, "ccim_test_map": CCIM_TEST_MAP}
(CFG.cache_dir / "floors.json").write_text(json.dumps(FLOORS, indent=2))
for s, d in SPLIT_DF.items():
    d.to_pickle(CFG.cache_dir / f"df_{s}.pkl")
V.check("split frames + rho persisted",
        all((CFG.cache_dir / f"df_{s}.pkl").is_file() for s in SPLIT_DF)
        and (CFG.array_dir / "protocol_rho.npy").is_file()
        and (CFG.array_dir / "protocol_beta.npy").is_file(), "written", "written")
free_memory()
BLOCK_03_OK = V.finish()


In [ ]:
# Block 04 — Description Cache Remap & Coverage Validation
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–03
#
# Unchanged logic from the all-green version.
#
#   The cached LLaVa descriptions were generated under an older Kaggle mount
#   (/kaggle/input/datasets/dropatiyadav/...). Every cache key embeds that old
#   absolute path, so a plain dict lookup would miss silently and fall back to
#   a placeholder — and text is the DOMINANT modality (34.25 mAP text-only vs
#   29.19 vision-only in Xenos et al.'s ablation). Keys are re-anchored on the
#   stable suffix <source>/images/<file>. 4.3 requires >= 99.5% coverage.
# ══════════════════════════════════════════════════════════════════════════════

import json
import numpy as np
from pathlib import Path

header("BLOCK 04 — Description Cache Remap & Coverage Validation")
V = BlockVerdict(4, "Descriptions")

FALLBACK_DESC = ("The person in the image appears to be experiencing emotions "
                 "related to their current situation and context.")

# ──────────────────────────────────────────────────────────────────────────────
# 4.1  LOAD RAW CACHES
# ──────────────────────────────────────────────────────────────────────────────
print("4.1  LOAD RAW CACHES\n")
RAW = {}
for split, path in CFG.desc_paths.items():
    with open(path) as f:
        RAW[split] = json.load(f)
    V.note(f"[{split}]", f"{len(RAW[split]):,} entries  ({path.stat().st_size / 1e6:.1f} MB)")
V.check("all three caches loaded", len(RAW) == 3, len(RAW), 3)

# ──────────────────────────────────────────────────────────────────────────────
# 4.2  KEY REMAPPING
# ──────────────────────────────────────────────────────────────────────────────
print("\n4.2  KEY REMAPPING\n")
SOURCES = ("ade20k", "emodb_small", "framesdb", "mscoco")


def rebase(path_str):
    """Re-anchor an absolute image path onto the CURRENT image root."""
    for src in SOURCES:
        marker = f"/{src}/"
        if marker in path_str:
            suffix = path_str.split(marker, 1)[1]
            return str(CFG.img_base / src / suffix)
    return path_str


def make_key(img_path, xmin, ymin, xmax, ymax):
    """Canonical cache key. Must match exactly between build and lookup."""
    return json.dumps([str(img_path), int(xmin), int(ymin), int(xmax), int(ymax)])


DESC, remap_stats = {}, {}
for split, raw in RAW.items():
    out, n_changed, n_bad = {}, 0, 0
    for k, v in raw.items():
        try:
            p, x1, y1, x2, y2 = json.loads(k)
        except Exception:
            n_bad += 1
            continue
        p2 = rebase(p)
        n_changed += p2 != p
        out[make_key(p2, x1, y1, x2, y2)] = v
    DESC[split] = out
    remap_stats[split] = {"in": len(raw), "out": len(out), "changed": n_changed, "unparsable": n_bad}
    V.note(f"[{split}] remapped", f"{n_changed:,}/{len(raw):,} keys rewritten, {n_bad} unparsable")

V.check("no unparsable keys", all(s["unparsable"] == 0 for s in remap_stats.values()),
        sum(s["unparsable"] for s in remap_stats.values()), 0)
V.check("no key collisions after remap", all(s["in"] == s["out"] for s in remap_stats.values()),
        {k: (s["in"] - s["out"]) for k, s in remap_stats.items()}, "0 lost")
for split in DESC:
    sample = [json.loads(k)[0] for k in list(DESC[split])[:300]]
    hit = sum(Path(p).is_file() for p in sample)
    V.check(f"[{split}] rebased paths exist on disk", hit / max(len(sample), 1) > 0.99,
            f"{hit}/{len(sample)}", f"{len(sample)}/{len(sample)}")

# ──────────────────────────────────────────────────────────────────────────────
# 4.3  COVERAGE
# ──────────────────────────────────────────────────────────────────────────────
print("\n4.3  COVERAGE\n")
COVERAGE, MISSING = {}, {}
for split, d in (("train", df_train), ("val", df_val), ("test", df_test)):
    keys = [make_key(r.img_path, r.xmin, r.ymin, r.xmax, r.ymax) for r in d.itertuples(index=False)]
    have = [k in DESC[split] for k in keys]
    COVERAGE[split] = float(np.mean(have)) if have else 0.0
    MISSING[split] = [k for k, h in zip(keys, have) if not h]
    V.note(f"[{split}] coverage", f"{100 * COVERAGE[split]:.2f}%  "
                                  f"({len(keys) - len(MISSING[split]):,}/{len(keys):,})")
for split in ("train", "val", "test"):
    V.check(f"[{split}] coverage >= 99.5%", COVERAGE[split] >= 0.995,
            f"{100 * COVERAGE[split]:.2f}%", ">= 99.50%")

# ──────────────────────────────────────────────────────────────────────────────
# 4.4  QUALITY AUDIT
# ──────────────────────────────────────────────────────────────────────────────
print("\n4.4  QUALITY AUDIT\n")
HEDGE_MARKERS = ("not possible to", "difficult to determine", "cannot be determined",
                 "without more context", "hard to tell", "cannot definitively")
QUALITY = {}
for split, d in DESC.items():
    vals = list(d.values())
    lens = np.array([len(v) for v in vals])
    QUALITY[split] = {"n": len(vals), "unique": len(set(vals)),
                      "mean_chars": float(lens.mean()), "p95_chars": float(np.percentile(lens, 95)),
                      "hedged_pct": 100.0 * sum(any(m in v.lower() for m in HEDGE_MARKERS) for v in vals) / len(vals),
                      "mentions_box_pct": 100.0 * sum("red box" in v.lower() for v in vals) / len(vals)}
    q = QUALITY[split]
    V.note(f"[{split}]", f"{q['mean_chars']:.0f} chars mean, p95 {q['p95_chars']:.0f}, "
                         f"{q['unique']:,} unique, {q['mentions_box_pct']:.1f}% cite the red box, "
                         f"{q['hedged_pct']:.1f}% hedge")
V.check("descriptions are mostly distinct",
        all(q["unique"] / q["n"] > 0.95 for q in QUALITY.values()),
        {k: round(q["unique"] / q["n"], 3) for k, q in QUALITY.items()}, "> 0.95")
V.check("hedging rate below 15%", all(q["hedged_pct"] < 15.0 for q in QUALITY.values()),
        {k: round(q["hedged_pct"], 1) for k, q in QUALITY.items()}, "< 15%", fatal=False)

# ──────────────────────────────────────────────────────────────────────────────
# 4.5  PERSIST
# ──────────────────────────────────────────────────────────────────────────────
print("\n4.5  PERSIST\n")
for split, d in DESC.items():
    p = CFG.cache_dir / f"desc_{split}_remapped.json"
    p.write_text(json.dumps(d, ensure_ascii=False))
    V.note(f"[{split}] saved", f"{p.name}  ({p.stat().st_size / 1e6:.1f} MB)")
desc_train, desc_val, desc_test = DESC["train"], DESC["val"], DESC["test"]
V.check("lookup handles available",
        all(isinstance(x, dict) and len(x) > 0 for x in (desc_train, desc_val, desc_test)),
        [len(desc_train), len(desc_val), len(desc_test)], "all non-empty")
del RAW
free_memory()
BLOCK_04_OK = V.finish()


In [ ]:
# Block 05 — Tokenisation & Training Arrays
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–04
#
# Writes everything a GPU worker needs, except the visual features (Block 07),
# to /kaggle/working/cache/arrays/:
#
#   {split}_ids_flat.npy / _ids_off.npy   Q-Former tokens of each description,
#                                         stored ragged (no padding on disk)
#   {split}_labels.npy                    official labels (benchmark protocol)
#   val_labels_tl.npy                     test-like val labels (selection)
#   {split}_geo.npy, {split}_vad.npy      9-D geometry, VAD targets
#   {split}_index.npz                     image path + raw box, for Block 07
#   protocol_rho.npy                      written by Block 03
#   train_folds.npy                       v7: cross-fitting folds [repeats, n_train]
#                                         for the self-distillation teachers.
#                                         Folds are GROUPED BY IMAGE: all persons
#                                         of one photo share a fold, so a teacher
#                                         never saw the scene of a person it
#                                         predicts out-of-fold. Balanced greedily
#                                         by person count; fixed seed.
#
# WHY 320 TOKENS
#   5.2 measures real token lengths with the Q-Former tokenizer. At 224 tokens
#   a fifth of descriptions lose their final sentence — typically LLaVa's
#   conclusion naming the emotions. Workers pad each batch only to its own
#   longest description, so the extra length costs little compute.
# ══════════════════════════════════════════════════════════════════════════════

import hashlib
import numpy as np
from transformers import InstructBlipProcessor

header("BLOCK 05 — Tokenisation & Training Arrays  [v7]")
V = BlockVerdict(5, "Arrays")

# ──────────────────────────────────────────────────────────────────────────────
# 5.1  TOKENIZER
# ──────────────────────────────────────────────────────────────────────────────
print("5.1  TOKENIZER\n")
_t0 = time.time()
_proc = InstructBlipProcessor.from_pretrained(CFG.backbone_ckpt)
QTOK = _proc.qformer_tokenizer
del _proc
V.note("loaded", f"{type(QTOK).__name__} in {time.time() - _t0:.1f}s")
PAD_ID, CLS_ID, SEP_ID = int(QTOK.pad_token_id), int(QTOK.cls_token_id), int(QTOK.sep_token_id)
V.note("special ids", f"pad {PAD_ID}  cls {CLS_ID}  sep {SEP_ID}  vocab {len(QTOK):,}")
V.check("Q-Former tokenizer is BERT-style (pad 0, cls 101, sep 102)",
        (PAD_ID, CLS_ID, SEP_ID) == (0, 101, 102), (PAD_ID, CLS_ID, SEP_ID), (0, 101, 102))
V.check("text length fits Q-Former position table (512)", CFG.max_text_len <= 512,
        CFG.max_text_len, "<= 512")

# ──────────────────────────────────────────────────────────────────────────────
# 5.2  TOKENISE (and measure true lengths)
# ──────────────────────────────────────────────────────────────────────────────
print("\n5.2  TOKENISE\n")
DESC_BY_SPLIT = {"train": desc_train, "val": desc_val, "test": desc_test}
TOKENS, LENGTH_STATS = {}, {}
for split, d in (("train", df_train), ("val", df_val), ("test", df_test)):
    texts, n_fallback = [], 0
    for r in d.itertuples(index=False):
        t = DESC_BY_SPLIT[split].get(make_key(r.img_path, r.xmin, r.ymin, r.xmax, r.ymax))
        if not isinstance(t, str) or len(t.strip()) < 20:
            t, n_fallback = FALLBACK_DESC, n_fallback + 1
        texts.append(t)
    full = QTOK(texts, truncation=False, padding=False)["input_ids"]
    true_len = np.array([len(x) for x in full])
    enc = QTOK(texts, truncation=True, max_length=CFG.max_text_len, padding=False)["input_ids"]
    TOKENS[split] = enc
    LENGTH_STATS[split] = {
        "n": len(texts), "fallback": n_fallback,
        "mean": float(true_len.mean()), "p50": float(np.percentile(true_len, 50)),
        "p95": float(np.percentile(true_len, 95)), "max": int(true_len.max()),
        "cut_at_224_pct": float(100 * (true_len > 224).mean()),
        "cut_at_max_pct": float(100 * (true_len > CFG.max_text_len).mean())}
    s = LENGTH_STATS[split]
    V.note(f"[{split}] tokens", f"mean {s['mean']:.0f}  p50 {s['p50']:.0f}  p95 {s['p95']:.0f}  "
                                f"max {s['max']}  | cut at 224: {s['cut_at_224_pct']:.1f}%  "
                                f"cut at {CFG.max_text_len}: {s['cut_at_max_pct']:.1f}%")

for split in TOKENS:
    V.check(f"[{split}] no fallback descriptions", LENGTH_STATS[split]["fallback"] == 0,
            LENGTH_STATS[split]["fallback"], 0)
_cut = max(s["cut_at_max_pct"] for s in LENGTH_STATS.values())
V.check(f"descriptions cut at {CFG.max_text_len} tokens below 5%", _cut < 5.0, f"{_cut:.1f}%", "< 5%")
V.check("every sequence starts [CLS] and ends [SEP]",
        all(seq[0] == CLS_ID and seq[-1] == SEP_ID for t in TOKENS.values() for seq in t),
        "yes", "yes")

# ──────────────────────────────────────────────────────────────────────────────
# 5.3  WRITE ARRAYS
# ──────────────────────────────────────────────────────────────────────────────
print("\n5.3  WRITE ARRAYS\n")
AD = CFG.array_dir
ORDER_HASH = {}
for split, d in (("train", df_train), ("val", df_val), ("test", df_test)):
    seqs = TOKENS[split]
    off = np.zeros(len(seqs) + 1, np.int64)
    off[1:] = np.cumsum([len(x) for x in seqs])
    flat = np.fromiter((t for s in seqs for t in s), dtype=np.int32, count=int(off[-1]))
    np.save(AD / f"{split}_ids_flat.npy", flat)
    np.save(AD / f"{split}_ids_off.npy", off)
    np.save(AD / f"{split}_labels.npy", np.stack(d["label_vec"].values).astype(np.float32))
    np.save(AD / f"{split}_geo.npy", np.stack(d["geometry"].values).astype(np.float32))
    np.save(AD / f"{split}_vad.npy", d[VAD_DIMS].to_numpy(np.float32))
    np.savez(AD / f"{split}_index.npz", paths=np.array([str(p) for p in d["img_path"]]),
             boxes=d[["xmin", "ymin", "xmax", "ymax"]].to_numpy(np.int32))
    h = hashlib.sha1()
    for r in d.itertuples(index=False):
        h.update(f"{r.img_path}|{r.xmin}|{r.ymin}|{r.xmax}|{r.ymax};".encode())
    ORDER_HASH[split] = h.hexdigest()
    V.note(f"[{split}]", f"{len(seqs):,} rows, {off[-1]:,} tokens, order {ORDER_HASH[split][:10]}")
np.save(AD / "val_labels_tl.npy", np.stack(df_val["label_vec_testlike"].values).astype(np.float32))

# ──────────────────────────────────────────────────────────────────────────────
# 5.4  CROSS-FITTING FOLDS (self-distillation teachers, Block 08)
# ──────────────────────────────────────────────────────────────────────────────
print("\n5.4  CROSS-FITTING FOLDS\n")
KD_K, KD_R = int(CFG.kd_folds), int(CFG.kd_repeats)
_img_key = df_train["img_path"].astype(str).to_numpy()
_uniq, _inv = np.unique(_img_key, return_inverse=True)
_gsize = np.bincount(_inv)
TRAIN_FOLDS = np.zeros((KD_R, len(df_train)), np.int8)
for _r in range(KD_R):
    _rs = np.random.RandomState(int(CFG.kd_fold_seed) + 1000 * _r)
    _load = np.zeros(KD_K)
    _gf = np.zeros(len(_uniq), np.int8)
    for _gi in _rs.permutation(len(_uniq)):          # random order, smallest fold first
        _f = int(np.argmin(_load + _rs.rand(KD_K) * 1e-3))
        _gf[_gi] = _f
        _load[_f] += _gsize[_gi]
    TRAIN_FOLDS[_r] = _gf[_inv]
np.save(AD / "train_folds.npy", TRAIN_FOLDS)
_fsz = np.array([[int((TRAIN_FOLDS[r] == k).sum()) for k in range(KD_K)] for r in range(KD_R)])
V.note("folds", f"{KD_K} folds x {KD_R} repeats over {len(_uniq):,} images / {len(df_train):,} persons; "
                f"persons per fold {_fsz.min()}–{_fsz.max()}")
V.check("every fold holds persons, sizes balanced (±3%)",
        _fsz.min() > 0 and (_fsz.max() - _fsz.min()) <= 0.03 * len(df_train) / KD_K + 2,
        f"{_fsz.min()}–{_fsz.max()}", f"≈ {len(df_train) / KD_K:.0f} each")
_split_imgs = sum(int(len(np.unique(TRAIN_FOLDS[r][_inv == g])) > 1)
                  for r in range(KD_R) for g in np.where(_gsize > 1)[0])
V.check("no image is split across folds (grouped by image)", _split_imgs == 0, _split_imgs, 0)
if KD_R > 1:
    _agree = float(np.mean([(TRAIN_FOLDS[0][_inv == g][0] == TRAIN_FOLDS[1][_inv == g][0])
                            for g in range(len(_uniq))]))
    V.check("repeats use different fold assignments", _agree < 0.5, f"{100 * _agree:.0f}% images same fold id",
            "< 50%")

ARRAYS_META = {"n": {s: len(TOKENS[s]) for s in TOKENS}, "max_text_len": CFG.max_text_len,
               "pad_id": PAD_ID, "cls_id": CLS_ID, "sep_id": SEP_ID, "order_hash": ORDER_HASH,
               "length_stats": LENGTH_STATS, "geometry_dim": GEOMETRY_DIM,
               "folds": {"k": KD_K, "repeats": KD_R, "seed": int(CFG.kd_fold_seed), "grouped_by": "image",
                         "sizes": _fsz.tolist()}}
(AD / "meta.json").write_text(json.dumps(ARRAYS_META, indent=2))

# ──────────────────────────────────────────────────────────────────────────────
# 5.5  VERIFY BY READING BACK (exactly what a worker will read)
# ──────────────────────────────────────────────────────────────────────────────
print("\n5.5  VERIFY\n")
_ok = True
for split in ("train", "val", "test"):
    flat = np.load(AD / f"{split}_ids_flat.npy")
    off = np.load(AD / f"{split}_ids_off.npy")
    lab = np.load(AD / f"{split}_labels.npy")
    geo = np.load(AD / f"{split}_geo.npy")
    z = np.load(AD / f"{split}_index.npz")
    n = len(lab)
    _ok &= V.check(f"[{split}] rows consistent", len(off) == n + 1 and len(geo) == n
                   and len(z["paths"]) == n and len(z["boxes"]) == n, n, "all equal")
    i = n // 2
    _ok &= V.check(f"[{split}] row {i} round-trips",
                   flat[off[i]:off[i + 1]].tolist() == list(TOKENS[split][i]), "match", "match")
    V.check(f"[{split}] token ids inside the vocabulary", int(flat.max()) < len(QTOK),
            int(flat.max()), f"< {len(QTOK)}")
    V.check(f"[{split}] labels binary and non-empty",
            bool(np.isin(lab, (0, 1)).all() and (lab.sum(1) >= 1).all()), "yes", "yes")
tl = np.load(AD / "val_labels_tl.npy")
V.check("test-like val labels present", tl.shape == np.load(AD / "val_labels.npy").shape,
        tl.shape, "same as val labels")
V.check("protocol rho present (Block 03)", (AD / "protocol_rho.npy").is_file(), "yes", "yes")
_fr = np.load(AD / "train_folds.npy")
V.check("fold file reads back", _fr.shape == (KD_R, len(df_train)) and np.array_equal(_fr, TRAIN_FOLDS),
        _fr.shape, (KD_R, len(df_train)))
V.note("arrays on disk", f"{dir_size_gb(AD) * 1024:.0f} MB in {AD}")
free_memory()
BLOCK_05_OK = V.finish()


In [ ]:
# Block 06 — Core Library, CPU Self-Test & GPU Pre-flight
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–05
#
# 6.1  Writes the training/evaluation library to /kaggle/working/emotic_core.py.
#      GPU workers (Blocks 07–09) are separate processes that run this file and
#      read only files on disk, never notebook variables.
# 6.2  Metric, LR-schedule, batching and initialisation checks (in the notebook):
#      length-bucketed batches cover every row once, resume at any batch, and
#      cut the padded text length (measured on the real train descriptions);
#      two seeds of one configuration start from IDENTICAL weights (soups).
# 6.3  The worker reads exactly the arrays Blocks 03–05 wrote (incl. folds).
# 6.4  CPU self-test of the worker on a tiny synthetic problem with every v7
#      component on (image + ROI + EVA crop + SigLIP crop + geometry; VAD head
#      and token dropout exercised too):
#        T1 reference                  T2 pause mid-epoch + resume = bit-identical
#        T3 fp16 overflow → fp32       T4 STOP file pause + resume = bit-identical
#        T5 session-deadline pause     T6 crash during finalisation → finalise only
#        T7 re-scoring fallback        T8 pre-flight command (SigLIP stream incl.)
#        T9 plateau run extended       T10 early stopping
#        T11 CUDA-OOM recovery         T12 two fold TEACHERS: out-of-fold train
#                                          probabilities, no test scores, weights freed
#        T13 STUDENT on the assembled soft targets
#        T14 SOUP of two seeds (uniform + greedy on val)   T15 soup of incompatible
#                                                            runs is refused
# 6.5  GPU pre-flight: the real 186 M-parameter Q-Former with all streams (incl.
#      SigLIP tokens) on the LONGEST batch (320 tokens), on GPU 0. Picks the
#      largest micro-batch that fits with 1.5 GB headroom.
# ══════════════════════════════════════════════════════════════════════════════

import hashlib
import importlib
import importlib.util
import tempfile
import numpy as np

header("BLOCK 06 — Core Library, CPU Self-Test & GPU Pre-flight  [v7]")
V = BlockVerdict(6, "Core & Self-test")
TEST_MODE = os.environ.get("EMOTIC_TEST_MODE") == "1"

CORE_SRC = r'''"""EMOTIC core library v7 (shared by the Kaggle notebook and the GPU workers).

The notebook imports this module for metrics and self-tests. Worker processes
run it directly:

    python emotic_core.py extract   --manifest <json>  cache frozen features (3 views)
    python emotic_core.py train     --spec <json>      train one run (resumable)
    python emotic_core.py evaluate  --spec <json>      score best.pt (val+test, or OOF)
    python emotic_core.py preflight --spec <json>      GPU memory at the worst-case batch
    python emotic_core.py soup      --spec <json>      weight-average finished seeds

A worker never depends on notebook state: everything it needs is on disk
(arrays written by Block 05, features written by the extract command).

v7 additions
  * SigLIP-2 person-crop stream: a second frozen encoder (so400m, 384 px) sees
    the person at 3x the pixel density of the 224 px views; 1 pooled + 9x9
    tokens, projected into the Q-Former's cross-attention like the other streams.
  * Length-bucketed batches: the same rows per epoch, grouped by description
    length inside seeded chunks, so little compute goes into padding.
    Deterministic per (seed, epoch) and resumable at any batch.
  * Shared initialisation (init_seed): every seed starts from identical
    weights and differs only in data order and dropout, so finished seeds can
    be weight-averaged into one network ("model soup", Wortsman et al. 2022).
  * Cross-fitted self-distillation: fold teachers are trained on 4/5 of the
    train persons (folds grouped by image) and predict the held-out fifth; the
    out-of-fold probabilities, averaged over repeats, become soft targets
    (1 - a) * label + a * teacher for student runs. Teachers never see test.
"""
from __future__ import annotations

import argparse
import copy
import csv
import ctypes
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import signal
import sys
import threading
import time
import traceback
import warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

CORE_VERSION = "2026.10.01-v7.0"

EMOTION_LABELS = [
    "Affection", "Anger", "Annoyance", "Anticipation", "Aversion",
    "Confidence", "Disapproval", "Disconnection", "Disquietment",
    "Doubt/Confusion", "Embarrassment", "Engagement", "Esteem",
    "Excitement", "Fatigue", "Fear", "Happiness", "Pain", "Peace",
    "Pleasure", "Sadness", "Sensitivity", "Suffering", "Surprise",
    "Sympathy", "Yearning",
]
NUM_CLASSES = len(EMOTION_LABELS)

G, R, Y, B, D, Z = "\033[92m", "\033[91m", "\033[93m", "\033[94m", "\033[90m", "\033[0m"


def log(msg=""):
    print(msg, flush=True)


# ══════════════════════════════════════════════════════════════════════════════
#  Small I/O helpers
# ══════════════════════════════════════════════════════════════════════════════
def write_json(path, obj):
    """Atomic JSON write: a crash can never leave a half-written file."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=_json_default))
    os.replace(tmp, path)


def _json_default(o):
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)


def read_json(path, default=None):
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return default


def free_gb(path):
    try:
        return shutil.disk_usage(str(path)).free / 1e9
    except Exception:
        return float("inf")


def atomic_torch_save(obj, path):
    import torch
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    with open(tmp, "rb") as f:
        os.fsync(f.fileno())
    os.replace(tmp, path)
    return path


def atomic_np_save(arr, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)
    return path


def array_sha(a):
    """Short content hash of an array (part of run signatures)."""
    return hashlib.sha1(np.ascontiguousarray(a).tobytes()).hexdigest()[:12]


def quiet_transformers():
    """transformers 5 prints per-tensor progress and load reports; keep worker logs clean."""
    try:
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
        hf_logging.disable_progress_bar()
    except Exception:
        pass
    try:
        from huggingface_hub.utils import disable_progress_bars
        disable_progress_bars()
    except Exception:
        pass


def resolve_file(repo, filename):
    """A file from a local model directory or from the Hugging Face hub."""
    p = Path(str(repo))
    if p.is_dir():
        f = p / filename
        if not f.exists():
            raise FileNotFoundError(str(f))
        return str(f)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(str(repo), filename)


def cuda_cleanup():
    """Return cached blocks to the driver; call between phases, never inside a step."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


# ══════════════════════════════════════════════════════════════════════════════
#  Metrics  (numpy / sklearn — identical to the notebook's reporting)
# ══════════════════════════════════════════════════════════════════════════════
def per_class_ap(scores, labels):
    """AP per class; NaN where a class has no positives (or no negatives)."""
    from sklearn.metrics import average_precision_score
    out = np.full(labels.shape[1], np.nan, np.float64)
    for c in range(labels.shape[1]):
        p = labels[:, c].sum()
        if 0 < p < len(labels):
            out[c] = average_precision_score(labels[:, c], scores[:, c])
    return out


def compute_map(scores, labels):
    ap = per_class_ap(scores, labels)
    return float(np.nanmean(ap) * 100.0) if np.isfinite(ap).any() else 0.0


def prevalence_floor(labels):
    """Random-ranker mAP: a random ranking scores AP equal to class prevalence."""
    prev = labels.mean(0)
    v = (prev > 0) & (prev < 1)
    return float(prev[v].mean() * 100.0) if v.any() else 0.0


def weighted_ap(scores_c, y_c, W):
    """
    AP of one class under per-person weights W [B, n] (bootstrap resample
    counts; a row of ones is the plain AP). Ties are grouped exactly as in
    sklearn: every person in a tie group gets the precision measured at the
    END of the group.
    """
    o = np.argsort(-scores_c, kind="mergesort")
    s = np.asarray(scores_c)[o]
    y = np.asarray(y_c, np.float64)[o]
    w = np.asarray(W, np.float64)[:, o]
    tp = np.cumsum(w * y, 1)
    fp = np.cumsum(w * (1.0 - y), 1)
    ends = np.r_[np.nonzero(np.diff(s))[0], len(s) - 1]
    grp_end = ends[np.searchsorted(ends, np.arange(len(s)))]
    prec = tp[:, grp_end] / np.maximum(tp[:, grp_end] + fp[:, grp_end], 1e-12)
    npos = (w * y).sum(1)
    ap = (prec * w * y).sum(1) / np.maximum(npos, 1e-12)
    return np.where(npos > 0, ap, np.nan)


def bootstrap_map(scores, labels, B=1000, seed=0):
    """mAP of B bootstrap resamples of the persons. The same seed gives the same
    resamples, so differences between two models are PAIRED."""
    n, C = labels.shape
    rs = np.random.RandomState(seed)
    W = rs.multinomial(n, np.full(n, 1.0 / n), size=B).astype(np.float64)
    valid = labels.sum(0) > 0
    out = np.zeros((B, C))
    for c in range(C):
        out[:, c] = weighted_ap(scores[:, c], labels[:, c], W) if valid[c] else np.nan
    return np.nanmean(out[:, valid], 1) * 100.0


def exact_map(scores, labels):
    """Same code path as the bootstrap with unit weights (checked against sklearn)."""
    n, C = labels.shape
    W = np.ones((1, n))
    aps = [weighted_ap(scores[:, c], labels[:, c], W)[0] for c in range(C) if labels[:, c].sum() > 0]
    return float(np.mean(aps) * 100.0)


def optimal_thresholds(scores, labels, min_t=0.02, max_t=0.98):
    """Per-class F1-optimal threshold (validation only; mAP is threshold-free)."""
    C = scores.shape[1]
    th = np.full(C, 0.5, np.float32)
    for c in range(C):
        y = labels[:, c]
        npos = y.sum()
        if npos == 0:
            continue
        order = np.argsort(-scores[:, c])
        ys, ss = y[order], scores[:, c][order]
        tp = np.cumsum(ys)
        fp = np.cumsum(1.0 - ys)
        f1 = 2.0 * tp / np.maximum(2.0 * tp + fp + (npos - tp), 1e-9)
        k = int(np.argmax(f1))
        th[c] = float(np.clip(ss[k], min_t, max_t))
    return th


def precision_recall_at_k(scores, labels, k):
    k = min(k, scores.shape[1])
    idx = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    hits = np.take_along_axis(labels, idx, axis=1).sum(1)
    npos = labels.sum(1)
    p = float(np.mean(hits / k))
    r = float(np.mean(np.divide(hits, npos, out=np.zeros_like(hits, dtype=np.float64),
                                where=npos > 0)))
    return p, r


def evaluate_scores(scores, labels, thresholds=None, ranking=True):
    """Full metric bundle. mAP and lift are the headline pair."""
    from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                                 hamming_loss, jaccard_score, coverage_error,
                                 label_ranking_loss)
    scores = np.asarray(scores, np.float64)
    labels = np.asarray(labels, np.float64)
    ap = per_class_ap(scores, labels)
    floor = prevalence_floor(labels)
    m = float(np.nanmean(ap) * 100.0)
    th = (np.full(scores.shape[1], 0.5, np.float32) if thresholds is None
          else np.asarray(thresholds, np.float32))
    preds = (scores >= th[None, :]).astype(np.int32)
    lab_i = labels.astype(np.int32)

    aucs = []
    for c in range(labels.shape[1]):
        p = labels[:, c].sum()
        if 0 < p < len(labels):
            try:
                aucs.append(roc_auc_score(labels[:, c], scores[:, c]))
            except Exception:
                pass
    prev = labels.mean(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        ap_over_prev = ap / np.where(prev > 0, prev, np.nan)

    r = {
        "mAP": m, "floor": floor, "lift": (m / floor if floor > 1e-9 else 0.0),
        "AUC": float(np.mean(aucs)) if aucs else 0.5,
        "micro_f1": float(f1_score(lab_i, preds, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(lab_i, preds, average="macro", zero_division=0)),
        "macro_prec": float(precision_score(lab_i, preds, average="macro", zero_division=0)),
        "macro_rec": float(recall_score(lab_i, preds, average="macro", zero_division=0)),
        "jaccard": float(jaccard_score(lab_i, preds, average="samples", zero_division=0)),
        "hamming": float(hamming_loss(lab_i, preds)),
        "n": int(len(labels)), "density": float(labels.sum(1).mean()),
        "pred_density": float(preds.sum(1).mean()),
        "per_class_ap": ap, "per_class_prev": prev, "ap_over_prev": ap_over_prev,
        "classes_below_floor": int(np.nansum(ap_over_prev < 1.0)),
        "thresholds": th,
    }
    if ranking:
        try:
            r["ranking_loss"] = float(label_ranking_loss(lab_i, scores))
            r["coverage"] = float(coverage_error(lab_i, scores))
        except Exception:
            r["ranking_loss"], r["coverage"] = float("nan"), float("nan")
        top1 = np.argmax(scores, 1)
        r["one_error"] = float(1.0 - np.mean(labels[np.arange(len(labels)), top1]))
        for k in (1, 3, 5):
            p, rec = precision_recall_at_k(scores, labels, k)
            r[f"P@{k}"], r[f"R@{k}"] = p, rec
    return r


def scalar_metrics(r):
    """Drop arrays so a metric dict can go into JSON."""
    return {k: float(v) for k, v in r.items()
            if isinstance(v, (int, float, np.floating, np.integer))}


# ══════════════════════════════════════════════════════════════════════════════
#  Run configuration
# ══════════════════════════════════════════════════════════════════════════════
DEFAULTS = {
    "name": "run", "config_name": "run", "seed": 1, "role": "run",
    # every seed starts from the same weights (init_seed); `seed` drives data
    # order and dropout only — this is what makes seeds weight-averageable
    "init_seed": 0,
    # input streams
    "use_roi": False, "use_geo": False, "use_crop": False, "use_sig": False,
    # regularisers (the v6 C2 components; off in every v7 configuration)
    "use_vad": False, "vad_weight": 0.2, "vis_token_drop": 0.0, "txt_token_drop": 0.0,
    # cross-fitted self-distillation
    "kd_path": None, "kd_alpha": 0.5, "fold_file": None, "fold_repeat": None, "fold": None,
    # length-bucketed batches (chunk = batches per sorting chunk)
    "bucket": True, "bucket_chunk": 50,
    # optimisation (Xenos et al.: lr 1e-4, backbone x0.1, batch 64)
    "batch_size": 32, "grad_accum": 2, "eval_batch_size": 64,
    "lr": 1e-4, "backbone_lr_mult": 0.1, "weight_decay": 0.05, "grad_clip": 1.0,
    # epochs: "cosine" decays to 0 over max_epochs (default); "plateau" never
    # looks at max_epochs, so only plateau runs can be extended later.
    "lr_schedule": "cosine", "max_epochs": 8, "min_epochs": 5, "patience": 3,
    "min_delta": 0.01, "warmup_epochs": 1.0,
    "plateau_factor": 0.5, "plateau_patience": 4, "plateau_min_scale": 0.03125,
    "keep_extendable": True, "floor_check_epoch": 2, "floor_min_lift": 1.5,
    "qformer_dropout": 0.4, "head_dropout": 0.1, "roi_grid": 7,
    "ema_decay": 0.998, "amp": True, "num_workers": 2, "log_every": 100,
    "max_oom_per_epoch": 20,
    # paths / control (filled by the scheduler)
    "run_dir": None, "array_dir": None, "feat_dir": None, "hf_config_dir": None,
    "init_path": None, "control_dir": None, "deadline": None, "parent_pid": None,
    # self-test hooks (always 0 in real runs)
    "test_inject_nonfinite_every": 0, "test_stop_after_steps": 0, "test_inject_oom_every": 0,
}


def merged_config(spec):
    cfg = dict(DEFAULTS)
    cfg.update({k: v for k, v in spec.items() if v is not None or k not in cfg})
    return cfg


def run_signature(cfg, steps_per_epoch, warmup_steps, crop_tokens=0, sig_tokens=0, n_train=0, extra=""):
    """
    Everything that shapes the trajectory must match on resume. For the cosine
    schedule that includes max_epochs (the LR curve depends on it); for the
    plateau schedule it does not, so plateau runs can be given more epochs.
    `extra` carries the content hash of the fold file (teachers) or of the
    soft targets (students).
    """
    sched = (f"cos{cfg['max_epochs']}" if cfg["lr_schedule"] == "cosine" else
             f"pl{cfg['plateau_factor']}:{cfg['plateau_patience']}:{cfg['plateau_min_scale']}")
    return (f"v7|{sched}|bs{cfg['batch_size']}x{cfg['grad_accum']}|spe{steps_per_epoch}|w{warmup_steps}|"
            f"n{n_train}|roi{int(cfg['use_roi'])}|geo{int(cfg['use_geo'])}|crop{int(cfg['use_crop'])}:{crop_tokens}|"
            f"sig{int(cfg['use_sig'])}:{sig_tokens}|vad{int(cfg['use_vad'])}:{cfg['vad_weight']}|"
            f"td{cfg['vis_token_drop']}:{cfg['txt_token_drop']}|bk{int(bool(cfg['bucket']))}:{cfg['bucket_chunk']}|"
            f"s{cfg['seed']}|i{cfg['init_seed']}|lr{cfg['lr']}|wd{cfg['weight_decay']}|"
            f"d{cfg['qformer_dropout']}|ema{cfg['ema_decay']}|{extra}")


def soup_key(signature):
    """A run signature with the seed removed: runs that may be weight-averaged share it."""
    return re.sub(r"\|s\d+\|", "|s*|", str(signature))


# ══════════════════════════════════════════════════════════════════════════════
#  Process control: stop files, signals, deadline, parent death
# ══════════════════════════════════════════════════════════════════════════════
class Control:
    def __init__(self, cfg):
        self.flag = threading.Event()
        self.reason = None
        self.name = cfg["name"]
        self.control_dir = Path(cfg["control_dir"]) if cfg.get("control_dir") else None
        self.deadline = float(cfg["deadline"]) if cfg.get("deadline") else None
        self.parent_pid = int(cfg["parent_pid"]) if cfg.get("parent_pid") else None

    def install(self):
        for sig in (signal.SIGTERM, signal.SIGINT):
            try:
                signal.signal(sig, self._on_signal)
            except Exception:
                pass
        try:  # ask the kernel to SIGTERM us if the notebook process dies
            libc = ctypes.CDLL("libc.so.6", use_errno=True)
            libc.prctl(1, signal.SIGTERM, 0, 0, 0)   # PR_SET_PDEATHSIG
        except Exception:
            pass
        if self.parent_pid:
            threading.Thread(target=self._watch_parent, daemon=True).start()

    def _on_signal(self, signum, frame):
        self.request(f"signal {signum}")

    def _watch_parent(self):
        while not self.flag.is_set():
            if os.getppid() != self.parent_pid:
                self.request("notebook process ended")
                return
            time.sleep(15)

    def request(self, reason):
        if not self.flag.is_set():
            self.reason = reason
            self.flag.set()

    def check(self):
        if self.flag.is_set():
            return self.reason
        if self.control_dir is not None:
            if (self.control_dir / "STOP").exists():
                self.request("STOP file")
            elif (self.control_dir / f"STOP_{self.name}").exists():
                self.request(f"STOP_{self.name} file")
        if self.deadline is not None and time.time() > self.deadline:
            self.request("session deadline")
        return self.reason if self.flag.is_set() else None


def set_status(run_dir, status, **extra):
    d = read_json(Path(run_dir) / "status.json", {}) or {}
    d.update(extra)
    d.update({"status": status, "updated": time.time(), "pid": os.getpid()})
    write_json(Path(run_dir) / "status.json", d)


# ══════════════════════════════════════════════════════════════════════════════
#  Data: cached features (EVA image, EVA crop, SigLIP crop) + tokenised text
# ══════════════════════════════════════════════════════════════════════════════
class SplitArrays:
    """Everything a worker needs for one split, loaded from disk."""

    def __init__(self, array_dir, split, feat_dir=None, need_crop=False, need_sig=False):
        a = Path(array_dir)
        self.split = split
        self.ids_flat = np.load(a / f"{split}_ids_flat.npy")
        self.ids_off = np.load(a / f"{split}_ids_off.npy")
        self.lengths = np.diff(self.ids_off).astype(np.int64)
        self.labels = np.load(a / f"{split}_labels.npy").astype(np.float32)
        tl = a / f"{split}_labels_tl.npy"
        self.labels_tl = np.load(tl).astype(np.float32) if tl.exists() else None
        self.geo = np.load(a / f"{split}_geo.npy").astype(np.float32)
        vp = a / f"{split}_vad.npy"
        self.n = int(len(self.labels))
        self.vad = (np.load(vp).astype(np.float32) if vp.exists()
                    else np.full((self.n, 3), 0.5, np.float32))
        assert len(self.ids_off) == self.n + 1, f"{split}: ids_off length mismatch"
        assert self.labels.shape[1] == NUM_CLASSES, f"{split}: label width"
        assert self.vad.shape == (self.n, 3), f"{split}: vad shape"
        self.feat_path, self.feat_shape, self.roi = None, None, None
        self.crop_path, self.crop_shape = None, None
        self.sig_path, self.sig_shape = None, None
        if feat_dir is not None:
            f = Path(feat_dir)
            self.feat_path = str(f / f"{split}_feats.npy")
            mm = np.load(self.feat_path, mmap_mode="r")
            self.feat_shape = tuple(mm.shape)
            del mm
            self.roi = np.load(f / f"{split}_roi.npy").astype(np.float32)
            assert self.feat_shape[0] == self.n, f"{split}: feature rows {self.feat_shape[0]} != {self.n}"
            assert len(self.roi) == self.n, f"{split}: roi rows"
            for view, attr in (("crop", "crop"), ("sig", "sig")):
                p = f / f"{split}_{view}.npy"
                if p.exists():
                    mm = np.load(str(p), mmap_mode="r")
                    setattr(self, f"{attr}_path", str(p))
                    setattr(self, f"{attr}_shape", tuple(mm.shape))
                    del mm
                    assert getattr(self, f"{attr}_shape")[0] == self.n, f"{split}: {view} rows"
        if need_crop and self.crop_path is None:
            raise FileNotFoundError(f"{split}: person-crop features missing in {feat_dir}")
        if need_sig and self.sig_path is None:
            raise FileNotFoundError(f"{split}: SigLIP person-crop features missing in {feat_dir}")


def _torch_dataset_base():
    import torch
    return torch.utils.data.Dataset


class FeatureDataset(_torch_dataset_base()):
    def __init__(self, arr, use_crop=False, use_sig=False, kd=None):
        self.a = arr
        self.use_crop, self.use_sig = bool(use_crop), bool(use_sig)
        self.kd = kd
        self._mm = self._mc = self._ms = None

    def __len__(self):
        return self.a.n

    def __getitem__(self, i):
        if self._mm is None:            # opened lazily, once per loader worker
            self._mm = np.load(self.a.feat_path, mmap_mode="r")
            if self.use_crop:
                self._mc = np.load(self.a.crop_path, mmap_mode="r")
            if self.use_sig:
                self._ms = np.load(self.a.sig_path, mmap_mode="r")
        s, e = int(self.a.ids_off[i]), int(self.a.ids_off[i + 1])
        item = {"vis": np.array(self._mm[i], dtype=np.float16, copy=True),
                "ids": self.a.ids_flat[s:e].astype(np.int64), "labels": self.a.labels[i],
                "roi": self.a.roi[i], "geo": self.a.geo[i], "vad": self.a.vad[i], "idx": int(i)}
        if self.use_crop:
            item["crop"] = np.array(self._mc[i], dtype=np.float16, copy=True)
        if self.use_sig:
            item["sig"] = np.array(self._ms[i], dtype=np.float16, copy=True)
        if self.kd is not None:
            item["kd"] = self.kd[i]
        return item


def collate(batch):
    import torch
    n = len(batch)
    lens = [len(b["ids"]) for b in batch]
    L = max(lens)
    ids = np.zeros((n, L), dtype=np.int64)
    mask = np.zeros((n, L), dtype=np.int64)
    for j, b in enumerate(batch):
        ids[j, :lens[j]] = b["ids"]
        mask[j, :lens[j]] = 1
    out = {
        "vis": torch.from_numpy(np.stack([b["vis"] for b in batch])),
        "input_ids": torch.from_numpy(ids),
        "attn_mask": torch.from_numpy(mask),
        "labels": torch.from_numpy(np.stack([b["labels"] for b in batch]).astype(np.float32)),
        "roi": torch.from_numpy(np.stack([b["roi"] for b in batch]).astype(np.float32)),
        "geo": torch.from_numpy(np.stack([b["geo"] for b in batch]).astype(np.float32)),
        "vad": torch.from_numpy(np.stack([b["vad"] for b in batch]).astype(np.float32)),
        "idx": torch.as_tensor([b["idx"] for b in batch], dtype=torch.long),
    }
    for k in ("crop", "sig"):
        if k in batch[0]:
            out[k] = torch.from_numpy(np.stack([b[k] for b in batch]))
    if "kd" in batch[0]:
        out["kd"] = torch.from_numpy(np.stack([b["kd"] for b in batch]).astype(np.float32))
    return out


def _torch_sampler_base():
    import torch
    return torch.utils.data.Sampler


class BucketBatchSampler(_torch_sampler_base()):
    """
    The batches of one epoch over the rows in `indices`, fixed by (seed, epoch):
    a seeded permutation, cut into chunks of `chunk` batches; inside a chunk the
    rows are sorted by description length and cut into batches; finally the
    batch order is shuffled. Every row appears at most once, the last partial
    batch is dropped (as drop_last=True did), and `start` skips the batches
    already done, so a resumed epoch continues with exactly the remaining ones.
    bucket=False gives plain shuffled batches.
    """

    def __init__(self, indices, lengths, batch_size, seed, epoch, start=0, bucket=True, chunk=50):
        idx = np.asarray(indices, dtype=np.int64)
        bs = int(batch_size)
        rs = np.random.RandomState((int(seed) * 100003 + int(epoch) * 7919 + 17) % (2 ** 32 - 1))
        perm = idx[rs.permutation(len(idx))]
        nb = len(perm) // bs
        perm = perm[: nb * bs]
        if bucket and nb > 1:
            lens = np.asarray(lengths)
            step = bs * max(1, int(chunk))
            parts = []
            for s in range(0, len(perm), step):
                c = perm[s:s + step]
                parts.append(c[np.argsort(lens[c], kind="mergesort")].reshape(-1, bs))
            batches = np.concatenate(parts, 0)
            batches = batches[rs.permutation(len(batches))]
        else:
            batches = perm.reshape(nb, bs)
        self.batches = batches[int(start):]

    def __iter__(self):
        return iter([b.tolist() for b in self.batches])

    def __len__(self):
        return len(self.batches)


def eval_batches(indices, lengths, batch_size):
    """Prediction batches: longest descriptions first (an OOM shows at once), little padding."""
    idx = np.asarray(indices, dtype=np.int64)
    o = idx[np.argsort(-np.asarray(lengths)[idx], kind="mergesort")]
    bs = int(batch_size)
    return [o[i:i + bs].tolist() for i in range(0, len(o), bs)]


def make_loader(ds, batch_sampler, num_workers=0, pin=False):
    import torch
    kw = dict(batch_sampler=batch_sampler, num_workers=num_workers, pin_memory=pin,
              collate_fn=collate, persistent_workers=False,
              # a private generator so building a loader never consumes the
              # global RNG — this is what makes mid-epoch resume bit-exact
              generator=torch.Generator().manual_seed(0))
    if num_workers > 0:
        kw["prefetch_factor"] = 4
    return torch.utils.data.DataLoader(ds, **kw)


BATCH_KEYS = ("vis", "crop", "sig", "input_ids", "attn_mask", "labels", "roi", "geo", "vad", "kd")


def _to_device(batch, device):
    nb = device.type == "cuda"
    return {k: batch[k].to(device, non_blocking=nb) for k in BATCH_KEYS if k in batch}


def _slice_batch(t, a, b):
    return {k: v[a:b] for k, v in t.items()}


# ══════════════════════════════════════════════════════════════════════════════
#  Model
# ══════════════════════════════════════════════════════════════════════════════
def build_qformer(hf_config_dir, dropout):
    from transformers import InstructBlipConfig
    from transformers.models.instructblip.modeling_instructblip import InstructBlipQFormerModel
    cfg = InstructBlipConfig.from_pretrained(str(hf_config_dir))
    q = cfg.qformer_config
    q.hidden_dropout_prob = float(dropout)
    q.attention_probs_dropout_prob = float(dropout)
    return InstructBlipQFormerModel(q), cfg


def _nn():
    import torch.nn as nn
    return nn


STREAM_ORDER = ("image", "roi", "crop", "sig", "geo")


class EmotionQFormer(_nn().Module):
    """
    Encoder tokens seen by the Q-Former's cross-attention (each stream gets a
    learned type embedding):
        image  257 cached EVA-ViT-g tokens of the full image with the red box
        roi     49 tokens roi-aligned from the image grid at the person box   (use_roi)
        crop    65 EVA tokens of a square PERSON CROP at 224 px, 2x2-pooled   (use_crop)
        sig     82 SigLIP-2 tokens of the same crop at 384 px: pooled + 9x9   (use_sig)
                   (Linear 1152 -> 1408 + LayerNorm initialised from EVA's
                   output LayerNorm, so they enter with EVA-like statistics)
        geo      1 token from 9-D box geometry                                (use_geo)
    Text: the LLaVA description of the person. 32 learned queries -> mean ->
    dropout -> Linear (fp32) -> 26 logits, plus an optional VAD head (use_vad).
    """

    def __init__(self, qformer, num_queries, hidden, vis_dim, geo_dim, use_roi=False, use_geo=False,
                 use_crop=False, use_sig=False, sig_dim=0, use_vad=False, head_dropout=0.1, roi_grid=7,
                 vis_token_drop=0.0, txt_token_drop=0.0, num_classes=NUM_CLASSES):
        import torch
        import torch.nn as nn
        super().__init__()
        self.qformer = qformer
        self.num_queries = int(num_queries)
        self.query_tokens = nn.Parameter(torch.zeros(1, self.num_queries, hidden))
        self.use_roi, self.use_geo, self.use_crop = bool(use_roi), bool(use_geo), bool(use_crop)
        self.use_sig = bool(use_sig)
        self.roi_grid = int(roi_grid)
        self.vis_token_drop, self.txt_token_drop = float(vis_token_drop), float(txt_token_drop)
        on = {"image": True, "roi": self.use_roi, "crop": self.use_crop, "sig": self.use_sig, "geo": self.use_geo}
        self.streams = [s for s in STREAM_ORDER if on[s]]
        if len(self.streams) > 1:
            self.type_emb = nn.Parameter(torch.zeros(len(self.streams), vis_dim))
            nn.init.normal_(self.type_emb, std=0.02)
        else:
            self.type_emb = None
        self.geo_proj = (nn.Sequential(nn.Linear(geo_dim, 256), nn.GELU(),
                                       nn.Linear(256, vis_dim), nn.LayerNorm(vis_dim))
                         if self.use_geo else None)
        if self.use_sig:
            assert int(sig_dim) > 0, "sig_dim required when use_sig=True"
            self.sig_proj = nn.Sequential(nn.Linear(int(sig_dim), vis_dim), nn.LayerNorm(vis_dim))
        else:
            self.sig_proj = None
        self.dropout = nn.Dropout(head_dropout)
        self.classifier = nn.Linear(hidden, num_classes)
        # The low starting sigmoid (~0.12) comes from the bias alone. A zero
        # weight would also start low but would block every upstream gradient
        # on the first step, so the weight is small and random instead.
        nn.init.normal_(self.classifier.weight, std=0.01)
        nn.init.constant_(self.classifier.bias, -2.0)
        self.vad_head = nn.Linear(hidden, 3) if use_vad else None
        if self.vad_head is not None:
            nn.init.normal_(self.vad_head.weight, std=0.01)
            nn.init.zeros_(self.vad_head.bias)

    def roi_tokens(self, vis, roi):
        import torch
        from torchvision.ops import roi_align
        n, p, c = vis.size(0), vis.size(1) - 1, vis.size(2)
        g = int(math.isqrt(p))
        assert g * g == p, f"patch tokens {p} are not a square grid"
        fmap = vis[:, 1:, :].transpose(1, 2).reshape(n, c, g, g)
        box = roi.to(fmap.device, torch.float32).clone() * float(g)
        box[:, 2] = torch.maximum(box[:, 2], box[:, 0] + 1e-2)
        box[:, 3] = torch.maximum(box[:, 3], box[:, 1] + 1e-2)
        idx = torch.arange(n, device=fmap.device, dtype=torch.float32).unsqueeze(1)
        with torch.autocast(device_type=fmap.device.type, enabled=False):
            pooled = roi_align(fmap.float(), torch.cat([idx, box], 1),
                               output_size=(self.roi_grid, self.roi_grid),
                               spatial_scale=1.0, sampling_ratio=2, aligned=True)
        return pooled.flatten(2).transpose(1, 2)

    def encoder_tokens(self, vis, roi=None, geo=None, crop=None, sig=None):
        """Returns (tokens [B,T,D], droppable [T] bool: patch tokens of image/crop/sig)."""
        import torch
        vis = vis.float()
        parts, flags = [], []

        def add(x, name, droppable_patches):
            t = self.streams.index(name)
            parts.append(x + self.type_emb[t] if self.type_emb is not None else x)
            f = torch.zeros(x.size(1), dtype=torch.bool, device=x.device)
            if droppable_patches:
                f[1:] = True                       # keep each stream's CLS / pooled token
            flags.append(f)

        add(vis, "image", True)
        if self.use_roi:
            if roi is None:
                roi = torch.tensor([[0.0, 0.0, 1.0, 1.0]], device=vis.device).expand(vis.size(0), 4)
            add(self.roi_tokens(vis, roi), "roi", False)
        if self.use_crop:
            if crop is None:
                raise ValueError("person-crop features required when use_crop=True")
            add(crop.to(vis.device).float(), "crop", True)
        if self.use_sig:
            if sig is None:
                raise ValueError("SigLIP features required when use_sig=True")
            add(self.sig_proj(sig.to(vis.device).float()).float(), "sig", True)
        if self.use_geo:
            if geo is None:
                raise ValueError("geometry features required when use_geo=True")
            add(self.geo_proj(geo.to(vis.device).float()).float().unsqueeze(1), "geo", False)
        return torch.cat(parts, dim=1), torch.cat(flags)

    def forward(self, vis, input_ids, attn_mask, roi=None, geo=None, crop=None, sig=None, return_vad=False):
        import torch
        n = vis.size(0)
        enc, droppable = self.encoder_tokens(vis, roi, geo, crop, sig)
        enc_mask = torch.ones(enc.shape[:2], dtype=torch.long, device=enc.device)
        if self.training and self.vis_token_drop > 0:
            drop = (torch.rand(enc.shape[:2], device=enc.device) < self.vis_token_drop) & droppable[None]
            enc_mask = enc_mask.masked_fill(drop, 0)
        am = attn_mask
        if self.training and self.txt_token_drop > 0:
            L = am.sum(1, keepdim=True)
            pos = torch.arange(am.size(1), device=am.device)[None]
            inner = (pos >= 1) & (pos < L - 1)                        # never [CLS] / [SEP]
            drop = (torch.rand(am.shape, device=am.device) < self.txt_token_drop) & inner
            am = am.masked_fill(drop, 0)
        # The Q-Former prepends the query tokens to the text internally, so
        # the self-attention mask must span num_queries + text length.
        q_mask = torch.ones(n, self.num_queries, dtype=am.dtype, device=am.device)
        out = self.qformer(input_ids=input_ids,
                           attention_mask=torch.cat([q_mask, am], dim=1),
                           query_embeds=self.query_tokens.expand(n, -1, -1),
                           encoder_hidden_states=enc,
                           encoder_attention_mask=enc_mask)
        h = out.last_hidden_state[:, :self.num_queries, :].mean(dim=1)
        with torch.autocast(device_type=h.device.type, enabled=False):
            hf = self.dropout(h.float())
            logits = self.classifier(hf)             # fp32 logits: fewer score ties
            vad = self.vad_head(hf) if self.vad_head is not None else None
        return (logits, vad) if return_vad else logits


def build_model(cfg, vis_shape, geo_dim, sig_dim=0):
    """Construct the head around a Q-Former built from the saved HF config."""
    qformer, hfcfg = build_qformer(cfg["hf_config_dir"], cfg["qformer_dropout"])
    qcfg = hfcfg.qformer_config
    assert vis_shape[2] == qcfg.encoder_hidden_size, \
        f"feature width {vis_shape[2]} != Q-Former encoder width {qcfg.encoder_hidden_size}"
    nq = int(getattr(hfcfg, "num_query_tokens", 32))
    return EmotionQFormer(qformer, nq, qcfg.hidden_size, vis_shape[2], geo_dim,
                          use_roi=cfg["use_roi"], use_geo=cfg["use_geo"], use_crop=cfg["use_crop"],
                          use_sig=cfg["use_sig"], sig_dim=sig_dim, use_vad=cfg["use_vad"],
                          head_dropout=cfg["head_dropout"], roi_grid=cfg["roi_grid"],
                          vis_token_drop=cfg["vis_token_drop"], txt_token_drop=cfg["txt_token_drop"])


def load_pretrained_init(model, init_path):
    """Pretrained Q-Former + query tokens; the SigLIP projection's LayerNorm
    starts as EVA's output LayerNorm (vis_ln) when the init file has it."""
    import torch
    init = torch.load(init_path, map_location="cpu", weights_only=False)
    model.qformer.load_state_dict(init["qformer"], strict=True)
    qt = init["query_tokens"].float()
    assert qt.numel() == model.query_tokens.numel(), "query token count mismatch"
    with torch.no_grad():
        model.query_tokens.copy_(qt.reshape_as(model.query_tokens))
        ln = init.get("vis_ln")
        if getattr(model, "sig_proj", None) is not None and ln is not None:
            model.sig_proj[1].weight.copy_(ln["weight"].float())
            model.sig_proj[1].bias.copy_(ln["bias"].float())
    return len(init["qformer"])


def build_optimizer(model, cfg):
    """
    Pretrained Q-Former + query tokens at lr x backbone_lr_mult; the randomly
    initialised head / geometry / SigLIP projection / type embeddings at the
    full lr. Weight decay only on matrices (not biases, LayerNorms, embeddings).
    Groups are built as LISTS: a generator is consumed by any filter pass and
    AdamW then silently gets an empty group.
    """
    import torch
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        part = "qformer" if n.startswith(("qformer.", "query_tokens")) else "head"
        decay = p.ndim >= 2 and "embeddings" not in n and not n.startswith(("query_tokens", "type_emb"))
        groups.setdefault((part, decay), []).append(p)
    out = []
    for (part, decay), ps in sorted(groups.items()):
        out.append({"params": ps, "name": f"{part}_{'decay' if decay else 'nodecay'}",
                    "lr": cfg["lr"] * (cfg["backbone_lr_mult"] if part == "qformer" else 1.0),
                    "weight_decay": cfg["weight_decay"] if decay else 0.0})
    assert out and all(len(g["params"]) > 0 for g in out), "empty param group"
    return torch.optim.AdamW(out, betas=(0.9, 0.999), eps=1e-8)


class WarmupCosine:
    """Linear warm-up, then cosine decay to 0 at the end of max_epochs (per optimizer step)."""

    def __init__(self, opt, warmup_steps, total_steps):
        self.opt = opt
        self.base = [float(g["lr"]) for g in opt.param_groups]
        self.warmup_steps, self.total_steps = int(warmup_steps), int(total_steps)
        self.step_n = 0
        self._apply()

    @property
    def scale(self):
        s = self.step_n
        if s < self.warmup_steps:
            return (s + 1) / max(1, self.warmup_steps)
        prog = (s - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    def _apply(self):
        sc = self.scale
        for g, b in zip(self.opt.param_groups, self.base):
            g["lr"] = b * sc

    def step(self):
        self.step_n += 1
        self._apply()

    def epoch_end(self, metric):
        return False

    def state_dict(self):
        return {"kind": "cosine", "base": self.base, "warmup_steps": self.warmup_steps,
                "total_steps": self.total_steps, "step_n": self.step_n}

    def load_state_dict(self, d):
        self.base, self.step_n = list(d["base"]), int(d["step_n"])
        self._apply()


class WarmupPlateau:
    """
    Per optimizer step: linear warm-up over `warmup_steps`.
    Per epoch: LR x `factor` after `patience` epochs without a val-TL gain above
    `min_delta` (never below `min_scale`). No notion of total length, so runs
    using it can be extended later. Its whole state is saved in last.pt.
    """

    def __init__(self, opt, warmup_steps, factor, patience, min_scale, min_delta):
        self.opt = opt
        self.base = [float(g["lr"]) for g in opt.param_groups]
        self.warmup_steps = int(warmup_steps)
        self.factor, self.patience = float(factor), int(patience)
        self.min_scale, self.min_delta = float(min_scale), float(min_delta)
        self.step_n, self.plateau, self.bad, self.best, self.reductions = 0, 1.0, 0, None, 0
        self._apply()

    @property
    def scale(self):
        w = 1.0 if self.warmup_steps <= 0 else min(1.0, (self.step_n + 1) / self.warmup_steps)
        return self.plateau * w

    def _apply(self):
        sc = self.scale
        for g, b in zip(self.opt.param_groups, self.base):
            g["lr"] = b * sc

    def step(self):
        self.step_n += 1
        self._apply()

    def epoch_end(self, metric):
        """Returns True when the LR was reduced."""
        if self.best is None or metric > self.best + self.min_delta:
            self.best, self.bad = float(metric), 0
            return False
        self.bad += 1
        if self.bad >= self.patience and self.plateau > self.min_scale * (1 + 1e-9):
            self.plateau = max(self.min_scale, self.plateau * self.factor)
            self.bad = 0
            self.reductions += 1
            self._apply()
            return True
        return False

    def state_dict(self):
        return {"kind": "plateau", "base": self.base, "warmup_steps": self.warmup_steps,
                "step_n": self.step_n, "plateau": self.plateau, "bad": self.bad, "best": self.best,
                "reductions": self.reductions}

    def load_state_dict(self, d):
        self.base = list(d["base"])
        self.step_n, self.plateau, self.bad = int(d["step_n"]), float(d["plateau"]), int(d["bad"])
        self.best, self.reductions = d["best"], int(d["reductions"])
        self._apply()


def make_scheduler(cfg, opt, steps_per_epoch, warmup_steps):
    if cfg["lr_schedule"] == "cosine":
        return WarmupCosine(opt, warmup_steps, steps_per_epoch * int(cfg["max_epochs"]))
    if cfg["lr_schedule"] == "plateau":
        return WarmupPlateau(opt, warmup_steps, cfg["plateau_factor"], cfg["plateau_patience"],
                             cfg["plateau_min_scale"], cfg["min_delta"])
    raise ValueError(f"unknown lr_schedule {cfg['lr_schedule']}")


class EMA:
    """Exponential moving average of the weights, kept as a second model (always eval mode)."""

    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.decay = float(decay)
        self.updates = 0

    def update(self, model):
        import torch
        with torch.no_grad():
            self.updates += 1
            d = min(self.decay, (1.0 + self.updates) / (10.0 + self.updates))
            msd = model.state_dict()
            for k, v in self.model.state_dict().items():
                if v.dtype.is_floating_point:
                    v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
                else:
                    v.copy_(msd[k])

    def state_dict(self):
        return {"model": self.model.state_dict(), "updates": self.updates, "decay": self.decay}

    def load_state_dict(self, sd):
        self.model.load_state_dict(sd["model"])
        self.updates = int(sd["updates"])


# ══════════════════════════════════════════════════════════════════════════════
#  Prediction (OOM-safe: a batch that does not fit is split in halves)
# ══════════════════════════════════════════════════════════════════════════════
def _model_logits(model, t, device, use_fp16):
    import torch
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_fp16):
        return model(t["vis"], t["input_ids"], t["attn_mask"], t["roi"], t["geo"],
                     t.get("crop"), t.get("sig")).float()


def _probs(model, t, device, amp, counters):
    import torch
    oom = False
    try:
        p = torch.sigmoid(_model_logits(model, t, device, amp))
        if not bool(torch.isfinite(p).all()):
            counters["retries"] += 1
            p = torch.sigmoid(_model_logits(model, t, device, False))
        return p
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        counters["oom"] += 1
        cuda_cleanup()
        n = t["vis"].size(0)
        if n <= 1:
            raise RuntimeError("CUDA out of memory for a single sample")
        h = n // 2
        return torch.cat([_probs(model, _slice_batch(t, 0, h), device, amp, counters),
                          _probs(model, _slice_batch(t, h, n), device, amp, counters)])


def predict(model, ds, batch_size, device, amp, num_workers, indices=None):
    """Sigmoid scores for the rows `indices` of a split (all rows by default),
    returned in the order of `indices`. fp32 retry on fp16 overflow."""
    import torch
    model.eval()
    idx = np.arange(len(ds), dtype=np.int64) if indices is None else np.asarray(indices, np.int64)
    loader = make_loader(ds, eval_batches(idx, ds.a.lengths, batch_size), num_workers,
                         pin=device.type == "cuda")
    out = np.zeros((len(ds), NUM_CLASSES), np.float32)
    seen = np.zeros(len(ds), bool)
    counters = {"retries": 0, "oom": 0}
    with torch.no_grad():
        for b in loader:
            p = _probs(model, _to_device(b, device), device, amp, counters)
            bi = b["idx"].numpy()
            out[bi] = p.cpu().numpy()
            seen[bi] = True
    del loader
    cuda_cleanup()
    assert seen[idx].all(), "prediction missed rows"
    res = out[idx]
    if not np.isfinite(res).all():
        raise RuntimeError("non-finite scores even after fp32 retry")
    return res, counters["retries"] + counters["oom"]


# ══════════════════════════════════════════════════════════════════════════════
#  Training
# ══════════════════════════════════════════════════════════════════════════════
def seed_everything(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _new_epoch_stats():
    return {"micro": 0, "loss_sum": 0.0, "loss_n": 0, "fp16_retry": 0, "skipped": 0,
            "overflow_steps": 0, "train_seconds": 0.0, "vad_sum": 0.0, "vad_n": 0,
            "kd_sum": 0.0, "kd_n": 0, "oom": 0}


HISTORY_FIELDS = ["epoch", "variant", "train_loss", "vad_loss", "kd_neg_target", "lr", "lr_scale",
                  "train_min", "eval_min", "val_tl_mAP", "val_tl_mAP_raw", "val_tl_mAP_ema", "val_tl_lift",
                  "val_tl_AUC", "val_tl_micro_f1", "val_tl_macro_f1", "val_tl_P@3", "val_tl_one_error",
                  "val_tl_ranking_loss", "val_tl_below_floor", "val_mAP", "val_lift",
                  "fp16_retry", "skipped", "overflow_steps", "oom", "gpu_peak_gb", "n_train",
                  "patience_left", "no_improve", "lr_reduced", "best_val_tl_mAP"]


def _write_history(run_dir, history):
    path = Path(run_dir) / "history.csv"
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=HISTORY_FIELDS, extrasaction="ignore")
        w.writeheader()
        for row in history:
            w.writerow(row)
    os.replace(tmp, path)


def _fmt_trend(name, v, prev, higher=True, fmt="{:.4f}"):
    s = f"{name} {fmt.format(v)}"
    if prev is None or not np.isfinite(prev):
        return s
    if abs(v - prev) < 1e-9:
        return s
    good = (v > prev) == higher
    return f"{G if good else R}{s}{Z}"


def train_rows(cfg, n):
    """(rows the run trains on, held-out rows or None, signature extra) for a run."""
    if cfg.get("fold") is None:
        return np.arange(n, dtype=np.int64), None, ""
    folds = np.load(cfg["fold_file"])
    r, k = int(cfg["fold_repeat"]), int(cfg["fold"])
    assert folds.ndim == 2 and folds.shape[1] == n, f"fold file shape {folds.shape} vs {n} train rows"
    assert 0 <= r < folds.shape[0], f"fold repeat {r} not in the fold file"
    tr_idx = np.where(folds[r] != k)[0].astype(np.int64)
    ho_idx = np.where(folds[r] == k)[0].astype(np.int64)
    assert len(tr_idx) > 0 and len(ho_idx) > 0, f"empty fold {r}/{k}"
    return tr_idx, ho_idx, f"fold{r}.{k}:{array_sha(folds)}"


def train_run(cfg):
    run_dir = Path(cfg["run_dir"])
    run_dir.mkdir(parents=True, exist_ok=True)
    ctl = Control(cfg)
    ctl.install()
    try:
        return _train(cfg, ctl, run_dir)
    except Exception as e:
        tb = traceback.format_exc()
        log(f"{R}{tb}{Z}")
        set_status(run_dir, "failed", message=f"{type(e).__name__}: {e}"[:800])
        write_json(run_dir / "result.json", {"name": cfg["name"], "status": "failed",
                                              "message": f"{type(e).__name__}: {e}"[:800]})
        return 2


def _train(cfg, ctl, run_dir):
    import torch
    import torch.nn.functional as F

    t_start = time.time()
    torch.set_num_threads(2)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = bool(cfg["amp"]) and device.type == "cuda"
    write_json(run_dir / "spec_resolved.json", cfg)

    res = read_json(run_dir / "result.json", {}) or {}
    if res.get("status") == "done":
        can_extend = (cfg["lr_schedule"] == "plateau" and res.get("stop_reason") == "max_epochs"
                      and (run_dir / "last.pt").exists()
                      and int(cfg["max_epochs"]) > int(res.get("epochs_run", 0)))
        if not can_extend:
            log(f"{G}== {cfg['name']}: already done ({res.get('stop_reason', 'finished')}) — nothing to do.{Z}")
            return 0
        log(f"{Y}== {cfg['name']}: EXTENDING from {res.get('epochs_run')} to {cfg['max_epochs']} epochs{Z}")
        os.replace(run_dir / "result.json", run_dir / f"result_at_{res.get('epochs_run')}ep.json")
        (run_dir / "finished.json").unlink(missing_ok=True)
    fin = read_json(run_dir / "finished.json")
    if fin:                     # training ended but finalisation was interrupted
        log(f"{Y}== {cfg['name']}: training already finished — finalising only.{Z}")
        return _finalize(cfg, run_dir, device, amp, fin, t_start)
    set_status(run_dir, "starting", name=cfg["name"])

    use_crop, use_sig = bool(cfg["use_crop"]), bool(cfg["use_sig"])
    tr = SplitArrays(cfg["array_dir"], "train", cfg["feat_dir"], need_crop=use_crop, need_sig=use_sig)
    va = SplitArrays(cfg["array_dir"], "val", cfg["feat_dir"], need_crop=use_crop, need_sig=use_sig)
    assert va.labels_tl is not None, "val test-like labels missing"
    crop_tokens = int(tr.crop_shape[1]) if use_crop else 0
    sig_tokens = int(tr.sig_shape[1]) if use_sig else 0
    sig_dim = int(tr.sig_shape[2]) if use_sig else 0
    train_idx, held_idx, extra = train_rows(cfg, tr.n)
    kd = None
    if cfg.get("kd_path"):
        kd = np.load(cfg["kd_path"]).astype(np.float32)
        assert kd.shape == (tr.n, NUM_CLASSES), f"soft targets {kd.shape} vs {(tr.n, NUM_CLASSES)}"
        assert np.isfinite(kd).all() and kd.min() >= 0.0 and kd.max() <= 1.0, "soft targets outside [0, 1]"
        extra = (extra + "|" if extra else "") + f"kd{cfg['kd_alpha']}:{array_sha(kd)}"
    # Some EMOTIC train persons have no continuous (VAD) annotation: NaN in the
    # .mat. Statistics ignore them and the VAD loss masks them (never NaN).
    vad_mu = torch.tensor(np.nanmean(tr.vad, 0), device=device, dtype=torch.float32)
    vad_sd = torch.tensor(np.nanstd(tr.vad, 0) + 1e-6, device=device, dtype=torch.float32)
    vad_missing = int((~np.isfinite(tr.vad)).any(1).sum())

    # Shared initialisation: the model is built under init_seed, so every seed
    # of a configuration starts from identical weights (needed for soups).
    seed_everything(int(cfg["init_seed"]))
    model = build_model(cfg, tr.feat_shape, tr.geo.shape[1], sig_dim)
    n_init = load_pretrained_init(model, cfg["init_path"])
    model.to(device)
    ema = EMA(model, cfg["ema_decay"])
    opt = build_optimizer(model, cfg)
    seed_everything(int(cfg["seed"]))          # the seed drives data order and dropout

    bs, accum = int(cfg["batch_size"]), int(cfg["grad_accum"])
    micro_per_epoch = len(train_idx) // bs
    steps_per_epoch = micro_per_epoch // accum
    assert steps_per_epoch >= 1, f"{len(train_idx)} rows give no full optimizer step at {bs} x {accum}"
    micro_used = steps_per_epoch * accum
    warmup_steps = int(round(float(cfg["warmup_epochs"]) * steps_per_epoch))
    sched = make_scheduler(cfg, opt, steps_per_epoch, warmup_steps)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    signature = run_signature(cfg, steps_per_epoch, warmup_steps, crop_tokens, sig_tokens, len(train_idx), extra)
    params = [p for p in model.parameters() if p.requires_grad]

    state = {"epoch": 0, "micro": 0, "global_step": 0, "best": None, "history": [],
             "no_improve": 0, "epoch_stats": _new_epoch_stats(),
             "train_seconds_total": 0.0, "sessions": 0}

    last = run_dir / "last.pt"
    resumed = False
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck.get("signature") != signature:
            raise RuntimeError(f"resume signature mismatch: checkpoint {ck.get('signature')} vs {signature}. "
                               f"Delete {last} to restart this run.")
        model.load_state_dict(ck["model"])
        ema.load_state_dict(ck["ema"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        torch.set_rng_state(ck["rng_cpu"])
        if device.type == "cuda" and ck.get("rng_cuda") is not None:
            try:
                torch.cuda.set_rng_state(ck["rng_cuda"])
            except Exception:
                pass
        resumed = True
        del ck
        cuda_cleanup()
    state["sessions"] = int(state.get("sessions", 0)) + 1

    def save_last():
        obj = {
            "signature": signature, "model": model.state_dict(), "ema": ema.state_dict(),
            "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
            "scaler": scaler.state_dict(), "state": state,
            "rng_cpu": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state() if device.type == "cuda" else None,
        }
        # An atomic save briefly needs room for the old AND the new file. When
        # /kaggle/working is nearly full the old one is removed first instead.
        if last.exists() and free_gb(run_dir) < last.stat().st_size / 1e9 * 1.15 + 0.5:
            log(f"{Y}   low disk ({free_gb(run_dir):.1f} GB free): replacing last.pt in place{Z}")
            last.unlink()
        atomic_torch_save(obj, last)
        del obj

    train_ds = FeatureDataset(tr, use_crop, use_sig, kd=kd)
    val_ds = FeatureDataset(va, use_crop, use_sig)
    nw = int(cfg["num_workers"])
    comps = [k for k in ("use_roi", "use_geo", "use_crop", "use_sig", "use_vad") if cfg[k]]
    role = cfg.get("role", "run")
    what = ("teacher " + f"repeat {cfg['fold_repeat']} fold {cfg['fold']}" if held_idx is not None
            else ("student (soft targets a = " + str(cfg["kd_alpha"]) + ")" if kd is not None else role))
    log(f"{B}== {cfg['name']} == core {CORE_VERSION} | device {device} amp {amp} | "
        f"{'+'.join(c.replace('use_', '') for c in comps) or 'base'} | {what} | "
        f"seed {cfg['seed']} (init {cfg['init_seed']}){Z}")
    log(f"   train {len(train_idx):,} of {tr.n:,} x {tr.feat_shape[1]}x{tr.feat_shape[2]}"
        + (f" + crop {crop_tokens}" if use_crop else "") + (f" + SigLIP {sig_tokens}x{sig_dim}" if use_sig else "")
        + (f" (VAD missing for {vad_missing} persons: masked)" if cfg["use_vad"] and vad_missing else "")
        + (f" | held out {len(held_idx):,}" if held_idx is not None else "")
        + f"  val {va.n:,} | {steps_per_epoch} steps/epoch (bs {bs} x {accum}, "
        f"{'length-bucketed' if cfg['bucket'] else 'plain'} batches) | {cfg['lr_schedule']} LR over "
        f"{cfg['max_epochs']} epochs, warmup {warmup_steps} steps | early stop after {cfg['patience']} "
        f"flat epochs (min {cfg['min_epochs']}) | init tensors {n_init}")
    if resumed:
        log(f"{G}   RESUMED at epoch {state['epoch']} micro-batch {state['micro']} "
            f"(global step {state['global_step']}) | best val-TL "
            f"{(state['best'] or {}).get('val_tl_mAP', float('nan')):.2f} | "
            f"{state['no_improve']}/{cfg['patience']} epochs without gain | LR scale {sched.scale:.3g}{Z}")
    set_status(run_dir, "running", name=cfg["name"], epoch=state["epoch"])

    inject_every = int(cfg.get("test_inject_nonfinite_every") or 0)
    oom_every = int(cfg.get("test_inject_oom_every") or 0)
    stop_after = int(cfg.get("test_stop_after_steps") or 0)
    diag_left = 3
    alpha = float(cfg["kd_alpha"])

    def forward_loss(t, use_fp16, inject):
        info = {"vad": None, "kd": None}
        target = t["labels"]
        if "kd" in t:
            target = (1.0 - alpha) * t["labels"] + alpha * t["kd"]
            neg = t["labels"] < 0.5
            info["kd"] = float((target * neg).sum() / neg.sum().clamp(min=1))
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_fp16):
            logits, vadp = model(t["vis"], t["input_ids"], t["attn_mask"], t["roi"], t["geo"],
                                 t.get("crop"), t.get("sig"), return_vad=True)
        logits = logits.float()
        loss = F.binary_cross_entropy_with_logits(logits, target)
        if vadp is not None:
            vt = (t["vad"] - vad_mu) / vad_sd
            ok = torch.isfinite(vt)
            if bool(ok.any()):
                vl = (((vadp.float() - torch.nan_to_num(vt)) ** 2) * ok).sum() / ok.sum()
                loss = loss + float(cfg["vad_weight"]) * vl
                info["vad"] = float(vl.detach())
        if inject:
            loss = loss * float("nan")
        return loss, logits, info

    def micro_step(t, frac, inject, stats):
        """Forward + backward of (a part of) a micro-batch. Returns (loss, info) or None if skipped."""
        nonlocal diag_left
        loss, logits, info = forward_loss(t, amp, inject)
        if not bool(torch.isfinite(loss)):
            stats["fp16_retry"] += 1
            del loss, logits
            loss, logits, info = forward_loss(t, False, False)
            if not bool(torch.isfinite(loss)):
                stats["skipped"] += 1
                if diag_left > 0:
                    diag_left -= 1
                    bad = (~torch.isfinite(logits)).any(1).nonzero().flatten().cpu().tolist()
                    log(f"{R}   non-finite in fp32: vis finite {bool(torch.isfinite(t['vis'].float()).all())}, "
                        f"rows {bad[:8]}{Z}")
                if stats["skipped"] >= 5 and stats["skipped"] > 0.01 * (stats["micro"] + 1):
                    raise RuntimeError(f"non-finite loss persists in fp32 "
                                       f"({stats['skipped']} micro-batches) — data or model bug")
                return None
        scaler.scale(loss * (frac / accum)).backward()
        out = (float(loss.detach()), info)
        del loss, logits
        return out

    for epoch in range(int(state["epoch"]), int(cfg["max_epochs"])):
        stats = state["epoch_stats"] if state["micro"] > 0 else _new_epoch_stats()
        sampler = BucketBatchSampler(train_idx, tr.lengths, bs, cfg["seed"], epoch, start=state["micro"],
                                     bucket=bool(cfg["bucket"]), chunk=int(cfg["bucket_chunk"]))
        loader = make_loader(train_ds, sampler, num_workers=nw, pin=device.type == "cuda")
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        model.train()
        opt.zero_grad(set_to_none=True)
        micro = int(state["micro"])
        window_ok = 0
        t_seg = time.time()
        t_log = time.time()

        for batch in loader:
            if micro >= micro_used:
                break
            t = _to_device(batch, device)
            del batch
            inject = inject_every > 0 and micro % inject_every == 0

            results, oom = [], False
            try:
                if oom_every > 0 and micro % oom_every == oom_every - 1:
                    raise torch.cuda.OutOfMemoryError("injected by self-test")
                results.append(micro_step(t, 1.0, inject, stats))
            except torch.cuda.OutOfMemoryError:
                oom = True
            if oom:
                # Gradients of this accumulation window may be partial: drop them,
                # free the cache, and redo this micro-batch in two halves.
                stats["oom"] += 1
                opt.zero_grad(set_to_none=True)
                window_ok = 0
                cuda_cleanup()
                if stats["oom"] > int(cfg["max_oom_per_epoch"]):
                    raise RuntimeError(f"{stats['oom']} CUDA OOM events this epoch — use micro-batch 16")
                n_b = t["labels"].size(0)
                h = n_b // 2
                results, oom2 = [], False
                try:
                    for a0, b0 in ((0, h), (h, n_b)):
                        results.append(micro_step(_slice_batch(t, a0, b0), (b0 - a0) / n_b, False, stats))
                except torch.cuda.OutOfMemoryError:
                    oom2 = True
                if oom2:
                    stats["oom"] += 1
                    stats["skipped"] += 1
                    opt.zero_grad(set_to_none=True)
                    results = []
                    cuda_cleanup()
                if micro % 50 == 0 or stats["oom"] <= 3:
                    log(f"{Y}   CUDA OOM at micro-batch {micro}: {'recovered in halves' if not oom2 else 'skipped'}"
                        f" (epoch total {stats['oom']}){Z}")
            del t
            good = [r for r in results if r is not None]
            if good:
                window_ok += 1
                stats["loss_sum"] += float(np.mean([r[0] for r in good]))
                stats["loss_n"] += 1
                for _, info in good:
                    if info["vad"] is not None:
                        stats["vad_sum"] += info["vad"]
                        stats["vad_n"] += 1
                    if info["kd"] is not None:
                        stats["kd_sum"] += info["kd"]
                        stats["kd_n"] += 1
            micro += 1
            stats["micro"] += 1

            if micro % accum == 0:
                if window_ok > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(params, float(cfg["grad_clip"]))
                    s_before = scaler.get_scale() if amp else 1.0
                    scaler.step(opt)
                    scaler.update()
                    if amp and scaler.get_scale() < s_before:
                        stats["overflow_steps"] += 1
                opt.zero_grad(set_to_none=True)
                sched.step()
                ema.update(model)
                state["global_step"] += 1
                window_ok = 0
                if state["global_step"] % 10 == 0 or stop_after:
                    if stop_after and state["global_step"] >= stop_after:
                        ctl.request("self-test stop")
                    reason = ctl.check()
                    if reason:
                        stats["train_seconds"] += time.time() - t_seg
                        state["train_seconds_total"] += time.time() - t_seg
                        state.update(epoch=epoch, micro=micro, epoch_stats=stats)
                        del loader
                        save_last()
                        set_status(run_dir, "paused", reason=reason, epoch=epoch, micro=micro,
                                   best=(state["best"] or {}).get("val_tl_mAP"))
                        log(f"{Y}   PAUSED ({reason}) at epoch {epoch} micro-batch {micro}/"
                            f"{micro_used} — state saved, resumes exactly here.{Z}")
                        return 0

            if micro % int(cfg["log_every"]) == 0:
                el = time.time() - t_log
                rate = int(cfg["log_every"]) / max(el, 1e-9)
                eta = (micro_used - micro) / max(rate, 1e-9)
                t_log = time.time()
                mem = (f"  gpu {torch.cuda.max_memory_allocated() / 1e9:.1f}GB"
                       if device.type == "cuda" else "")
                kdm = stats["kd_sum"] / max(stats["kd_n"], 1)
                log(f"   ep{epoch + 1:02d} [{micro:>4}/{micro_used}] loss "
                    f"{stats['loss_sum'] / max(stats['loss_n'], 1):.4f}  lr "
                    f"{opt.param_groups[-1]['lr']:.2e}  {rate:.2f} it/s  eta {eta / 60:.1f}m  "
                    f"retry {stats['fp16_retry']} skip {stats['skipped']} oom {stats['oom']}"
                    + (f"  kd-neg {kdm:.3f}" if kd is not None else "") + mem)
                write_json(run_dir / "progress.json", {
                    "name": cfg["name"], "epoch": epoch, "micro": micro, "micro_used": micro_used,
                    "it_s": rate, "eta_epoch_s": eta, "updated": time.time(),
                    "best": (state["best"] or {}).get("val_tl_mAP")})

        if micro < micro_used:
            raise RuntimeError(f"loader ended early: {micro}/{micro_used} micro-batches")
        del loader
        stats["train_seconds"] += time.time() - t_seg
        state["train_seconds_total"] += time.time() - t_seg
        gpu_peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
        opt.zero_grad(set_to_none=True)
        cuda_cleanup()

        # ── validation: raw and EMA, select on test-like val mAP ────────────
        t_eval = time.time()
        raw_scores, rr = predict(model, val_ds, int(cfg["eval_batch_size"]), device, amp, nw)
        ema_scores, re_ = predict(ema.model, val_ds, int(cfg["eval_batch_size"]), device, amp, nw)
        m_raw = compute_map(raw_scores, va.labels_tl)
        m_ema = compute_map(ema_scores, va.labels_tl)
        variant, sel = ("ema", ema_scores) if m_ema > m_raw else ("raw", raw_scores)
        th = optimal_thresholds(sel, va.labels_tl)
        r_tl = evaluate_scores(sel, va.labels_tl, thresholds=th)
        r_off = evaluate_scores(sel, va.labels, thresholds=th, ranking=False)
        eval_s = time.time() - t_eval

        prev_best = (state["best"] or {}).get("val_tl_mAP")
        flag = ""
        if prev_best is None or r_tl["mAP"] > prev_best:
            gain = r_tl["mAP"] - (prev_best if prev_best is not None else -1e9)
            src = ema.model if variant == "ema" else model
            meta = {"epoch": epoch, "variant": variant, "val_tl_mAP": r_tl["mAP"],
                    "val_tl_lift": r_tl["lift"], "val_tl_AUC": r_tl["AUC"],
                    "val_mAP": r_off["mAP"], "val_lift": r_off["lift"],
                    "thresholds": th.tolist(), "config": {k: cfg[k] for k in
                    ("name", "config_name", "seed", "init_seed", "role", "use_roi", "use_geo", "use_crop",
                     "use_sig", "use_vad", "vis_token_drop", "txt_token_drop", "qformer_dropout", "head_dropout",
                     "roi_grid", "kd_alpha", "fold", "fold_repeat")}}
            meta["config"]["kd"] = bool(kd is not None)
            atomic_torch_save({"model": src.state_dict(), "meta": meta}, run_dir / "best.pt")
            state["best"] = meta
            if gain > float(cfg["min_delta"]):
                state["no_improve"] = 0
                flag = f"  {G}* NEW BEST ({variant}){Z}"
            else:
                state["no_improve"] += 1
                flag = f"  {Y}* best (+{gain:.3f} ≤ min_delta, {variant}){Z}"
        else:
            state["no_improve"] += 1
        reduced = sched.epoch_end(r_tl["mAP"])
        if reduced:
            flag += f"  {Y}LR x{cfg['plateau_factor']} → scale {sched.scale:.3g}{Z}"
        state["patience_left"] = int(cfg["patience"]) - int(state["no_improve"])

        prev_row = state["history"][-1] if state["history"] else {}
        row = {"epoch": epoch, "variant": variant,
               "train_loss": stats["loss_sum"] / max(stats["loss_n"], 1),
               "vad_loss": (stats["vad_sum"] / stats["vad_n"]) if stats["vad_n"] else "",
               "kd_neg_target": (stats["kd_sum"] / stats["kd_n"]) if stats["kd_n"] else "",
               "lr": opt.param_groups[-1]["lr"], "lr_scale": sched.scale,
               "train_min": stats["train_seconds"] / 60.0,
               "eval_min": eval_s / 60.0, "val_tl_mAP": r_tl["mAP"], "val_tl_mAP_raw": m_raw,
               "val_tl_mAP_ema": m_ema, "val_tl_lift": r_tl["lift"], "val_tl_AUC": r_tl["AUC"],
               "val_tl_micro_f1": r_tl["micro_f1"], "val_tl_macro_f1": r_tl["macro_f1"],
               "val_tl_P@3": r_tl["P@3"], "val_tl_one_error": r_tl["one_error"],
               "val_tl_ranking_loss": r_tl["ranking_loss"],
               "val_tl_below_floor": r_tl["classes_below_floor"],
               "val_mAP": r_off["mAP"], "val_lift": r_off["lift"],
               "fp16_retry": stats["fp16_retry"], "skipped": stats["skipped"],
               "overflow_steps": stats["overflow_steps"], "oom": stats["oom"],
               "gpu_peak_gb": gpu_peak, "n_train": len(train_idx),
               "patience_left": state["patience_left"], "no_improve": state["no_improve"],
               "lr_reduced": int(reduced), "best_val_tl_mAP": state["best"]["val_tl_mAP"]}
        state["history"].append(row)
        _write_history(run_dir, state["history"])

        p = prev_row
        log(f"  {'─' * 72}")
        log(f"  {cfg['name']}  ep{epoch + 1:02d}/{cfg['max_epochs']}  train {row['train_min']:.1f}m + "
            f"eval {row['eval_min']:.1f}m  {_fmt_trend('loss', row['train_loss'], p.get('train_loss'), False)}"
            f"  lr {row['lr']:.2e}  no-gain {state['no_improve']}/{cfg['patience']}{flag}")
        log(f"     SELECT val-TL mAP {r_tl['mAP']:6.2f} ({r_tl['lift']:.3f}x)   "
            f"raw {m_raw:.2f}  ema {m_ema:.2f}  -> {variant}   "
            f"best {state['best']['val_tl_mAP']:.2f} @ep{state['best']['epoch'] + 1}")
        log(f"     rank  {_fmt_trend('AUC', r_tl['AUC'], p.get('val_tl_AUC'))}  "
            f"{_fmt_trend('P@3', r_tl['P@3'], p.get('val_tl_P@3'), fmt='{:.3f}')}  "
            f"{_fmt_trend('1-err', r_tl['one_error'], p.get('val_tl_one_error'), False, '{:.3f}')}  "
            f"{_fmt_trend('rankloss', r_tl['ranking_loss'], p.get('val_tl_ranking_loss'), False)}")
        log(f"     set   {_fmt_trend('miF1', r_tl['micro_f1'], p.get('val_tl_micro_f1'))}  "
            f"{_fmt_trend('maF1', r_tl['macro_f1'], p.get('val_tl_macro_f1'))}  "
            f"below-floor {r_tl['classes_below_floor']}/26   official val {r_off['mAP']:.2f} "
            f"({r_off['lift']:.3f}x)")
        hl = G if (stats["fp16_retry"] == 0 and stats["skipped"] == 0 and stats["oom"] == 0) else Y
        log(f"     {hl}health fp16->fp32 retries {stats['fp16_retry']}  skipped {stats['skipped']}  "
            f"CUDA OOM {stats['oom']}  overflow steps {stats['overflow_steps']}  eval retries {rr + re_}  "
            f"GPU peak {gpu_peak:.1f} GB{Z}"
            + (f"  mean soft target on train negatives {row['kd_neg_target']:.3f}" if row["kd_neg_target"] != "" else "")
            + (f"  VAD mse {row['vad_loss']:.3f}" if row["vad_loss"] != "" else ""))

        state.update(epoch=epoch + 1, micro=0, epoch_stats=_new_epoch_stats())
        save_last()
        del raw_scores, ema_scores, sel
        cuda_cleanup()
        set_status(run_dir, "running", epoch=epoch + 1, best=state["best"]["val_tl_mAP"],
                   no_improve=state["no_improve"], lr_scale=sched.scale)
        write_json(run_dir / "progress.json", {
            "name": cfg["name"], "epoch": epoch + 1, "micro": 0, "micro_used": micro_used,
            "updated": time.time(), "best": state["best"]["val_tl_mAP"],
            "no_improve": state["no_improve"], "lr_scale": sched.scale,
            "epoch_minutes": row["train_min"] + row["eval_min"]})

        if epoch + 1 >= int(cfg["floor_check_epoch"]) and state["best"]["val_tl_lift"] < float(cfg["floor_min_lift"]):
            raise RuntimeError(f"floor check failed: best val-TL lift {state['best']['val_tl_lift']:.3f} "
                               f"< {cfg['floor_min_lift']} after {epoch + 1} epochs")
        if epoch + 1 >= int(cfg["min_epochs"]) and state["no_improve"] >= int(cfg["patience"]):
            log(f"  {G}early stop: {state['no_improve']} epochs without a gain > {cfg['min_delta']} "
                f"(patience {cfg['patience']}); best val-TL {state['best']['val_tl_mAP']:.2f} "
                f"at epoch {state['best']['epoch'] + 1}{Z}")
            state["stop_reason"] = "early_stop"
            break
        reason = ctl.check()
        if reason and epoch + 1 < int(cfg["max_epochs"]):
            set_status(run_dir, "paused", reason=reason, epoch=epoch + 1, micro=0,
                       best=state["best"]["val_tl_mAP"])
            log(f"{Y}   PAUSED ({reason}) at the end of epoch {epoch + 1} — resumes at epoch {epoch + 2}.{Z}")
            return 0

    # Training is over. finished.json makes the end crash-safe: if the process
    # dies during finalisation, a relaunch finalises instead of training on.
    stop_reason = state.pop("stop_reason", None) or "max_epochs"
    fin = {"best": state["best"], "epochs_run": len(state["history"]),
           "train_seconds_total": state["train_seconds_total"],
           "sessions": state["sessions"], "signature": signature, "stop_reason": stop_reason,
           "no_improve": state["no_improve"], "lr_scale": sched.scale, "n_train": int(len(train_idx))}
    write_json(run_dir / "finished.json", fin)
    model = ema = opt = sched = scaler = params = None     # free GPU memory before scoring
    cuda_cleanup()
    return _finalize(cfg, run_dir, device, amp, fin, t_start)


# |recomputed - selection| allowed when re-scoring best.pt: exact weights give
# the same val-TL up to GPU nondeterminism; the fp16 archive copy moves it a
# little (a few hundredths on the real model).
SCORE_TOLERANCE = {"float32": 0.05, "float16": 0.30}


def _finalize(cfg, run_dir, device, amp, fin, t_start):
    """
    Plain runs and students: score best.pt on val and test ONCE, then shrink
    best.pt to fp16. Teachers: score val and the held-out train fold (oof.npz)
    — never test — and delete their weights afterwards.

    The model was chosen on test-like val only, before this point. Test scores
    are written to scores.npz and not printed; Block 09 reports them.
    """
    best = fin["best"]
    teacher = cfg.get("fold") is not None
    set_status(run_dir, "scoring", best=best["val_tl_mAP"])
    scored = False
    try:
        ev = score_teacher(cfg, run_dir, device, amp) if teacher else score_best(cfg, run_dir, device, amp)
        diff = abs(ev["val_tl_mAP_recomputed"] - ev["val_tl_mAP_at_selection"])
        tol = SCORE_TOLERANCE[ev["weights_dtype"]]
        c = G if diff < tol else R
        what = (f"out-of-fold train probabilities sealed in oof.npz ({ev['n_oof']:,} persons)" if teacher
                else "test scores sealed in scores.npz")
        log(f"{c}   scored best.pt: val-TL recomputed {ev['val_tl_mAP_recomputed']:.2f} "
            f"(selection {ev['val_tl_mAP_at_selection']:.2f}); {what}{Z}")
        scored = diff < tol
    except Exception as e:
        log(f"{Y}   scoring after training failed ({type(e).__name__}: {e}); "
            f"the scheduler / Block 09 will score this run instead.{Z}")
    cuda_cleanup()
    stop_reason = fin.get("stop_reason", "early_stop")
    extendable = bool(cfg["lr_schedule"] == "plateau" and stop_reason == "max_epochs" and not teacher
                      and cfg.get("keep_extendable", True)
                      and (run_dir / "last.pt").exists() and free_gb(run_dir) > 8.0)
    if not extendable:
        (run_dir / "last.pt").unlink(missing_ok=True)
        if scored:
            if teacher:
                (run_dir / "best.pt").unlink(missing_ok=True)     # its product is oof.npz
            else:
                compact_best(run_dir)
    result = {"name": cfg["name"], "config_name": cfg["config_name"], "seed": cfg["seed"],
              "init_seed": cfg["init_seed"], "role": ("teacher" if teacher else cfg.get("role", "run")),
              "fold": cfg.get("fold"), "fold_repeat": cfg.get("fold_repeat"),
              "status": "done", "best": best, "epochs_run": fin["epochs_run"],
              "stop_reason": stop_reason, "extendable": extendable,
              "max_epochs": int(cfg["max_epochs"]), "no_improve": fin.get("no_improve"),
              "lr_scale": fin.get("lr_scale"), "n_train": fin.get("n_train"),
              "train_hours": fin["train_seconds_total"] / 3600.0,
              "sessions": fin["sessions"], "signature": fin["signature"], "scored": scored,
              "components": {k: cfg[k] for k in ("use_roi", "use_geo", "use_crop", "use_sig", "use_vad",
                                                 "vis_token_drop", "txt_token_drop", "kd_alpha")},
              "kd": bool(cfg.get("kd_path"))}
    write_json(run_dir / "result.json", result)
    set_status(run_dir, "done", best=best["val_tl_mAP"], epoch=best["epoch"], stop_reason=stop_reason)
    log(f"{G}== {cfg['name']} DONE ({stop_reason}) — best val-TL "
        f"{best['val_tl_mAP']:.2f} ({best['val_tl_lift']:.3f}x) at epoch {best['epoch'] + 1} "
        f"[{best['variant']}] after {fin['epochs_run']} epochs | {time.time() - t_start:.0f}s this process ==={Z}")
    return 0


# ══════════════════════════════════════════════════════════════════════════════
#  Scoring of a finished run
# ══════════════════════════════════════════════════════════════════════════════
def _load_scoring_model(cfg, ck, arr, device):
    use_sig = bool(cfg["use_sig"])
    model = build_model(cfg, arr.feat_shape, arr.geo.shape[1], int(arr.sig_shape[2]) if use_sig else 0)
    model.load_state_dict(ck["model"], strict=True)     # fp16 copies are cast back to fp32
    return model.to(device)


def score_best(cfg, run_dir, device, amp):
    """val + test scores of best.pt -> scores.npz (test is never printed here)."""
    import torch
    run_dir = Path(run_dir)
    ck = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
    uc, us = bool(cfg["use_crop"]), bool(cfg["use_sig"])
    va = SplitArrays(cfg["array_dir"], "val", cfg["feat_dir"], need_crop=uc, need_sig=us)
    te = SplitArrays(cfg["array_dir"], "test", cfg["feat_dir"], need_crop=uc, need_sig=us)
    model = _load_scoring_model(cfg, ck, va, device)
    nw, ebs = int(cfg["num_workers"]), int(cfg["eval_batch_size"])
    vs, r1 = predict(model, FeatureDataset(va, uc, us), ebs, device, amp, nw)
    ts, r2 = predict(model, FeatureDataset(te, uc, us), ebs, device, amp, nw)
    tmp = run_dir / "scores.npz.tmp"
    with open(tmp, "wb") as f:
        np.savez(f, val_scores=vs, test_scores=ts, val_labels=va.labels,
                 val_labels_tl=va.labels_tl, test_labels=te.labels)
    os.replace(tmp, run_dir / "scores.npz")
    chk = compute_map(vs, va.labels_tl)
    ev = {"name": cfg["name"], "val_tl_mAP_recomputed": chk,
          "val_tl_mAP_at_selection": ck["meta"]["val_tl_mAP"], "retries": r1 + r2,
          "n_val": va.n, "n_test": te.n, "updated": time.time(),
          "weights_dtype": ck["meta"].get("weights_dtype", "float32"),
          "thresholds": ck["meta"]["thresholds"], "epoch": ck["meta"]["epoch"],
          "variant": ck["meta"]["variant"], "config": ck["meta"]["config"]}
    write_json(run_dir / "eval.json", ev)
    del model, ck
    cuda_cleanup()
    return ev


def score_teacher(cfg, run_dir, device, amp):
    """A fold teacher: val (to verify the selection value) and the held-out train
    fold -> oof.npz. The test split is not opened."""
    import torch
    run_dir = Path(run_dir)
    ck = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
    uc, us = bool(cfg["use_crop"]), bool(cfg["use_sig"])
    tr = SplitArrays(cfg["array_dir"], "train", cfg["feat_dir"], need_crop=uc, need_sig=us)
    va = SplitArrays(cfg["array_dir"], "val", cfg["feat_dir"], need_crop=uc, need_sig=us)
    _, held_idx, _ = train_rows(cfg, tr.n)
    model = _load_scoring_model(cfg, ck, va, device)
    nw, ebs = int(cfg["num_workers"]), int(cfg["eval_batch_size"])
    vs, r1 = predict(model, FeatureDataset(va, uc, us), ebs, device, amp, nw)
    ps, r2 = predict(model, FeatureDataset(tr, uc, us), ebs, device, amp, nw, indices=held_idx)
    tmp = run_dir / "oof.npz.tmp"
    with open(tmp, "wb") as f:
        np.savez(f, idx=held_idx, probs=ps.astype(np.float32), fold=int(cfg["fold"]),
                 repeat=int(cfg["fold_repeat"]))
    os.replace(tmp, run_dir / "oof.npz")
    chk = compute_map(vs, va.labels_tl)
    ev = {"name": cfg["name"], "role": "teacher", "val_tl_mAP_recomputed": chk,
          "val_tl_mAP_at_selection": ck["meta"]["val_tl_mAP"], "retries": r1 + r2,
          "n_val": va.n, "n_oof": int(len(held_idx)), "updated": time.time(),
          "oof_map_vs_train_labels": compute_map(ps, tr.labels[held_idx]),
          "weights_dtype": ck["meta"].get("weights_dtype", "float32"),
          "epoch": ck["meta"]["epoch"], "variant": ck["meta"]["variant"], "config": ck["meta"]["config"]}
    write_json(run_dir / "eval.json", ev)
    del model, ck
    cuda_cleanup()
    return ev


def compact_best(run_dir):
    import torch
    p = Path(run_dir) / "best.pt"
    ck = torch.load(p, map_location="cpu", weights_only=False)
    if ck["meta"].get("weights_dtype") == "float16":
        return
    ck["model"] = {k: (v.half() if v.is_floating_point() else v) for k, v in ck["model"].items()}
    ck["meta"]["weights_dtype"] = "float16"
    atomic_torch_save(ck, p)


def evaluate_run(cfg):
    """CLI fallback for a run whose scoring did not complete (teachers: OOF only)."""
    import torch
    run_dir = Path(cfg["run_dir"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = bool(cfg["amp"]) and device.type == "cuda"
    torch.set_num_threads(2)
    teacher = cfg.get("fold") is not None
    try:
        ev = score_teacher(cfg, run_dir, device, amp) if teacher else score_best(cfg, run_dir, device, amp)
        ok = abs(ev["val_tl_mAP_recomputed"] - ev["val_tl_mAP_at_selection"]) < SCORE_TOLERANCE[ev["weights_dtype"]]
        if ok and teacher:
            (run_dir / "best.pt").unlink(missing_ok=True)
            res = read_json(run_dir / "result.json", {}) or {}
            if res.get("status") == "done":
                res["scored"] = True
                write_json(run_dir / "result.json", res)
        log(f"{G if ok else R}[eval] {cfg['name']}: val-TL {ev['val_tl_mAP_recomputed']:.2f} "
            f"(selection {ev['val_tl_mAP_at_selection']:.2f}); "
            f"{'out-of-fold probabilities' if teacher else 'test scores'} written{Z}")
        return 0 if ok else 3
    except Exception as e:
        log(f"{R}{traceback.format_exc()}{Z}")
        write_json(run_dir / "eval.json", {"name": cfg["name"], "error": f"{type(e).__name__}: {e}"})
        return 2


# ══════════════════════════════════════════════════════════════════════════════
#  Cross-fitted soft targets
# ══════════════════════════════════════════════════════════════════════════════
def assemble_kd_targets(teacher_dirs, n_train, out_path, train_labels=None):
    """
    Average the teachers' out-of-fold probabilities per train person. Every
    person must be held out exactly once per repeat (checked). Returns stats.
    """
    acc = np.zeros((int(n_train), NUM_CLASSES), np.float64)
    cnt = np.zeros(int(n_train), np.int64)
    reps, per = set(), []
    for d in teacher_dirs:
        z = np.load(Path(d) / "oof.npz")
        idx, pr = z["idx"].astype(np.int64), z["probs"].astype(np.float64)
        if pr.shape != (len(idx), NUM_CLASSES) or not np.isfinite(pr).all():
            raise RuntimeError(f"{Path(d).name}: malformed out-of-fold probabilities")
        acc[idx] += pr
        cnt[idx] += 1
        reps.add(int(z["repeat"]))
        per.append({"run": Path(d).name, "repeat": int(z["repeat"]), "fold": int(z["fold"]), "n": int(len(idx))})
    if not (cnt == len(reps)).all():
        raise RuntimeError(f"out-of-fold coverage incomplete: persons covered 0/1/2.. times = "
                           f"{np.bincount(cnt).tolist()} for {len(reps)} repeat(s)")
    kd = (acc / cnt[:, None]).astype(np.float32)
    kd = np.clip(kd, 0.0, 1.0)
    atomic_np_save(kd, out_path)
    st = {"n": int(n_train), "repeats": len(reps), "teachers": per, "sha": array_sha(kd),
          "mean": float(kd.mean())}
    if train_labels is not None:
        y = np.asarray(train_labels)
        st["mean_on_positives"] = float(kd[y > 0.5].mean())
        st["mean_on_negatives"] = float(kd[y < 0.5].mean())
        st["oof_map_vs_train_labels"] = compute_map(kd, y)
    return st


# ══════════════════════════════════════════════════════════════════════════════
#  Model soups (one network from several seeds of one configuration)
# ══════════════════════════════════════════════════════════════════════════════
def soup_run(spec):
    """
    Uniform soup: the average of every ingredient's best.pt weights.
    Greedy soup (Wortsman et al. 2022): ingredients sorted by their own val-TL,
    each added only if the soup's test-like val mAP does not drop.
    Decisions use test-like val only; test scores of the two soups are sealed
    in soup_scores.npz (Block 09 reads them). Ingredients must share their run
    signature up to the seed (same init, same data, same recipe).
    """
    import torch
    out_dir = Path(spec["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    torch.set_num_threads(2)
    t0 = time.time()
    try:
        ings = [Path(p) for p in spec["ingredients"]]
        res = [read_json(d / "result.json", {}) or {} for d in ings]
        if any(r.get("status") != "done" for r in res):
            raise RuntimeError("an ingredient has not finished")
        keys = {soup_key(r.get("signature", "")) for r in res}
        if len(keys) != 1:
            raise RuntimeError(f"ingredients differ in more than the seed: {sorted(keys)}")
        cfg = merged_config(read_json(ings[0] / "spec_resolved.json", {}) or {})
        for k in ("array_dir", "feat_dir", "hf_config_dir", "init_path"):
            if spec.get(k):
                cfg[k] = spec[k]
        cfg["num_workers"] = int(spec.get("num_workers", cfg["num_workers"]))
        cfg["eval_batch_size"] = int(spec.get("eval_batch_size", cfg["eval_batch_size"]))
        uc, us = bool(cfg["use_crop"]), bool(cfg["use_sig"])
        va = SplitArrays(cfg["array_dir"], "val", cfg["feat_dir"], need_crop=uc, need_sig=us)
        te = SplitArrays(cfg["array_dir"], "test", cfg["feat_dir"], need_crop=uc, need_sig=us)
        weights, own = {}, {}
        for d, r in zip(ings, res):
            p = d / "best.pt"
            alt = spec.get("archive", {}).get(d.name)
            if not p.exists() and alt and Path(alt).exists():
                p = Path(alt)
            weights[d.name] = torch.load(p, map_location="cpu", weights_only=False)["model"]
            own[d.name] = float(r["best"]["val_tl_mAP"])
        names = [d.name for d in ings]
        model = build_model(cfg, va.feat_shape, va.geo.shape[1], int(va.sig_shape[2]) if us else 0).to(device)
        vds, tds = FeatureDataset(va, uc, us), FeatureDataset(te, uc, us)
        ebs, nw = int(cfg["eval_batch_size"]), int(cfg["num_workers"])

        def load_avg(members):
            avg = {}
            for k, v0 in weights[members[0]].items():
                if v0.is_floating_point():
                    avg[k] = torch.stack([weights[m][k].float() for m in members]).mean(0)
                else:
                    avg[k] = v0
            model.load_state_dict(avg, strict=True)

        def val_of(members):
            load_avg(members)
            s, _ = predict(model, vds, ebs, device, amp, nw)
            return compute_map(s, va.labels_tl), s

        u_map, u_val = val_of(names)
        u_test, _ = predict(model, tds, ebs, device, amp, nw)
        log(f"[soup] {spec['name']}: uniform soup of {len(names)} → val-TL {u_map:.2f} "
            f"(ingredients {min(own.values()):.2f}–{max(own.values()):.2f})")
        order = sorted(names, key=lambda n: -own[n])
        g = [order[0]]
        g_map, g_val = val_of(g)
        steps = [{"add": order[0], "val_tl": g_map, "accepted": True, "own_recorded": own[order[0]]}]
        for n in order[1:]:
            m, s = val_of(g + [n])
            acc = m >= g_map
            steps.append({"add": n, "val_tl": m, "accepted": bool(acc)})
            log(f"[soup] {spec['name']}: greedy + {n} → val-TL {m:.2f} ({'kept' if acc else 'rejected'})")
            if acc:
                g, g_map, g_val = g + [n], m, s
        if sorted(g) == sorted(names):
            g_test = u_test
        else:
            load_avg(g)
            g_test, _ = predict(model, tds, ebs, device, amp, nw)
        tmp = out_dir / "soup_scores.npz.tmp"
        with open(tmp, "wb") as f:
            np.savez(f, uniform_val=u_val, uniform_test=u_test, greedy_val=g_val, greedy_test=g_test,
                     val_labels_tl=va.labels_tl, test_labels=te.labels)
        os.replace(tmp, out_dir / "soup_scores.npz")
        info = {"name": spec["name"], "status": "done", "config": cfg["config_name"], "ingredients": names,
                "own_val_tl": own, "uniform": {"members": names, "val_tl_mAP": u_map,
                                               "thresholds": optimal_thresholds(u_val, va.labels_tl).tolist()},
                "greedy": {"members": g, "val_tl_mAP": g_map, "steps": steps,
                           "thresholds": optimal_thresholds(g_val, va.labels_tl).tolist()},
                "first_ingredient_recomputed": steps[0]["val_tl"], "seconds": time.time() - t0,
                "core_version": CORE_VERSION}
        write_json(out_dir / "soup.json", info)
        log(f"{G}[soup] {spec['name']}: uniform {u_map:.2f} | greedy {g_map:.2f} with {len(g)}/{len(names)} "
            f"ingredients | test scores sealed | {time.time() - t0:.0f}s{Z}")
        model = weights = None          # free memory (closures above are done)
        cuda_cleanup()
        return 0
    except Exception as e:
        log(f"{R}{traceback.format_exc()}{Z}")
        write_json(out_dir / "soup.json", {"name": spec.get("name"), "status": "failed",
                                           "message": f"{type(e).__name__}: {e}"[:800]})
        return 2


# ══════════════════════════════════════════════════════════════════════════════
#  GPU memory pre-flight (worst-case batch, before any real run starts)
# ══════════════════════════════════════════════════════════════════════════════
def preflight(spec):
    """
    Train a randomly initialised FULL model (image + ROI + EVA crop + SigLIP
    crop + geometry + VAD, soft targets) for a few steps on the LONGEST
    possible batch and measure peak memory for each candidate micro-batch
    size. The scheduler uses the largest size that fits with headroom;
    gradient accumulation keeps the effective batch at 64.
    """
    import torch
    import torch.nn.functional as F
    out_path = Path(spec["out"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    torch.manual_seed(0)
    n_sig, sig_dim = int(spec.get("sig_tokens", 0)), int(spec.get("sig_dim", 0))
    cfg = merged_config({"hf_config_dir": spec["hf_config_dir"], "use_roi": True, "use_geo": True,
                         "use_crop": True, "use_sig": n_sig > 0, "use_vad": True,
                         "vis_token_drop": 0.25, "txt_token_drop": 0.1})
    _, hfcfg = build_qformer(spec["hf_config_dir"], cfg["qformer_dropout"])
    vis_dim = int(hfcfg.qformer_config.encoder_hidden_size)
    n_tok, n_crop = int(spec["vis_tokens"]), int(spec["crop_tokens"])
    L, geo_dim = int(spec["text_len"]), int(spec["geo_dim"])
    vocab = int(hfcfg.qformer_config.vocab_size)
    model = build_model(cfg, (1, n_tok, vis_dim), geo_dim, sig_dim).to(device)
    ema = EMA(model, cfg["ema_decay"])
    opt = build_optimizer(model, cfg)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9 if amp else None
    results = []

    def batch(bs):
        ids = torch.randint(1000, vocab - 1, (bs, L), device=device)
        ids[:, 0], ids[:, -1] = 101, 102
        r = torch.rand(bs, 4, device=device) * 0.5
        r[:, 2:] = r[:, :2] + 0.3
        t = {"vis": torch.randn(bs, n_tok, vis_dim, device=device).half(),
             "crop": torch.randn(bs, n_crop, vis_dim, device=device).half(),
             "input_ids": ids, "attn_mask": torch.ones(bs, L, dtype=torch.long, device=device),
             "labels": (torch.rand(bs, NUM_CLASSES, device=device) < 0.1).float(),
             "kd": torch.rand(bs, NUM_CLASSES, device=device),
             "roi": r, "geo": torch.rand(bs, geo_dim, device=device),
             "vad": torch.rand(bs, 3, device=device)}
        if n_sig:
            t["sig"] = torch.randn(bs, n_sig, sig_dim, device=device).half()
        return t

    for bs in spec["batch_sizes"]:
        rec = {"batch_size": int(bs), "ok": False}
        try:
            if amp:
                cuda_cleanup()
                torch.cuda.reset_peak_memory_stats()
            model.train()
            times = []
            for step in range(int(spec.get("steps", 3))):
                t = batch(bs)
                t0 = time.time()
                target = 0.5 * t["labels"] + 0.5 * t["kd"]
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                    logits, vadp = model(t["vis"], t["input_ids"], t["attn_mask"], t["roi"], t["geo"],
                                         t["crop"], t.get("sig"), return_vad=True)
                loss = F.binary_cross_entropy_with_logits(logits.float(), target) + \
                    0.2 * F.mse_loss(vadp.float(), t["vad"])
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                ema.update(model)
                if amp:
                    torch.cuda.synchronize()
                times.append(time.time() - t0)
                del t, target, logits, vadp, loss
            model.eval()
            with torch.no_grad():
                t = batch(int(spec["eval_batch_size"]))
                p = torch.sigmoid(_model_logits(model, t, device, amp))
            if amp:
                torch.cuda.synchronize()
            rec.update(ok=bool(torch.isfinite(p).all()),
                       peak_gb=(torch.cuda.max_memory_allocated() / 1e9 if amp else None),
                       reserved_gb=(torch.cuda.max_memory_reserved() / 1e9 if amp else None),
                       sec_per_micro=float(np.median(times[1:] if len(times) > 1 else times)))
            del t, p
        except Exception as e:          # torch.cuda.OutOfMemoryError included
            rec.update(ok=False, error=f"{type(e).__name__}: {str(e)[:300]}")
        opt.zero_grad(set_to_none=True)
        cuda_cleanup()
        rec["fits"] = bool(rec["ok"] and (total is None or
                                          rec["reserved_gb"] <= total - float(spec.get("headroom_gb", 1.5))))
        results.append(rec)
        log(f"[preflight] micro-batch {bs}: " + json.dumps(rec))
        if rec["fits"]:
            break                       # sizes are tried largest first
    write_json(out_path, {"device": str(device), "total_gb": total, "text_len": L,
                          "vis_tokens": n_tok, "crop_tokens": n_crop, "sig_tokens": n_sig, "sig_dim": sig_dim,
                          "vis_dim": vis_dim, "results": results, "core_version": CORE_VERSION,
                          "n_params": int(sum(p.numel() for p in model.parameters()))})
    return 0


# ══════════════════════════════════════════════════════════════════════════════
#  Feature extraction (frozen EVA-ViT-g + frozen SigLIP-2, run once per session)
# ══════════════════════════════════════════════════════════════════════════════
CROP_FILL = (124, 116, 104)        # ImageNet mean colour for padding outside the image


class ImageBoxDataset(_torch_dataset_base()):
    """
    View 1: the full image with the subject's red box drawn (paper-parity input), EVA 224 px.
    View 2: a square crop around the person (side = max(w, h) x margin, aspect
            kept, padded with the mean colour where it leaves the image), no box, EVA 224 px.
    View 3: the same crop through the SigLIP processor (384 px)       (sig_processor given)
    """

    def __init__(self, paths, boxes, global_idx, image_processor, crop_margin=1.2, sig_processor=None):
        self.paths, self.boxes = paths, boxes
        self.gidx, self.ip = global_idx, image_processor
        self.margin = float(crop_margin)
        self.sip = sig_processor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        import torch
        from PIL import Image, ImageDraw
        ok = True
        try:
            img = Image.open(str(self.paths[i])).convert("RGB")
        except Exception:
            img, ok = Image.new("RGB", (224, 224), (128, 128, 128)), False
        W, H = img.size
        xmin, ymin, xmax, ymax = [int(v) for v in self.boxes[i]]
        x1 = max(0, min(xmin, W - 2)); y1 = max(0, min(ymin, H - 2))
        x2 = max(x1 + 1, min(xmax, W - 1)); y2 = max(y1 + 1, min(ymax, H - 1))
        # person crop first (from the clean image)
        side = max(16.0, max(x2 - x1, y2 - y1) * self.margin)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        S = int(round(side))
        Lx, Ty = int(round(cx - side / 2.0)), int(round(cy - side / 2.0))
        crop = Image.new("RGB", (S, S), CROP_FILL)
        ix1, iy1, ix2, iy2 = max(0, Lx), max(0, Ty), min(W, Lx + S), min(H, Ty + S)
        if ix2 > ix1 and iy2 > iy1:
            crop.paste(img.crop((ix1, iy1, ix2, iy2)), (ix1 - Lx, iy1 - Ty))
        ps = (self.sip(images=[crop], return_tensors="pt")["pixel_values"][0] if self.sip is not None
              else torch.zeros(0))
        # full image with the red box
        lw = max(2, int(round(min(W, H) / 150)))
        ImageDraw.Draw(img).rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=lw)
        pv = self.ip(images=[img, crop], return_tensors="pt")["pixel_values"]
        roi = torch.tensor([x1 / W, y1 / H, x2 / W, y2 / H], dtype=torch.float32)
        return pv[0], pv[1], ps, roi, torch.tensor(ok), torch.tensor(int(self.gidx[i]))


def pool_tokens(h, pool):
    """[B, 1 + g*g, D] -> CLS + (g/pool)^2 average-pooled patch tokens."""
    import torch
    import torch.nn.functional as F
    if int(pool) <= 1:
        return h
    n, t, d = h.shape
    g = int(math.isqrt(t - 1))
    assert g * g == t - 1 and g % int(pool) == 0, f"cannot pool {t - 1} patches by {pool}"
    x = h[:, 1:, :].float().reshape(n, g, g, d).permute(0, 3, 1, 2)
    x = F.avg_pool2d(x, int(pool)).flatten(2).transpose(1, 2)
    return torch.cat([h[:, :1, :].float(), x], dim=1)


def sig_pool(last, pooled, grid):
    """SigLIP [B, g*g, D] patch tokens (no CLS) + pooled [B, D] -> pooled + grid^2 averaged tokens."""
    import torch
    import torch.nn.functional as F
    n, t, d = last.shape
    g = int(math.isqrt(t))
    assert g * g == t, f"SigLIP tokens {t} are not a square grid"
    x = last.float().reshape(n, g, g, d).permute(0, 3, 1, 2)
    x = F.adaptive_avg_pool2d(x, min(int(grid), g)).flatten(2).transpose(1, 2)
    p = pooled.float() if pooled is not None else last.float().mean(1)
    return torch.cat([p[:, None, :], x], dim=1)


def load_vision_and_qformer(repo, icfg, want_half):
    """Selective load of vision_model.* / qformer.* / query_tokens (skips the 3B LLM)."""
    import torch
    from transformers.models.instructblip.modeling_instructblip import InstructBlipVisionModel
    try:
        from safetensors import safe_open
        try:
            wm = json.load(open(resolve_file(repo, "model.safetensors.index.json")))["weight_map"]
            shards = sorted({f for k, f in wm.items()
                             if k.startswith(("vision_model.", "qformer.")) or k == "query_tokens"})
        except Exception:
            shards = ["model.safetensors"]          # single-file checkpoint
        vis_sd, q_sd, qtok = {}, {}, None
        for sh in shards:
            with safe_open(resolve_file(repo, sh), framework="pt") as f:
                for k in f.keys():
                    if k.startswith("vision_model."):
                        vis_sd[k[len("vision_model."):]] = f.get_tensor(k)
                    elif k.startswith("qformer."):
                        q_sd[k[len("qformer."):]] = f.get_tensor(k).float()
                    elif k == "query_tokens":
                        qtok = f.get_tensor(k).float()
        with torch.device("meta"):
            vm = InstructBlipVisionModel(icfg.vision_config)
        vm.load_state_dict(vis_sd, strict=True, assign=True)
        assert qtok is not None and len(q_sd) > 0, "Q-Former weights not found"
        how = f"selective safetensors ({len(vis_sd)} vision + {len(q_sd)} Q-Former tensors)"
    except Exception as e:
        log(f"{Y}[extract] selective load failed ({type(e).__name__}: {e}); "
            f"falling back to the full model load{Z}")
        from transformers import InstructBlipForConditionalGeneration
        full = InstructBlipForConditionalGeneration.from_pretrained(
            repo, torch_dtype=torch.float16, low_cpu_mem_usage=True)
        vm = full.vision_model
        q_sd = {k: v.detach().float().clone() for k, v in full.qformer.state_dict().items()}
        qtok = full.query_tokens.detach().float().clone()
        del full
        how = "full from_pretrained (fallback)"
    vis_ln = {"weight": vm.post_layernorm.weight.detach().float().cpu().clone(),
              "bias": vm.post_layernorm.bias.detach().float().cpu().clone()}
    vm = vm.half() if want_half else vm.float()
    vm.eval()
    for p in vm.parameters():
        p.requires_grad_(False)
    return vm, q_sd, qtok, vis_ln, how


def load_siglip_vision(repos):
    """The first loadable SigLIP checkpoint of `repos`: vision tower (fp32) + image processor."""
    from transformers import AutoConfig, AutoImageProcessor, SiglipVisionModel
    errors = []
    for repo in repos:
        try:
            conf = AutoConfig.from_pretrained(repo)
            if getattr(conf, "model_type", "") not in ("siglip", "siglip_vision_model"):
                raise ValueError(f"model_type {getattr(conf, 'model_type', '?')} is not a fixed-resolution SigLIP")
            vm = SiglipVisionModel.from_pretrained(repo)
            vm = vm.float().eval()
            for p in vm.parameters():
                p.requires_grad_(False)
            ip = AutoImageProcessor.from_pretrained(repo)
            return vm, ip, str(repo)
        except Exception as e:
            errors.append(f"{repo}: {type(e).__name__}: {str(e)[:300]}")
            log(f"{Y}[extract] SigLIP {repo} not usable ({type(e).__name__}); trying the next candidate{Z}")
    raise RuntimeError("no SigLIP checkpoint could be loaded — " + " | ".join(errors))


def extract_features(man):
    import torch
    from transformers import InstructBlipConfig, InstructBlipProcessor
    t0 = time.time()
    torch.set_num_threads(2)
    feat_dir = Path(man["feat_dir"])
    feat_dir.mkdir(parents=True, exist_ok=True)
    repo = man["repo"]
    pool = int(man.get("crop_pool", 2))
    margin = float(man.get("crop_margin", 1.2))
    sig_repos = list(man.get("siglip_repos") or [])
    sig_grid = int(man.get("sig_grid", 9))
    n_gpu = torch.cuda.device_count()
    devices = [torch.device(f"cuda:{i}") for i in range(min(2, n_gpu))] or [torch.device("cpu")]
    use_half = devices[0].type == "cuda"

    icfg = InstructBlipConfig.from_pretrained(repo)
    icfg.save_pretrained(str(feat_dir / "hf_config"))
    vcfg = icfg.vision_config
    g = vcfg.image_size // vcfg.patch_size
    n_tok = g * g + 1
    n_crop = (g // pool) ** 2 + 1 if pool > 1 else n_tok
    hid = vcfg.hidden_size
    log(f"[extract] EVA vision {vcfg.num_hidden_layers} layers x {hid}d | image view {n_tok} tokens, "
        f"person-crop view {n_crop} tokens (pool {pool}, margin {margin}) | devices {[str(d) for d in devices]}")

    init_path = feat_dir / "qformer_init.pt"
    vm, q_sd, qtok, vis_ln, how = load_vision_and_qformer(repo, icfg, use_half)
    atomic_torch_save({"qformer": q_sd, "query_tokens": qtok, "vis_ln": vis_ln}, init_path)
    log(f"[extract] weights: {how}; Q-Former init saved ({len(q_sd)} tensors) in {time.time() - t0:.0f}s")
    del q_sd

    ip = InstructBlipProcessor.from_pretrained(repo).image_processor
    log(f"[extract] EVA image processor: {type(ip).__name__} size {getattr(ip, 'size', None)}")
    sig_models, sip, sig_repo, n_sig, sig_dim = [], None, None, 0, 0
    if sig_repos:
        t_s = time.time()
        svm, sip, sig_repo = load_siglip_vision(sig_repos)
        sv = svm.config
        sg = sv.image_size // sv.patch_size
        n_sig, sig_dim = min(sig_grid, sg) ** 2 + 1, int(sv.hidden_size)
        log(f"[extract] SigLIP {sig_repo}: {sv.num_hidden_layers} layers x {sig_dim}d at {sv.image_size} px "
            f"({sg}x{sg} patches) -> pooled + {min(sig_grid, sg)}x{min(sig_grid, sg)} = {n_sig} tokens | "
            f"processor {type(sip).__name__} {getattr(sip, 'size', None)} | loaded in {time.time() - t_s:.0f}s")
        for k, dev in enumerate(devices):
            m = svm if k == len(devices) - 1 else copy.deepcopy(svm)
            sig_models.append(m.to(dev))
        del svm
    models = []
    for k, dev in enumerate(devices):
        m = vm if k == len(devices) - 1 else copy.deepcopy(vm)
        models.append(m.to(dev))
    del vm

    bs = int(man.get("batch_size", 32))
    nw = int(man.get("workers_per_device", 2))
    summary, sig_retries = {}, 0
    for sp in man["splits"]:
        name = sp["name"]
        out = feat_dir / f"{name}_feats.npy"
        outc = feat_dir / f"{name}_crop.npy"
        outs = feat_dir / f"{name}_sig.npy"
        done = feat_dir / f"{name}_done.json"
        if out.exists() and outc.exists() and done.exists() and (outs.exists() or not sig_models):
            log(f"[extract] {name}: already complete — skipped")
            summary[name] = read_json(done)
            continue
        z = np.load(sp["index_path"], allow_pickle=False)
        paths, boxes = z["paths"], z["boxes"].astype(np.int64)
        n = len(paths)
        tmp = feat_dir / f"{name}_feats.partial.npy"
        tmpc = feat_dir / f"{name}_crop.partial.npy"
        tmps = feat_dir / f"{name}_sig.partial.npy"
        mm = np.lib.format.open_memmap(str(tmp), mode="w+", dtype=np.float16, shape=(n, n_tok, hid))
        mc = np.lib.format.open_memmap(str(tmpc), mode="w+", dtype=np.float16, shape=(n, n_crop, hid))
        ms = (np.lib.format.open_memmap(str(tmps), mode="w+", dtype=np.float16, shape=(n, n_sig, sig_dim))
              if sig_models else None)
        roi = np.zeros((n, 4), np.float32)
        ok = np.ones(n, bool)
        shards = [s for s in np.array_split(np.arange(n), len(devices)) if len(s)]
        loaders = []
        for k, s in enumerate(shards):
            ds = ImageBoxDataset(paths[s], boxes[s], s, ip, margin, sip)
            loaders.append(iter(torch.utils.data.DataLoader(
                ds, batch_size=bs, shuffle=False, num_workers=nw,
                pin_memory=devices[k].type == "cuda")))
        active = list(range(len(shards)))
        t_split, done_n, nb = time.time(), 0, 0
        while active:
            launched = []
            for k in list(active):
                try:
                    pv, pc, ps, r, okb, gi = next(loaders[k])
                except StopIteration:
                    active.remove(k)
                    continue
                dev = devices[k]
                x = torch.cat([pv, pc]).to(dev, non_blocking=True)
                if use_half:
                    x = x.half()
                with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16,
                                                     enabled=use_half):
                    h = models[k](pixel_values=x).last_hidden_state
                nb_ = pv.size(0)
                hc = pool_tokens(h[nb_:], pool)
                hs = None
                if sig_models:
                    xs = ps.to(dev, non_blocking=True).float()
                    with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16,
                                                         enabled=use_half):
                        o = sig_models[k](pixel_values=xs)
                    hs = sig_pool(o.last_hidden_state, o.pooler_output, sig_grid)
                    if not bool(torch.isfinite(hs).all()):       # fp16 overflow: redo this batch in fp32
                        sig_retries += 1
                        with torch.no_grad():
                            o = sig_models[k](pixel_values=xs)
                        hs = sig_pool(o.last_hidden_state, o.pooler_output, sig_grid)
                    del o
                launched.append((h[:nb_], hc, hs, r, okb, gi))
            for h, hc, hs, r, okb, gi in launched:   # both GPUs ran concurrently above
                gidx = gi.numpy()
                a = h.to(torch.float16).cpu().numpy()
                c = hc.to(torch.float16).cpu().numpy()
                if not (np.isfinite(a).all() and np.isfinite(c).all()):
                    raise RuntimeError(f"non-finite EVA features in split {name}")
                mm[gidx] = a
                mc[gidx] = c
                if hs is not None:
                    s_ = hs.to(torch.float16).cpu().numpy()
                    if not np.isfinite(s_).all():
                        raise RuntimeError(f"non-finite SigLIP features in split {name} (fp32 retry included)")
                    ms[gidx] = s_
                roi[gidx] = r.numpy()
                ok[gidx] = okb.numpy()
                done_n += len(gidx)
                nb += 1
            del launched
            if nb and nb % 60 < len(shards) and active:
                el = time.time() - t_split
                log(f"[extract] {name}: {done_n:,}/{n:,} ({100 * done_n / n:.0f}%)  "
                    f"{done_n / max(el, 1e-9):.0f} persons/s  "
                    f"eta {(n - done_n) / max(done_n / max(el, 1e-9), 1e-9) / 60:.1f}m")
        mm.flush()
        mc.flush()
        del mm, mc
        if ms is not None:
            ms.flush()
            del ms
            os.replace(tmps, outs)
        os.replace(tmp, out)
        os.replace(tmpc, outc)
        np.save(feat_dir / f"{name}_roi.npy", roi)
        chk = np.load(out, mmap_mode="r")
        chc = np.load(outc, mmap_mode="r")
        pick = np.sort(np.random.RandomState(0).choice(n, size=min(n, 256), replace=False))
        smp = np.asarray(chk[pick], np.float32)
        smc = np.asarray(chc[pick], np.float32)
        info = {"n": n, "shape": list(chk.shape), "crop_shape": list(chc.shape), "dtype": str(chk.dtype),
                "failed_images": int((~ok).sum()),
                "finite": bool(np.isfinite(smp).all() and np.isfinite(smc).all()),
                "mean_abs": float(np.abs(smp).mean()), "std": float(smp.std()),
                "crop_mean_abs": float(np.abs(smc).mean()), "seconds": time.time() - t_split}
        if sig_models:
            chs = np.load(outs, mmap_mode="r")
            sms = np.asarray(chs[pick], np.float32)
            info.update(sig_shape=list(chs.shape), sig_mean_abs=float(np.abs(sms).mean()),
                        sig_finite=bool(np.isfinite(sms).all()))
            del chs
        del chk, chc
        write_json(done, info)
        summary[name] = info
        cuda_cleanup()
        log(f"{G}[extract] {name}: {n:,} done in {info['seconds'] / 60:.1f}m  "
            f"mean|x| image {info['mean_abs']:.3f} crop {info['crop_mean_abs']:.3f}"
            + (f" SigLIP {info['sig_mean_abs']:.3f}" if sig_models else "")
            + f"  failed images {info['failed_images']}{Z}")
    write_json(feat_dir / "extract_done.json", {
        "splits": summary, "seconds": time.time() - t0, "weights": how,
        "core_version": CORE_VERSION, "n_tokens": n_tok, "crop_tokens": n_crop, "hidden": hid,
        "crop_pool": pool, "crop_margin": margin, "devices": [str(d) for d in devices],
        "siglip_repo": sig_repo, "sig_tokens": n_sig, "sig_dim": sig_dim, "sig_grid": sig_grid,
        "sig_fp32_retries": sig_retries})
    log(f"{G}[extract] ALL DONE in {(time.time() - t0) / 60:.1f} min{Z}")
    return 0


# ══════════════════════════════════════════════════════════════════════════════
#  CLI
# ══════════════════════════════════════════════════════════════════════════════
def main(argv=None):
    ap = argparse.ArgumentParser(description="EMOTIC worker")
    sub = ap.add_subparsers(dest="cmd", required=True)
    for c in ("train", "evaluate", "preflight", "soup"):
        s = sub.add_parser(c)
        s.add_argument("--spec", required=True)
    s = sub.add_parser("extract")
    s.add_argument("--manifest", required=True)
    a = ap.parse_args(argv)
    quiet_transformers()
    if a.cmd == "preflight":
        return preflight(json.loads(Path(a.spec).read_text()))
    if a.cmd == "soup":
        return soup_run(json.loads(Path(a.spec).read_text()))
    if a.cmd == "extract":
        man = json.loads(Path(a.manifest).read_text())
        try:
            return extract_features(man)
        except Exception:
            log(f"{R}{traceback.format_exc()}{Z}")
            write_json(Path(man["feat_dir"]) / "extract_failed.json",
                       {"error": traceback.format_exc()[-2000:], "time": time.time()})
            return 2
    cfg = merged_config(json.loads(Path(a.spec).read_text()))
    if a.cmd == "train":
        return train_run(cfg)
    return evaluate_run(cfg)


if __name__ == "__main__":
    sys.exit(main())
'''

# ──────────────────────────────────────────────────────────────────────────────
# 6.1  WRITE + IMPORT THE CORE LIBRARY
# ──────────────────────────────────────────────────────────────────────────────
print("6.1  CORE LIBRARY\n")
_tmp = CFG.core_path.with_name(CFG.core_path.name + ".tmp")
_tmp.write_text(CORE_SRC)
os.replace(_tmp, CFG.core_path)
CORE_SHA1 = hashlib.sha1(CFG.core_path.read_bytes()).hexdigest()
_spec = importlib.util.spec_from_file_location("emotic_core", str(CFG.core_path))
core = importlib.util.module_from_spec(_spec)
sys.modules["emotic_core"] = core
_spec.loader.exec_module(core)
V.note("written", f"{CFG.core_path}  ({CFG.core_path.stat().st_size / 1e3:.0f} kB, sha1 {CORE_SHA1[:10]})")
V.note("version", core.CORE_VERSION)
V.check("core file matches the embedded source", CFG.core_path.read_text() == CORE_SRC, "identical", "identical")
V.check("core label order equals notebook label order", core.EMOTION_LABELS == EMOTION_LABELS,
        "same" if core.EMOTION_LABELS == EMOTION_LABELS else "DIFFERENT", "same")
_miss = [k for k in CFG.train if k not in core.DEFAULTS]
V.check("every training setting is known to the worker", not _miss, _miss or "none unknown", "none unknown")

# ──────────────────────────────────────────────────────────────────────────────
# 6.2  METRICS, LR SCHEDULES, BATCHING, INITIALISATION
# ──────────────────────────────────────────────────────────────────────────────
print("\n6.2  METRICS, LR SCHEDULES, BATCHING, INITIALISATION\n")
from sklearn.metrics import average_precision_score
_rs = np.random.RandomState(0)
_y = Y_TESTLIKE_VAL
_rand = np.mean([core.compute_map(_rs.rand(*_y.shape), _y) for _ in range(10)])
_floor = core.prevalence_floor(_y)
V.note("test-like val", f"random-ranker mAP {_rand:.2f} vs floor {_floor:.2f}")
_tol_floor = 0.40 if TEST_MODE else 0.10      # small samples bias random AP upwards
V.check("random scores land on the prevalence floor", abs(_rand - _floor) / _floor < _tol_floor,
        f"{_rand:.2f}", f"{_floor:.2f} ± {100 * _tol_floor:.0f}%")
_pv = PREVALENCE["val_testlike"]
_floor_nb = float(_pv[(_pv > 0) & (_pv < 1)].mean() * 100.0)
V.check("core floor equals Block 03 floor", abs(_floor - _floor_nb) < 1e-4, f"{_floor:.4f}", f"{_floor_nb:.4f}")
V.check("perfect scores give 100 mAP", abs(core.compute_map(_y + 0.0, _y) - 100.0) < 1e-6,
        f"{core.compute_map(_y + 0.0, _y):.2f}", "100.00")
# tie handling: heavily rounded scores, as fp16 logits produce
_ty = (_rs.rand(2000, NUM_CLASSES) < np.linspace(0.4, 0.02, NUM_CLASSES)).astype(float)
_ts = np.round(_rs.rand(2000, NUM_CLASSES) * 0.3 + _ty * _rs.rand(2000, NUM_CLASSES) * 0.4, 2)
_sk = np.mean([average_precision_score(_ty[:, c], _ts[:, c]) for c in range(NUM_CLASSES)]) * 100
_mine = core.exact_map(_ts, _ty)
V.check("bootstrap AP code equals sklearn with tied scores", abs(_mine - _sk) < 1e-9,
        f"{_mine:.6f} vs {_sk:.6f}", "identical")
_b1, _b2 = core.bootstrap_map(_ts, _ty, B=50, seed=3), core.bootstrap_map(_ts, _ty, B=50, seed=3)
V.check("bootstrap resamples are reproducible (paired tests)", np.array_equal(_b1, _b2), "same", "same")
_r = core.evaluate_scores(_rs.rand(*_y.shape), _y)
V.check("full metric bundle computes", all(k in _r for k in ("mAP", "lift", "AUC", "micro_f1", "P@3")),
        f"{len(_r)} keys", "mAP/lift/AUC/F1/P@k")

# cosine: warm-up then decay to 0 exactly at the end
_opt = torch.optim.AdamW([torch.nn.Parameter(torch.zeros(1))], lr=1e-4)
_cs = core.WarmupCosine(_opt, warmup_steps=4, total_steps=20)
_lrs = [_opt.param_groups[0]["lr"]]
for _ in range(20):
    _cs.step()
    _lrs.append(_opt.param_groups[0]["lr"])
V.check("cosine LR: linear warm-up, peak 1e-4, 0 at the last step",
        np.allclose(_lrs[:4], [2.5e-5, 5e-5, 7.5e-5, 1e-4]) and max(_lrs) <= 1e-4 + 1e-12 and _lrs[-1] < 1e-12,
        f"{_lrs[0]:.1e} → {max(_lrs):.1e} → {_lrs[-1]:.1e}", "2.5e-05 → 1e-04 → 0")
_opt2 = torch.optim.AdamW([torch.nn.Parameter(torch.zeros(1))], lr=1e-4)
_s = core.WarmupPlateau(_opt2, warmup_steps=4, factor=0.5, patience=2, min_scale=0.25, min_delta=0.01)
for _ in range(5):
    _s.step()
_red = [_s.epoch_end(m) for m in [30.0, 31.0, 31.005, 30.9, 30.8, 30.7, 30.6, 30.5, 30.4]]
V.check("plateau LR: halves after 2 flat epochs, floors at 0.25×",
        _red == [False, False, False, True, False, True, False, False, False] and _s.scale == 0.25,
        f"reductions at {[i for i, r in enumerate(_red) if r]}", "[3, 5]")
_c60 = dict(core.DEFAULTS, max_epochs=60)
V.check("resume signature: cosine depends on max_epochs, plateau does not",
        core.run_signature(_c60, 370, 370) != core.run_signature(dict(_c60, max_epochs=80), 370, 370)
        and core.run_signature(dict(_c60, lr_schedule="plateau"), 370, 370)
        == core.run_signature(dict(_c60, lr_schedule="plateau", max_epochs=80, patience=30), 370, 370),
        "as designed", "cosine ≠, plateau =")
V.check("soup key: seeds of one configuration match, other settings do not",
        core.soup_key(core.run_signature(dict(_c60, seed=1), 370, 370))
        == core.soup_key(core.run_signature(dict(_c60, seed=5), 370, 370))
        and core.soup_key(core.run_signature(dict(_c60, seed=1), 370, 370))
        != core.soup_key(core.run_signature(dict(_c60, seed=1, use_sig=True), 370, 370, sig_tokens=82)),
        "as designed", "seed ignored, streams not")

# length-bucketed batches on the REAL train descriptions
_off = np.load(CFG.array_dir / "train_ids_off.npy")
_len = np.diff(_off)
_n = len(_len)
_bs = 32 if not TEST_MODE else 8
_bb = core.BucketBatchSampler(np.arange(_n), _len, _bs, seed=3, epoch=2)
_bp = core.BucketBatchSampler(np.arange(_n), _len, _bs, seed=3, epoch=2, bucket=False)
_rows = np.concatenate([np.asarray(b) for b in _bb])
V.check("bucketed batches: each row at most once, only the last partial batch dropped",
        len(_rows) == len(set(_rows.tolist())) == (_n // _bs) * _bs and all(len(b) == _bs for b in _bb),
        f"{len(_rows):,} rows in {len(_bb)} batches", f"{(_n // _bs) * _bs:,} rows")
_again = core.BucketBatchSampler(np.arange(_n), _len, _bs, seed=3, epoch=2)
_resume = core.BucketBatchSampler(np.arange(_n), _len, _bs, seed=3, epoch=2, start=7)
V.check("bucketed batches: same (seed, epoch) → same batches; resume at batch 7 continues exactly",
        [list(b) for b in _bb] == [list(b) for b in _again] and [list(b) for b in _bb][7:] == [list(b) for b in _resume],
        "identical", "identical")
V.check("bucketed batches: a new epoch or seed reshuffles",
        [list(b) for b in core.BucketBatchSampler(np.arange(_n), _len, _bs, 3, 3)] != [list(b) for b in _bb]
        and [list(b) for b in core.BucketBatchSampler(np.arange(_n), _len, _bs, 4, 2)] != [list(b) for b in _bb],
        "different", "different")
_pad_b = float(np.mean([_len[b].max() for b in (np.asarray(x) for x in _bb)]))
_pad_p = float(np.mean([_len[b].max() for b in (np.asarray(x) for x in _bp)]))
_q = CFG.num_query_tokens
PADDING_GAIN = (_pad_p + _q) / (_pad_b + _q)
V.note("padded sequence per batch (32 queries + text)", f"plain {_pad_p + _q:.0f} → bucketed {_pad_b + _q:.0f} tokens "
                                                        f"(mean real {float(_len.mean()) + _q:.0f})")
V.check("bucketing shortens the padded sequence", _pad_b < _pad_p, f"{PADDING_GAIN:.2f}x shorter", "> 1x")
_eb = core.eval_batches(np.arange(_n), _len, 64)
_er = np.concatenate([np.asarray(b) for b in _eb])
V.check("evaluation batches cover every row exactly once", np.array_equal(np.sort(_er), np.arange(_n)),
        f"{len(_er):,}", f"{_n:,}")

# ──────────────────────────────────────────────────────────────────────────────
# 6.3  WORKER READS WHAT THE NOTEBOOK REPORTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n6.3  WORKER DATA VIEW\n")
for _s_, _Yref in (("train", TRAIN_LABELS), ("val", VAL_LABELS), ("test", TEST_LABELS)):
    _a = core.SplitArrays(CFG.array_dir, _s_)
    V.check(f"[{_s_}] worker labels == notebook labels", np.array_equal(_a.labels, _Yref),
            _a.labels.shape, _Yref.shape)
    if _s_ == "val":
        V.check("[val] worker test-like labels == notebook", np.array_equal(_a.labels_tl, Y_TESTLIKE_VAL),
                "equal" if np.array_equal(_a.labels_tl, Y_TESTLIKE_VAL) else "DIFFERENT", "equal")
    _fin = np.isfinite(_a.vad)
    _vv = _a.vad[_fin]
    _miss = int((~_fin).any(1).sum())
    V.check(f"[{_s_}] VAD targets: shape ok, every known value in [0, 1]",
            _a.vad.shape == (_a.n, 3) and bool(((_vv >= 0) & (_vv <= 1)).all()),
            f"{_a.vad.shape}, range [{_vv.min():.2f}, {_vv.max():.2f}]", f"({_a.n}, 3), [0, 1]")
    V.check(f"[{_s_}] token rows within {CFG.max_text_len}", int(_a.lengths.max()) <= CFG.max_text_len,
            int(_a.lengths.max()), f"<= {CFG.max_text_len}")
    if _s_ == "train":
        V.note("[train] persons without VAD", f"{_miss:,} (masked if a VAD loss is ever used)")
        _folds = np.load(CFG.array_dir / "train_folds.npy")
        V.check("[train] fold file matches the train rows", _folds.shape == (CFG.kd_repeats, _a.n)
                and int(_folds.min()) == 0 and int(_folds.max()) == CFG.kd_folds - 1,
                _folds.shape, (CFG.kd_repeats, _a.n))
        _tr, _ho, _ex = core.train_rows({"fold": 0, "fold_repeat": 0,
                                         "fold_file": str(CFG.array_dir / "train_folds.npy")}, _a.n)
        V.check("[train] teacher rows + held-out rows partition the train split",
                len(_tr) + len(_ho) == _a.n and not set(_tr.tolist()) & set(_ho.tolist()),
                f"{len(_tr):,} + {len(_ho):,}", f"{_a.n:,}")
    del _a

# ──────────────────────────────────────────────────────────────────────────────
# 6.4  CPU SELF-TEST OF THE WORKER
# ──────────────────────────────────────────────────────────────────────────────
print("\n6.4  CPU SELF-TEST OF THE WORKER (tiny synthetic problem, every v7 component on)\n")
ST = Path(tempfile.gettempdir()) / "emotic_selftest"
shutil.rmtree(ST, ignore_errors=True)
for _d in ("arrays", "feats", "runs", "control"):
    (ST / _d).mkdir(parents=True)
_t_st = time.time()
_ST_HID, _ST_SIG, _ST_NSIG = 32, 24, 50


def _tiny_hf_config(out_dir):
    from transformers import InstructBlipConfig
    c = InstructBlipConfig(
        vision_config=dict(hidden_size=_ST_HID, intermediate_size=64, num_hidden_layers=1,
                           num_attention_heads=2, image_size=224, patch_size=14),
        qformer_config=dict(hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                            intermediate_size=128, encoder_hidden_size=_ST_HID, vocab_size=30523,
                            max_position_embeddings=512),
        num_query_tokens=32)
    c.save_pretrained(str(out_dir))


def _make_synth(root, hid=_ST_HID, n=(320, 120, 140)):
    """Learnable synthetic data in the exact on-disk format of Blocks 03, 05 and 07."""
    rs = np.random.RandomState(0)
    C, T, TC = NUM_CLASSES, 257, 65
    Vc = rs.randn(C, hid).astype(np.float32)
    Vp = rs.randn(C, hid).astype(np.float32)
    Vs = rs.randn(C, _ST_SIG).astype(np.float32)
    prev = np.linspace(0.45, 0.04, C)

    def one(split, n_rows, keep):
        z = (rs.rand(n_rows, C) < prev).astype(np.float32)
        z[np.arange(n_rows), rs.randint(0, C, n_rows)] = 1.0
        lab = z * (rs.rand(n_rows, C) < keep) if keep < 1 else z.copy()
        for i in range(n_rows):
            if lab[i].sum() == 0:
                lab[i, np.argmax(z[i])] = 1.0
        x = rs.randn(n_rows, T, hid).astype(np.float32) * 0.7
        x[:, :8, :] += (z @ Vc)[:, None, :] * 3.0
        np.save(root / "feats" / f"{split}_feats.npy", x.astype(np.float16))
        xc = rs.randn(n_rows, TC, hid).astype(np.float32) * 0.7
        xc[:, :4, :] += (z @ Vp)[:, None, :] * 2.0
        np.save(root / "feats" / f"{split}_crop.npy", xc.astype(np.float16))
        xs = rs.randn(n_rows, _ST_NSIG, _ST_SIG).astype(np.float32) * 0.7
        xs[:, :3, :] += (z @ Vs)[:, None, :] * 2.0
        np.save(root / "feats" / f"{split}_sig.npy", xs.astype(np.float16))
        r = rs.rand(n_rows, 4).astype(np.float32) * 0.5
        r[:, 2:] = np.minimum(1.0, r[:, :2] + 0.1 + rs.rand(n_rows, 2).astype(np.float32) * 0.4)
        np.save(root / "feats" / f"{split}_roi.npy", r)
        ids, off = [], [0]
        for i in range(n_rows):
            toks = ([101] + [2000 + c for c in np.where(z[i] > 0)[0] if rs.rand() < 0.6]
                    + list(rs.randint(3000, 9000, rs.randint(5, 40))) + [102])
            ids.extend(toks)
            off.append(len(ids))
        np.save(root / "arrays" / f"{split}_ids_flat.npy", np.array(ids, np.int32))
        np.save(root / "arrays" / f"{split}_ids_off.npy", np.array(off, np.int64))
        np.save(root / "arrays" / f"{split}_labels.npy", lab.astype(np.float32))
        np.save(root / "arrays" / f"{split}_geo.npy", rs.rand(n_rows, 9).astype(np.float32))
        vad = np.clip(0.5 + 0.1 * (z[:, :3] - z[:, 3:6]) + 0.05 * rs.randn(n_rows, 3), 0, 1)
        if split == "train":        # like EMOTIC: a few persons lack VAD (whole row or one value)
            vad[rs.rand(n_rows) < 0.05] = np.nan
            vad[rs.rand(n_rows) < 0.03, 1] = np.nan
        np.save(root / "arrays" / f"{split}_vad.npy", vad.astype(np.float32))
        return z

    one("train", n[0], keep=0.5)
    zv = one("val", n[1], keep=1.0)
    np.save(root / "arrays" / "val_labels_tl.npy", (zv * (rs.rand(*zv.shape) < 0.85)).astype(np.float32))
    one("test", n[2], keep=1.0)
    np.save(root / "arrays" / "protocol_rho.npy", np.linspace(0.36, 0.02, C).astype(np.float32))
    np.save(root / "arrays" / "train_folds.npy", (rs.rand(1, n[0]) < 0.5).astype(np.int8))   # 2 folds


_tiny_hf_config(ST / "feats" / "hf_config")
_q, _hc = core.build_qformer(ST / "feats" / "hf_config", 0.0)
_vis_ln = {"weight": torch.rand(_ST_HID) + 0.5, "bias": torch.randn(_ST_HID) * 0.1}
torch.save({"qformer": _q.state_dict(),
            "query_tokens": torch.randn(1, 32, _hc.qformer_config.hidden_size) * 0.02, "vis_ln": _vis_ln},
           ST / "feats" / "qformer_init.pt")
del _q, _hc
_make_synth(ST)

# shared initialisation: two seeds of one configuration start from identical weights
_icfg = core.merged_config({"hf_config_dir": str(ST / "feats" / "hf_config"), "use_roi": True, "use_geo": True,
                            "use_crop": True, "use_sig": True})
_models = []
for _seed in (7, 8):
    core.seed_everything(int(_icfg["init_seed"]))
    _m = core.build_model(_icfg, (1, 257, _ST_HID), 9, _ST_SIG)
    core.load_pretrained_init(_m, ST / "feats" / "qformer_init.pt")
    core.seed_everything(_seed)
    _models.append(_m.state_dict())
V.check("shared init: every tensor identical across seeds (soups need this)",
        all(torch.equal(_models[0][k], _models[1][k]) for k in _models[0]), f"{len(_models[0])} tensors", "identical")
V.check("SigLIP projection LayerNorm starts as EVA's output LayerNorm",
        torch.equal(_models[0]["sig_proj.1.weight"], _vis_ln["weight"])
        and torch.equal(_models[0]["sig_proj.1.bias"], _vis_ln["bias"]), "copied", "copied")
del _models, _m

_ST_BASE = {"array_dir": str(ST / "arrays"), "feat_dir": str(ST / "feats"),
            "hf_config_dir": str(ST / "feats" / "hf_config"),
            "init_path": str(ST / "feats" / "qformer_init.pt"),
            "control_dir": str(ST / "control"), "batch_size": 16, "grad_accum": 2,
            "eval_batch_size": 32, "num_workers": 0, "max_epochs": 4, "min_epochs": 1,
            "patience": 10, "min_delta": 0.01, "lr_schedule": "cosine",
            "use_roi": True, "use_geo": True, "use_crop": True, "use_sig": True, "use_vad": True,
            "vis_token_drop": 0.25, "txt_token_drop": 0.1, "bucket": True, "bucket_chunk": 4,
            "warmup_epochs": 1.0, "log_every": 5, "floor_check_epoch": 2, "floor_min_lift": 1.05,
            "lr": 1e-3, "backbone_lr_mult": 1.0, "seed": 7, "amp": True}   # tiny random Q-Former
_PLATEAU = dict(lr_schedule="plateau", plateau_patience=1)
_TEACH = dict(fold_file=str(ST / "arrays" / "train_folds.npy"), fold_repeat=0, role="teacher",
              use_vad=False, vis_token_drop=0.0, txt_token_drop=0.0, min_epochs=2, max_epochs=3,
              floor_min_lift=0.0)
_ST_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")


def _st_launch(name, cmd="train", **over):
    d = ST / "runs" / name
    d.mkdir(exist_ok=True)
    spec = dict(_ST_BASE, name=name, config_name=over.pop("config_name", name), run_dir=str(d), **over)
    (d / "spec.json").write_text(json.dumps(spec))
    logf = open(d / f"{cmd}.{len(list(d.glob('*.log')))}.log", "w")
    p = subprocess.Popen([sys.executable, str(CFG.core_path), cmd, "--spec", str(d / "spec.json")],
                         env=_ST_ENV, stdout=logf, stderr=subprocess.STDOUT)
    return p, logf


def _st_soup(name, ingredients):
    d = ST / "runs" / name
    d.mkdir(exist_ok=True)
    (d / "spec.json").write_text(json.dumps({"name": name, "out_dir": str(d), "ingredients": ingredients,
                                             "num_workers": 0, "eval_batch_size": 32}))
    logf = open(d / "soup.log", "w")
    p = subprocess.Popen([sys.executable, str(CFG.core_path), "soup", "--spec", str(d / "spec.json")],
                         env=_ST_ENV, stdout=logf, stderr=subprocess.STDOUT)
    return p, logf


def _st_wait(procs, timeout=1500):
    rcs = {}
    for name, (p, f) in procs.items():
        try:
            rcs[name] = p.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            p.kill()
            rcs[name] = -9
        f.close()
    return rcs


def _st_tail(name, n=25):
    logs = sorted((ST / "runs" / name).glob("*.log"))
    if logs:
        print("\n".join(logs[-1].read_text().splitlines()[-n:]))


def _st_hist(name):
    import csv
    p = ST / "runs" / name / "history.csv"
    return list(csv.DictReader(open(p))) if p.exists() else []


def _st_json(name, f):
    return core.read_json(ST / "runs" / name / f, {}) or {}


def _st_log(name):
    logs = sorted((ST / "runs" / name).glob("*.log"))
    return logs[-1].read_text() if logs else ""


def _same(a, b):
    return len(a) == len(b) > 0 and all(
        abs(float(x["val_tl_mAP"]) - float(y["val_tl_mAP"])) < 1e-6 and
        abs(float(x["train_loss"]) - float(y["train_loss"])) < 1e-6 for x, y in zip(a, b))


# wave 1 (parallel)
(ST / "control" / "STOP_T4_stop").write_text("self-test")
_w1 = {"T1_ref": _st_launch("T1_ref", config_name="Cst"),
       "T2_pause": _st_launch("T2_pause", test_stop_after_steps=13),
       "T3_fp32": _st_launch("T3_fp32", test_inject_nonfinite_every=7, num_workers=2),
       "T4_stop": _st_launch("T4_stop"),
       "T5_deadline": _st_launch("T5_deadline", deadline=time.time() - 1.0),
       "T9_ref": _st_launch("T9_ref", **_PLATEAU),
       "T9_extend": _st_launch("T9_extend", max_epochs=2, **_PLATEAU),
       "T10_early": _st_launch("T10_early", min_delta=100.0, patience=1),
       "T11_oom": _st_launch("T11_oom", test_inject_oom_every=6),
       "T12_f0": _st_launch("T12_f0", fold=0, **_TEACH),
       "T12_f1": _st_launch("T12_f1", fold=1, **_TEACH),
       "T14_seed8": _st_launch("T14_seed8", seed=8, config_name="Cst")}
_rc1 = _st_wait(_w1)
V.note("wave 1 exit codes", _rc1)

_res1, _ev1, _h1 = _st_json("T1_ref", "result.json"), _st_json("T1_ref", "eval.json"), _st_hist("T1_ref")
if not V.check("T1 reference run (all v7 streams) completes",
               _rc1["T1_ref"] == 0 and _res1.get("status") == "done",
               _res1.get("status", f"rc {_rc1['T1_ref']}"), "done"):
    _st_tail("T1_ref")
_lift1 = (_res1.get("best") or {}).get("val_tl_lift", 0.0)
V.check("T1 learns well above the floor", _lift1 > 1.3, f"lift {_lift1:.3f}", "> 1.3")
V.check("T1 SigLIP stream and length buckets in use",
        "+ SigLIP 50x24" in _st_log("T1_ref") and "length-bucketed" in _st_log("T1_ref")
        and "|sig1:50|" in str(_res1.get("signature")) and "|bk1:4|" in str(_res1.get("signature")),
        "yes" if "+ SigLIP" in _st_log("T1_ref") else "no", "yes")
V.check("T1 missing VAD targets are masked (no fp32 retries, nothing skipped)",
        len(_h1) == 4 and all(int(r["fp16_retry"]) == 0 and int(r["skipped"]) == 0 for r in _h1)
        and all(np.isfinite(float(r["vad_loss"])) for r in _h1),
        f"retries {sum(int(r['fp16_retry']) for r in _h1)}, skipped {sum(int(r['skipped']) for r in _h1)}",
        "0, 0")
V.check("T1 cosine LR reaches 0 at the last epoch", len(_h1) == 4 and float(_h1[-1]["lr_scale"]) < 1e-6,
        " ".join(f"{float(r['lr_scale']):.2f}" for r in _h1), "… 0.00")
V.check("T1 test scores sealed after training", (ST / "runs" / "T1_ref" / "scores.npz").exists()
        and _res1.get("scored") is True, _res1.get("scored"), True)
V.check("T1 re-scored val-TL equals the selection value",
        abs(_ev1.get("val_tl_mAP_recomputed", -1) - _ev1.get("val_tl_mAP_at_selection", 1)) < 1e-3,
        f"{_ev1.get('val_tl_mAP_recomputed', float('nan')):.4f} vs "
        f"{_ev1.get('val_tl_mAP_at_selection', float('nan')):.4f}", "equal")
_b1 = torch.load(ST / "runs" / "T1_ref" / "best.pt", map_location="cpu", weights_only=False)
V.check("T1 frees last.pt and archives best.pt in fp16 (cosine runs are not extendable)",
        not (ST / "runs" / "T1_ref" / "last.pt").exists() and _b1["meta"].get("weights_dtype") == "float16",
        _b1["meta"].get("weights_dtype"), "float16")
del _b1
_st2 = _st_json("T2_pause", "status.json")
V.check("T2 pauses mid-epoch and saves state", _rc1["T2_pause"] == 0 and _st2.get("status") == "paused"
        and _st2.get("micro", 0) > 0 and (ST / "runs" / "T2_pause" / "last.pt").exists(),
        f"{_st2.get('status')} ep{_st2.get('epoch')} micro {_st2.get('micro')}", "paused mid-epoch")
_h3, _res3 = _st_hist("T3_fp32"), _st_json("T3_fp32", "result.json")
if not V.check("T3 run with injected overflows completes (2 loader workers)",
               _rc1["T3_fp32"] == 0 and _res3.get("status") == "done", _res3.get("status"), "done"):
    _st_tail("T3_fp32")
_retry3, _skip3 = sum(int(r["fp16_retry"]) for r in _h3), sum(int(r["skipped"]) for r in _h3)
V.check("T3 injected overflows recovered in fp32 (none skipped)", _retry3 > 0 and _skip3 == 0,
        f"retries {_retry3}, skipped {_skip3}", "retries > 0, skipped 0")
_st4 = _st_json("T4_stop", "status.json")
V.check("T4 STOP_<run> file pauses the worker", _rc1["T4_stop"] == 0 and _st4.get("status") == "paused"
        and "STOP" in str(_st4.get("reason")), _st4.get("reason"), "STOP_T4_stop file")
_st5 = _st_json("T5_deadline", "status.json")
V.check("T5 session deadline pauses the worker", _rc1["T5_deadline"] == 0 and _st5.get("status") == "paused"
        and _st5.get("reason") == "session deadline", _st5.get("reason"), "session deadline")
_res9 = _st_json("T9_extend", "result.json")
V.check("T9 plateau run stops at its 2-epoch limit as extendable", _res9.get("stop_reason") == "max_epochs"
        and _res9.get("extendable") is True, f"{_res9.get('stop_reason')}, extendable {_res9.get('extendable')}",
        "max_epochs, True")
_res10 = _st_json("T10_early", "result.json")
V.check("T10 early stopping ends the run after `patience` flat epochs",
        _rc1["T10_early"] == 0 and _res10.get("stop_reason") == "early_stop" and _res10.get("epochs_run") == 2,
        f"{_res10.get('stop_reason')} after {_res10.get('epochs_run')} epochs", "early_stop after 2")
_h11, _res11 = _st_hist("T11_oom"), _st_json("T11_oom", "result.json")
_oom11 = sum(int(r["oom"]) for r in _h11)
V.check("T11 CUDA OOM (injected) recovered in halves, nothing skipped",
        _rc1["T11_oom"] == 0 and _res11.get("status") == "done" and _oom11 > 0
        and sum(int(r["skipped"]) for r in _h11) == 0, f"{_oom11} OOM events, status {_res11.get('status')}",
        "> 0 events, done")
# T12: fold teachers
_t12 = {k: (_st_json(k, "result.json"), _st_json(k, "eval.json")) for k in ("T12_f0", "T12_f1")}
_ok12 = all(_rc1[k] == 0 and r.get("status") == "done" and r.get("role") == "teacher" and r.get("scored") is True
            for k, (r, _) in _t12.items())
if not V.check("T12 two fold teachers finish and score their held-out fold", _ok12,
               [r.get("status") for r, _ in _t12.values()], "done, scored"):
    for k in _t12:
        _st_tail(k)
_nfold = np.load(ST / "arrays" / "train_folds.npy")
V.check("T12 teachers trained on 4/5-style subsets (their own fold held out)",
        all(r.get("n_train") == int((_nfold[0] != i).sum()) for i, (r, _) in enumerate(_t12.values())),
        [r.get("n_train") for r, _ in _t12.values()], [int((_nfold[0] != i).sum()) for i in range(2)])
V.check("T12 teachers never produce test scores; weights freed after oof.npz",
        all((ST / "runs" / k / "oof.npz").exists() and not (ST / "runs" / k / "scores.npz").exists()
            and not (ST / "runs" / k / "best.pt").exists() for k in _t12),
        "oof only", "oof.npz, no scores.npz, no best.pt")
_ST_KD = ST / "arrays" / "kd_targets.npy"
try:
    _kd_stats = core.assemble_kd_targets([ST / "runs" / k for k in _t12], 320, _ST_KD,
                                         train_labels=np.load(ST / "arrays" / "train_labels.npy"))
    _kd_ok = True
except Exception as _e:
    _kd_stats, _kd_ok = {"error": str(_e)}, False
V.check("soft targets assembled: every train person covered once per repeat", _kd_ok,
        _kd_stats.get("error", f"{_kd_stats.get('n')} persons, sha {_kd_stats.get('sha')}"), "complete")
V.check("teachers rank labelled positives above negatives (out of fold)",
        _kd_ok and _kd_stats["mean_on_positives"] > _kd_stats["mean_on_negatives"],
        f"{_kd_stats.get('mean_on_positives', 0):.3f} vs {_kd_stats.get('mean_on_negatives', 0):.3f}", "pos > neg")
try:
    core.assemble_kd_targets([ST / "runs" / "T12_f0"], 320, ST / "arrays" / "kd_bad.npy")
    _incomplete_refused = False
except RuntimeError:
    _incomplete_refused = True
V.check("incomplete teacher set is refused", _incomplete_refused, _incomplete_refused, True)

# wave 2 (parallel): resumes, extension, crash-during-finalisation, re-scoring, pre-flight,
# student, soups
(ST / "control" / "STOP_T4_stop").unlink()
for _f in ("result.json", "scores.npz", "eval.json"):
    (ST / "runs" / "T3_fp32" / _f).unlink(missing_ok=True)
(ST / "runs" / "T1_ref" / "eval_spec.json").write_text((ST / "runs" / "T1_ref" / "spec_resolved.json").read_text())
_pf_spec = ST / "preflight.json"
_pf_spec.write_text(json.dumps({"hf_config_dir": str(ST / "feats" / "hf_config"), "batch_sizes": [8, 4],
                                "text_len": 64, "vis_tokens": 257, "crop_tokens": 65, "geo_dim": 9,
                                "sig_tokens": _ST_NSIG, "sig_dim": _ST_SIG,
                                "eval_batch_size": 8, "steps": 2, "out": str(ST / "preflight_out.json")}))
_w2 = {"T2_pause": _st_launch("T2_pause"),
       "T4_stop": _st_launch("T4_stop"),
       "T9_extend": _st_launch("T9_extend", max_epochs=4, **_PLATEAU),
       "T3_fp32": _st_launch("T3_fp32", test_inject_nonfinite_every=7, num_workers=2),
       "T13_student": _st_launch("T13_student", kd_path=str(_ST_KD), kd_alpha=0.5, role="student",
                                 use_vad=False, vis_token_drop=0.0, txt_token_drop=0.0),
       "T14_soup": _st_soup("T14_soup", [str(ST / "runs" / "T1_ref"), str(ST / "runs" / "T14_seed8")]),
       "T15_badsoup": _st_soup("T15_badsoup", [str(ST / "runs" / "T1_ref"), str(ST / "runs" / "T9_ref")])}
_p7 = subprocess.Popen([sys.executable, str(CFG.core_path), "evaluate", "--spec",
                        str(ST / "runs" / "T1_ref" / "eval_spec.json")], env=_ST_ENV,
                       stdout=open(ST / "T7.log", "w"), stderr=subprocess.STDOUT)
_p8 = subprocess.Popen([sys.executable, str(CFG.core_path), "preflight", "--spec", str(_pf_spec)],
                       env=_ST_ENV, stdout=open(ST / "T8.log", "w"), stderr=subprocess.STDOUT)
_rc2 = _st_wait(_w2)
_rc7, _rc8 = _p7.wait(timeout=1500), _p8.wait(timeout=1500)
V.note("wave 2 exit codes", dict(_rc2, T7_eval=_rc7, T8_preflight=_rc8))

_h2, _h4 = _st_hist("T2_pause"), _st_hist("T4_stop")
V.check("T2 resumes from the saved micro-batch", _rc2["T2_pause"] == 0 and "RESUMED at epoch" in _st_log("T2_pause"),
        "RESUMED" if "RESUMED at epoch" in _st_log("T2_pause") else "no", "RESUMED")
V.check("T2 paused+resumed run is bit-identical to T1 (bucketed batches, token dropout)", _same(_h1, _h2),
        " | ".join(f"{float(r['val_tl_mAP']):.3f}" for r in _h2), "same as T1 every epoch")
V.check("T4 run resumed after STOP is bit-identical to T1", _rc2["T4_stop"] == 0 and _same(_h1, _h4),
        _st_json("T4_stop", "result.json").get("status"), "done, identical")
_h9, _h9r = _st_hist("T9_extend"), _st_hist("T9_ref")
V.check("T9 plateau run extended 2 → 4 epochs", _rc2["T9_extend"] == 0
        and "EXTENDING from 2 to 4" in _st_log("T9_extend") and len(_h9) == 4, f"{len(_h9)} epochs", "4")
V.check("T9 extended run is bit-identical to a 4-epoch run from the start", _same(_h9r, _h9),
        " | ".join(f"{float(r['val_tl_mAP']):.3f}" for r in _h9), "same as T9_ref")
_res3b = _st_json("T3_fp32", "result.json")
V.check("T6 crash during finalisation → relaunch only finalises",
        _rc2["T3_fp32"] == 0 and "finalising only" in _st_log("T3_fp32") and _res3b.get("status") == "done"
        and (ST / "runs" / "T3_fp32" / "scores.npz").exists(), _res3b.get("status"), "done, no retraining")
_ev7 = _st_json("T1_ref", "eval.json")
_d7 = abs(_ev7.get("val_tl_mAP_recomputed", -9) - _ev7.get("val_tl_mAP_at_selection", 9))
_tol7 = core.SCORE_TOLERANCE[_ev7.get("weights_dtype", "float32")]
V.check("T7 re-scoring fallback reproduces the selection value", _rc7 == 0 and _d7 < _tol7,
        f"|Δ val-TL| {_d7:.3f} ({_ev7.get('weights_dtype')})", f"< {_tol7}")
_pf = core.read_json(ST / "preflight_out.json", {}) or {}
V.check("T8 pre-flight command runs (all streams incl. SigLIP)", _rc8 == 0
        and any(r.get("ok") for r in _pf.get("results", [])) and _pf.get("sig_tokens") == _ST_NSIG,
        [r.get("ok") for r in _pf.get("results", [])], "ok")
_res13, _h13 = _st_json("T13_student", "result.json"), _st_hist("T13_student")
if not V.check("T13 student on the soft targets completes and is scored",
               _rc2["T13_student"] == 0 and _res13.get("status") == "done" and _res13.get("kd") is True
               and (ST / "runs" / "T13_student" / "scores.npz").exists(), _res13.get("status"), "done"):
    _st_tail("T13_student")
V.check("T13 soft targets enter the loss (mean target on train negatives > 0) and the signature",
        len(_h13) > 0 and all(r["kd_neg_target"] != "" and float(r["kd_neg_target"]) > 0 for r in _h13)
        and f"kd0.5:{_kd_stats.get('sha')}" in str(_res13.get("signature")),
        " ".join(f"{float(r['kd_neg_target']):.3f}" for r in _h13 if r["kd_neg_target"] != ""), "> 0")
_sp14 = _st_json("T14_soup", "soup.json")
_ok14 = (_rc2["T14_soup"] == 0 and _sp14.get("status") == "done"
         and (ST / "runs" / "T14_soup" / "soup_scores.npz").exists())
if not V.check("T14 soup of two seeds: uniform + greedy, scores sealed", _ok14, _sp14.get("status"), "done"):
    print((ST / "runs" / "T14_soup" / "soup.log").read_text()[-2000:])
_d14 = abs(_sp14.get("first_ingredient_recomputed", -9) - max((_sp14.get("own_val_tl") or {"": 9}).values()))
V.check("T14 greedy soup's first ingredient reproduces its own val-TL (fp16 weights)", _ok14 and _d14 < 0.30,
        f"|Δ| {_d14:.3f}", "< 0.30")
_sp15 = _st_json("T15_badsoup", "soup.json")
V.check("T15 soup of runs that differ beyond the seed is refused",
        _rc2["T15_badsoup"] != 0 and _sp15.get("status") == "failed" and "differ" in str(_sp15.get("message")),
        str(_sp15.get("message", ""))[:50], "refused")
if not all(v == 0 for v in list(_rc1.values()) + [v for k, v in _rc2.items() if k != "T15_badsoup"] + [_rc7, _rc8]):
    for _n in list(_rc1) + list(_rc2):
        if _rc1.get(_n, 0) != 0 or (_n != "T15_badsoup" and _rc2.get(_n, 0) != 0):
            _st_tail(_n)
    for _lf in ("T7.log", "T8.log"):
        print((ST / _lf).read_text()[-2000:])
V.note("self-test time", f"{time.time() - _t_st:.0f}s")
shutil.rmtree(ST, ignore_errors=True)

# ──────────────────────────────────────────────────────────────────────────────
# 6.5  GPU PRE-FLIGHT (real Q-Former, worst-case batch, all streams)
# ──────────────────────────────────────────────────────────────────────────────
print("\n6.5  GPU PRE-FLIGHT\n")
from transformers import InstructBlipConfig
PREFLIGHT_DIR = CFG.cache_dir / "preflight"
PREFLIGHT_DIR.mkdir(exist_ok=True)
InstructBlipConfig.from_pretrained(CFG.backbone_ckpt).save_pretrained(str(PREFLIGHT_DIR / "hf_config"))
_pf_spec = {"hf_config_dir": str(PREFLIGHT_DIR / "hf_config"), "batch_sizes": [32, 16],
            "text_len": int(CFG.max_text_len), "vis_tokens": int(CFG.vis_tokens),
            "crop_tokens": int(CFG.crop_tokens), "geo_dim": int(GEOMETRY_DIM),
            "sig_tokens": int(CFG.sig_tokens), "sig_dim": int(CFG.sig_dim),
            "eval_batch_size": int(CFG.train["eval_batch_size"]),
            "steps": 3, "headroom_gb": 1.5, "out": str(PREFLIGHT_DIR / "result.json")}
(PREFLIGHT_DIR / "spec.json").write_text(json.dumps(_pf_spec))
_env = dict(os.environ, PYTHONUNBUFFERED="1")
if N_GPUS > 0:
    _env["CUDA_VISIBLE_DEVICES"] = "0"
_t_pf = time.time()
_p = subprocess.run([sys.executable, str(CFG.core_path), "preflight", "--spec", str(PREFLIGHT_DIR / "spec.json")],
                    env=_env, capture_output=True, text=True, timeout=1800)
PREFLIGHT = core.read_json(PREFLIGHT_DIR / "result.json", {}) or {}
for _rec in PREFLIGHT.get("results", []):
    V.note(f"micro-batch {_rec['batch_size']}",
           (f"peak {_rec['peak_gb']:.2f} GB, reserved {_rec['reserved_gb']:.2f} GB of "
            f"{PREFLIGHT['total_gb']:.1f} GB, {_rec['sec_per_micro']:.2f} s/step at {CFG.max_text_len} tokens"
            if _rec.get("ok") and PREFLIGHT.get("total_gb") else json.dumps(_rec)))
if _p.returncode != 0:
    print(_p.stdout[-3000:], _p.stderr[-3000:])
V.check("pre-flight process succeeded", _p.returncode == 0, f"rc {_p.returncode}", 0)
V.note("device", f"{PREFLIGHT.get('device')}  |  model {PREFLIGHT.get('n_params', 0) / 1e6:.1f} M parameters "
                 f"(all streams incl. SigLIP projection)")
_fit = [r for r in PREFLIGHT.get("results", []) if r.get("fits")]
MICRO_BATCH = int(_fit[0]["batch_size"]) if _fit else None
GRAD_ACCUM = (64 // MICRO_BATCH) if MICRO_BATCH else None
V.check("a micro-batch fits in GPU memory", MICRO_BATCH is not None, MICRO_BATCH, "32 or 16")
if MICRO_BATCH:
    V.check("effective batch stays 64", MICRO_BATCH * GRAD_ACCUM == 64, f"{MICRO_BATCH} x {GRAD_ACCUM}", "64")
    EPOCH_MIN_UPPER = len(df_train) / MICRO_BATCH * _fit[0]["sec_per_micro"] / 60.0
    V.note("epoch time upper bound", f"{EPOCH_MIN_UPPER:.1f} min (every batch at {CFG.max_text_len} tokens; "
                                     f"bucketed batches are ~{PADDING_GAIN:.2f}x shorter than plain ones)")
else:
    EPOCH_MIN_UPPER = None
if not TEST_MODE:
    V.check("pre-flight ran on a GPU", str(PREFLIGHT.get("device", "")).startswith("cuda"),
            PREFLIGHT.get("device"), "cuda")
V.note("pre-flight time", f"{time.time() - _t_pf:.0f}s")
(CFG.cache_dir / "preflight.json").write_text(json.dumps(
    {"micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM, "epoch_min_upper": EPOCH_MIN_UPPER,
     "padding_gain": PADDING_GAIN, "core_version": core.CORE_VERSION, "core_sha1": CORE_SHA1,
     "raw": PREFLIGHT}, indent=2))
free_memory()
BLOCK_06_OK = V.finish()


In [ ]:
# Block 07 — Visual Feature Cache: EVA image + EVA person crop + SigLIP-2 person crop
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–06
#
# WHY CACHE
#   Xenos et al. keep the EVA-ViT-g image encoder FROZEN; only the Q-Former and
#   the head train. With no augmentation (Phase A), a frozen encoder's output
#   for a person never changes, so it is computed ONCE per session here instead
#   of ~8 times per run. The features exceed the 20 GB /kaggle/working quota,
#   so they live on the scratch disk and are rebuilt at the start of each
#   session (~35–45 min incl. downloads). Within a session, re-running this
#   block reuses them.
#
# THREE VIEWS PER PERSON (the image is decoded once for all three)
#   1  EVA-ViT-g, full image with the person's red box, 224 px   257 tokens (paper input)
#   2  EVA-ViT-g, square person crop (side 1.2 x max(w, h)), 224 px, 2x2-pooled   65 tokens
#   3  v7: SigLIP-2 so400m/14, the SAME person crop at 384 px: 27x27 patches
#      averaged to 9x9 + the model's pooled (MAP-head) embedding     82 tokens x 1152
#   View 3 is new information, not a re-encoding: 3x the pixel density on the
#   person (faces, hands, posture) from an encoder trained differently from
#   EVA (sigmoid image–text loss on WebLI). SigLIP (v1) so400m is used if
#   SigLIP-2 cannot be downloaded; the model actually used is recorded.
#
#   Only the checkpoint shard holding vision_model / qformer / query_tokens of
#   InstructBLIP is read; the 3B-parameter language model is never loaded. Of
#   SigLIP only the vision tower is loaded.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
from transformers import InstructBlipConfig

header("BLOCK 07 — Visual Feature Cache: EVA image + EVA crop + SigLIP-2 crop  [v7]")
V = BlockVerdict(7, "Feature cache")
TEST_MODE = os.environ.get("EMOTIC_TEST_MODE") == "1"

# ──────────────────────────────────────────────────────────────────────────────
# 7.1  WHERE
# ──────────────────────────────────────────────────────────────────────────────
print("7.1  LOCATION\n")
_icfg = InstructBlipConfig.from_pretrained(CFG.backbone_ckpt)
_vc = _icfg.vision_config
N_TOK = (_vc.image_size // _vc.patch_size) ** 2 + 1
HID = int(_vc.hidden_size)
SPLIT_N = {"train": len(df_train), "val": len(df_val), "test": len(df_test)}
N_CROP = ((_vc.image_size // _vc.patch_size) // CFG.crop_pool) ** 2 + 1
NEED_GB = sum(SPLIT_N.values()) * ((N_TOK + N_CROP) * HID + CFG.sig_tokens * CFG.sig_dim) * 2 / 1e9 + 1.5
V.note("feature geometry", f"EVA image {N_TOK} + EVA crop {N_CROP} tokens × {HID}; SigLIP crop "
                           f"{CFG.sig_tokens} × {CFG.sig_dim}; fp16")
V.note("space needed", f"{NEED_GB:.1f} GB")
if not TEST_MODE:
    V.check("encoder output matches the Q-Former input (257 × 1408)",
            (N_TOK, HID) == (CFG.vis_tokens, CFG.vis_dim), (N_TOK, HID), (CFG.vis_tokens, CFG.vis_dim))
V.check("crop token count matches the config", N_CROP == CFG.crop_tokens, N_CROP, CFG.crop_tokens)

_cands = [os.environ.get("EMOTIC_FEAT_ROOT", ""), "/kaggle/temp", "/tmp", "/kaggle/tmp", "/root", "/var/tmp"]
_work = str(CFG.work_dir.resolve())
FEAT_DIR, _table = None, []
for _c in [c for c in _cands if c]:
    _p = Path(_c)
    _anchor = _p                                              # nearest existing parent
    while not _anchor.exists() and _anchor != _anchor.parent:
        _anchor = _anchor.parent
    if str(_anchor.resolve()).startswith(_work) or str(_p).startswith(_work):
        continue
    if not os.access(str(_anchor), os.W_OK):
        continue
    _fd = _p / "emotic_feats_v7"
    _have = dir_size_gb(_fd) if _fd.exists() else 0.0          # an existing cache counts as space
    _free = disk_free_gb(_anchor) + _have
    _table.append((str(_p), _free))
    if FEAT_DIR is None and _free >= NEED_GB + 3.0:
        FEAT_DIR = _fd
for _c, _f in _table:
    V.note(f"  {_c}", f"{_f:.1f} GB available")
V.check("a scratch location outside /kaggle/working has room", FEAT_DIR is not None,
        str(FEAT_DIR), f">= {NEED_GB + 3:.0f} GB free")
if FEAT_DIR is None:
    BLOCK_07_OK = V.finish()
    raise RuntimeError("No scratch disk with enough free space. Set EMOTIC_FEAT_ROOT to a "
                       "directory outside /kaggle/working with ~45 GB free and re-run Block 07.")
FEAT_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# 7.2  REUSE OR EXTRACT
# ──────────────────────────────────────────────────────────────────────────────
print("\n7.2  EXTRACTION\n")
_meta = json.loads((CFG.array_dir / "meta.json").read_text())
CACHE_KEY = {"order_hash": _meta["order_hash"], "backbone": CFG.backbone_ckpt,
             "n": SPLIT_N, "tokens": N_TOK, "hidden": HID, "crop_tokens": N_CROP,
             "crop_pool": CFG.crop_pool, "crop_margin": CFG.crop_margin,
             "siglip_repos": CFG.siglip_repos, "sig_grid": CFG.sig_grid, "extract": "v7",
             "views": ["eva_image_redbox", "eva_person_crop", "siglip_person_crop"]}
_key_path = FEAT_DIR / "cache_key.json"
_done = core.read_json(FEAT_DIR / "extract_done.json")
_reuse = bool(_done) and core.read_json(_key_path) == CACHE_KEY and all(
    (FEAT_DIR / f"{s}_{v}.npy").exists() for s in SPLIT_N for v in ("feats", "crop", "sig")) and all(
    (FEAT_DIR / f"{s}_done.json").exists() for s in SPLIT_N)
if not _reuse and core.read_json(_key_path) not in (None, CACHE_KEY):
    V.note("stale cache", "key changed (different data order, backbone or views) — rebuilding")
    shutil.rmtree(FEAT_DIR, ignore_errors=True)
    FEAT_DIR.mkdir(parents=True)
core.write_json(_key_path, CACHE_KEY)
(FEAT_DIR / "extract_failed.json").unlink(missing_ok=True)

EXTRACT_RC = 0
if _reuse:
    V.note("cache", f"complete and matching — reused ({_done.get('seconds', 0) / 60:.1f} min saved)")
else:
    _man = {"repo": CFG.backbone_ckpt, "feat_dir": str(FEAT_DIR), "batch_size": 32,
            "workers_per_device": 0 if TEST_MODE else 2,
            "crop_pool": CFG.crop_pool, "crop_margin": CFG.crop_margin,
            "siglip_repos": CFG.siglip_repos, "sig_grid": CFG.sig_grid,
            "splits": [{"name": s, "index_path": str(CFG.array_dir / f"{s}_index.npz")} for s in SPLIT_N]}
    (FEAT_DIR / "manifest.json").write_text(json.dumps(_man, indent=2))
    print(f"  {_B}Extracting on {max(N_GPUS, 1)} device(s). The first minutes download the InstructBLIP "
          f"shard (~10 GB) and SigLIP-2 (~4.5 GB) silently.{_0}\n")
    _t0 = time.time()
    _log = open(CFG.log_dir / "extract.log", "a")
    _proc = subprocess.Popen([sys.executable, str(CFG.core_path), "extract", "--manifest",
                              str(FEAT_DIR / "manifest.json")],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env=dict(os.environ, PYTHONUNBUFFERED="1"))
    try:
        for _line in _proc.stdout:
            _log.write(_line)
            print("  " + _line, end="", flush=True)
        EXTRACT_RC = _proc.wait()
    except KeyboardInterrupt:
        _proc.terminate()
        try:
            _proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            _proc.kill()
        EXTRACT_RC = -2
        print(f"\n  {_Y}Interrupted. Finished splits are kept; re-run this block to continue.{_0}")
    finally:
        _log.close()
    V.note("extraction time", f"{(time.time() - _t0) / 60:.1f} min")
V.check("extraction process succeeded", EXTRACT_RC == 0, EXTRACT_RC, 0)
if (FEAT_DIR / "extract_failed.json").exists():
    print(core.read_json(FEAT_DIR / "extract_failed.json", {}).get("error", "")[-2500:])

# ──────────────────────────────────────────────────────────────────────────────
# 7.3  VERIFY
# ──────────────────────────────────────────────────────────────────────────────
print("\n7.3  VERIFY\n")
_ed = core.read_json(FEAT_DIR / "extract_done.json", {}) or {}
N_SIG, SIG_DIM = int(_ed.get("sig_tokens", 0)), int(_ed.get("sig_dim", 0))
V.note("SigLIP model used", f"{_ed.get('siglip_repo')}  ({N_SIG} tokens × {SIG_DIM}, "
                            f"fp32 retries {_ed.get('sig_fp32_retries', 0)})")
V.check("SigLIP-2 (or the SigLIP fallback) loaded", _ed.get("siglip_repo") is not None,
        _ed.get("siglip_repo"), "a SigLIP checkpoint")
if not TEST_MODE:
    V.check("SigLIP token geometry matches the config", (N_SIG, SIG_DIM) == (CFG.sig_tokens, CFG.sig_dim),
            (N_SIG, SIG_DIM), (CFG.sig_tokens, CFG.sig_dim))
    V.check("primary SigLIP-2 checkpoint used (not the fallback)", _ed.get("siglip_repo") == CFG.siglip_repos[0],
            _ed.get("siglip_repo"), CFG.siglip_repos[0], fatal=False)
FEAT_INFO = {}
for s, n in SPLIT_N.items():
    f = FEAT_DIR / f"{s}_feats.npy"
    if not f.exists():
        V.check(f"[{s}] features present", False, "missing", str(f))
        continue
    mm = np.load(f, mmap_mode="r")
    roi = np.load(FEAT_DIR / f"{s}_roi.npy")
    info = core.read_json(FEAT_DIR / f"{s}_done.json", {}) or {}
    FEAT_INFO[s] = info
    V.check(f"[{s}] EVA image view = persons × {N_TOK} × {HID}, fp16",
            tuple(mm.shape) == (n, N_TOK, HID) and mm.dtype == np.float16, tuple(mm.shape), (n, N_TOK, HID))
    _pick = np.sort(np.random.RandomState(1).choice(n, size=min(n, 64), replace=False))
    _x = np.asarray(mm[_pick], np.float32)
    _rows_zero = int((np.abs(_x).reshape(len(_pick), -1).max(1) == 0).sum())
    V.check(f"[{s}] EVA image features finite, non-trivial, no unwritten rows",
            bool(np.isfinite(_x).all() and _x.std() > 1e-3) and _rows_zero == 0,
            f"std {_x.std():.3f}, empty rows {_rows_zero}", "finite, std > 0, 0 empty")
    V.check(f"[{s}] person boxes normalised to [0, 1]",
            roi.shape == (n, 4) and bool((roi >= 0).all() and (roi <= 1).all()
                                         and (roi[:, 2] > roi[:, 0]).all() and (roi[:, 3] > roi[:, 1]).all()),
            roi.shape, (n, 4))
    _fail = int(info.get("failed_images", -1))
    V.check(f"[{s}] every image decoded", _fail == 0, _fail, 0)
    fc, fs = FEAT_DIR / f"{s}_crop.npy", FEAT_DIR / f"{s}_sig.npy"
    if fc.exists():
        mc = np.load(fc, mmap_mode="r")
        _xc = np.asarray(mc[_pick], np.float32)
        V.check(f"[{s}] EVA crop view = persons × {N_CROP} × {HID}, finite, all rows written",
                tuple(mc.shape) == (n, N_CROP, HID) and mc.dtype == np.float16 and bool(np.isfinite(_xc).all())
                and int((np.abs(_xc).reshape(len(_pick), -1).max(1) == 0).sum()) == 0,
                tuple(mc.shape), (n, N_CROP, HID))
        _rel = float(np.median(np.linalg.norm(_x[:, 0] - _xc[:, 0], axis=1)
                               / (np.linalg.norm(_x[:, 0], axis=1) + 1e-9)))
        V.check(f"[{s}] EVA image and crop views differ (CLS relative difference)", _rel > 0.01,
                f"{_rel:.3f}", "> 0.01")
        del mc
    else:
        V.check(f"[{s}] EVA crop view present", False, "missing", str(fc))
    if fs.exists():
        ms = np.load(fs, mmap_mode="r")
        _xs = np.asarray(ms[_pick], np.float32)
        V.check(f"[{s}] SigLIP crop view = persons × {N_SIG} × {SIG_DIM}, fp16",
                tuple(ms.shape) == (n, N_SIG, SIG_DIM) and ms.dtype == np.float16, tuple(ms.shape), (n, N_SIG, SIG_DIM))
        _sz = int((np.abs(_xs).reshape(len(_pick), -1).max(1) == 0).sum())
        V.check(f"[{s}] SigLIP features finite, non-trivial, no unwritten rows",
                bool(np.isfinite(_xs).all() and _xs.std() > 1e-3) and _sz == 0,
                f"std {_xs.std():.3f}, empty rows {_sz}", "finite, std > 0, 0 empty")
        _pooled = _xs[:, 0]
        _cos = (_pooled / (np.linalg.norm(_pooled, axis=1, keepdims=True) + 1e-9))
        _sim = _cos @ _cos.T
        _off_diag = float(np.median(_sim[~np.eye(len(_sim), dtype=bool)]))
        V.check(f"[{s}] SigLIP pooled embeddings differ between persons", _off_diag < 0.999,
                f"median cosine {_off_diag:.3f}", "< 0.999", fatal=not TEST_MODE)
        del ms
    else:
        V.check(f"[{s}] SigLIP crop view present", False, "missing", str(fs))
    V.note(f"[{s}]", f"{f.stat().st_size / 1e9:.2f} GB + crop {fc.stat().st_size / 1e9 if fc.exists() else 0:.2f} GB"
                     f" + SigLIP {fs.stat().st_size / 1e9 if fs.exists() else 0:.2f} GB, mean|x| "
                     f"{info.get('mean_abs', float('nan')):.3f} / {info.get('crop_mean_abs', float('nan')):.3f} / "
                     f"{info.get('sig_mean_abs', float('nan')):.3f}, {info.get('seconds', 0) / 60:.1f} min")
    del mm

_init = torch.load(FEAT_DIR / "qformer_init.pt", map_location="cpu", weights_only=False)
_nq = len(_init["qformer"])
_qt = tuple(_init["query_tokens"].shape)
V.note("Q-Former init", f"{_nq} tensors, query tokens {_qt}")
V.check("EVA output LayerNorm saved (initialises the SigLIP projection)",
        "vis_ln" in _init and tuple(_init["vis_ln"]["weight"].shape) == (HID,),
        tuple(_init.get("vis_ln", {}).get("weight", torch.zeros(0)).shape), (HID,))
_probe_cfg = core.merged_config({"hf_config_dir": str(FEAT_DIR / "hf_config"), "use_roi": True, "use_geo": True,
                                 "use_crop": True, "use_sig": True})
_probe = core.build_model(_probe_cfg, (1, N_TOK, HID), GEOMETRY_DIM, SIG_DIM)
_n_loaded = core.load_pretrained_init(_probe, FEAT_DIR / "qformer_init.pt")
V.check("pretrained Q-Former loads strictly into the full v7 model", _n_loaded == _nq, _n_loaded, _nq)
V.check("32 pretrained query tokens", _qt[-2] == 32 and _qt[-1] == _icfg.qformer_config.hidden_size,
        _qt, f"(1, 32, {_icfg.qformer_config.hidden_size})")
V.note("full v7 model", f"{sum(p.numel() for p in _probe.parameters()) / 1e6:.1f} M trainable parameters")
del _init, _probe
V.note("weights", _ed.get("weights", "?"))
if not TEST_MODE:
    V.check("InstructBLIP weights read selectively (language model skipped)", "selective" in _ed.get("weights", ""),
            _ed.get("weights", "?")[:40], "selective safetensors", fatal=False)

core.write_json(CFG.cache_dir / "feat_dir.json", {"feat_dir": str(FEAT_DIR), "cache_key": CACHE_KEY,
                                                  "session_start": SESSION_START, "siglip_repo": _ed.get("siglip_repo"),
                                                  "sig_tokens": N_SIG, "sig_dim": SIG_DIM})
V.note("scratch disk left", f"{disk_free_gb(FEAT_DIR):.1f} GB")
free_memory()
BLOCK_07_OK = V.finish()


In [ ]:
# Block 08 — Two-GPU Experiment Scheduler (Phase A v7: NO augmentation)
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–07 in this session
#
# WHAT RUNS (every configuration with 5 seeds; all runs of one configuration
# start from the SAME initial weights and differ in data order and dropout):
#
#   C0_base    InstructBLIP Q-Former on the red-box image + LLaVA description
#              (Xenos et al. architecture), v6 recipe (8-epoch cosine, EMA, wd)
#   C1_person  + EVA person crop (65 tokens), ROI tokens (49), geometry token
#   C4_siglip  + SigLIP-2 person crop at 384 px (82 tokens)          ← full model
#   KD_siglip  C4_siglip trained on cross-fitted soft targets        ← proposed
#              (1 − a)·label + a·teacher, a = 0.5
#   Teachers   T_siglip_r{1,2}f{1..5}: C4_siglip trained on 4 of 5 image-grouped
#              folds, predicting the held-out fold (2 repeats → every train
#              person gets 2 out-of-fold predictions, averaged). Teachers are
#              scored on val (their own model selection) and on their held-out
#              TRAIN fold only — never on test. Their weights are deleted after.
#   Soups      SOUP_<config>: one network = the weight average of a
#              configuration's 5 seeds (uniform), and a greedy soup that adds
#              seeds only while test-like val improves (Wortsman et al. 2022).
#
# ORDER (both GPUs always busy; a job waits only for what it needs)
#   1. the 10 teachers   2. KD students (as soon as all teachers are done) and
#   C4 seeds (fill the gap)   3. soups of C4 / KD   4. C1 and C0 seeds (the
#   rest of the ablation)   5. soups of C1 / C0.
#   Nothing here reads test results. The final configuration is chosen in
#   Block 09 on test-like val only, by a rule fixed in advance.
#
# CHECKPOINTING / CRASH SAFETY (per run, /kaggle/working/runs_v7/<run>/)
#   last.pt   every epoch end + on any pause: weights, EMA, AdamW, LR schedule,
#             AMP scaler, epoch + micro-batch position, best score, early-stop
#             counter, history, RNG states. A crash loses at most one epoch.
#   Pauses save at the exact optimizer step: session deadline (11.5 h − 30 min),
#   STOP (notebook interrupt / Block 08S), SIGTERM, or the notebook dying.
#   Next session: Blocks 01–07, then this block — runs resume bit-identically;
#   finished teachers / soft targets / runs are never redone.
#
# MEMORY
#   A pre-flight (Block 06) picks the micro-batch with 1.5 GB headroom; a CUDA
#   OOM inside a step is recovered by redoing that micro-batch in halves;
#   caches are released after every epoch, evaluation and scoring.
# ══════════════════════════════════════════════════════════════════════════════

import csv
import signal
import numpy as np

header("BLOCK 08 — Two-GPU Experiment Scheduler (Phase A v7, no augmentation)  [v7]")
V = BlockVerdict(8, "Training")
TEST_MODE = os.environ.get("EMOTIC_TEST_MODE") == "1"

# ──────────────────────────────────────────────────────────────────────────────
# 8.1  PLAN
# ──────────────────────────────────────────────────────────────────────────────
print("8.1  PLAN\n")
_PERSON = dict(use_roi=True, use_geo=True, use_crop=True)
CONFIGS = {
    "C0_base":   dict(),
    "C1_person": dict(_PERSON),
    "C4_siglip": dict(_PERSON, use_sig=True),
    "KD_siglip": dict(_PERSON, use_sig=True, role="student"),
}
KD_BASE, TEACHER_CFG = "C4_siglip", "T_siglip"          # fixed before any run
CONFIG_DESC = {"C0_base": "image + description (paper architecture)",
               "C1_person": "+ EVA person crop, ROI tokens, geometry",
               "C4_siglip": "+ SigLIP-2 person crop (384 px)",
               "KD_siglip": "C4 + cross-fitted self-distillation (proposed)",
               TEACHER_CFG: "fold teacher (C4 on 4/5 of train)",
               "SOUP": "weight average of the configuration's seeds"}
SEEDS = [1, 2] if TEST_MODE else [1, 2, 3, 4, 5]
KD_K, KD_R = int(CFG.kd_folds), int(CFG.kd_repeats)
TEACHERS = [f"{TEACHER_CFG}_r{r}f{k}" for r in range(1, KD_R + 1) for k in range(1, KD_K + 1)]
KD_FILE = CFG.runs_dir / f"kd_targets_{KD_BASE}.npy"
KD_INFO = CFG.runs_dir / f"kd_targets_{KD_BASE}.json"


def _pairs(a, b):
    return [x for s in SEEDS for x in (f"{a}_s{s}", f"{b}_s{s}")]


PLAN = (TEACHERS + _pairs("KD_siglip", "C4_siglip") + ["SOUP_C4_siglip", "SOUP_KD_siglip"]
        + _pairs("C1_person", "C0_base") + ["SOUP_C1_person", "SOUP_C0_base"])
RUN_JOBS = [n for n in PLAN if not n.startswith("SOUP_")]
TRACK_CCIM = CFG.ccim_test_map / CFG.val_to_test
TRACK_XENOS = CFG.xenos_test_map / CFG.val_to_test
EPOCH_LIMIT = (int(os.environ.get("EMOTIC_TEST_MAX_EPOCHS", "3")) if TEST_MODE
               else int(CFG.train["max_epochs"]))
EXTENSIBLE_KEYS = ("min_epochs", "patience", "min_delta", "log_every", "floor_check_epoch",
                   "floor_min_lift", "max_oom_per_epoch", "num_workers", "eval_batch_size")

if "FEAT_DIR" not in globals():
    FEAT_DIR = Path(core.read_json(CFG.cache_dir / "feat_dir.json", {})["feat_dir"])
if "MICRO_BATCH" not in globals() or MICRO_BATCH is None:
    _pf = core.read_json(CFG.cache_dir / "preflight.json", {}) or {}
    MICRO_BATCH, GRAD_ACCUM = _pf.get("micro_batch"), _pf.get("grad_accum")
    EPOCH_MIN_UPPER = _pf.get("epoch_min_upper")
ORDER_HASH_NOW = json.loads((CFG.array_dir / "meta.json").read_text())["order_hash"]
ARCHIVE_DIR = FEAT_DIR.parent / "emotic_weights_archive_v7"

V.check("feature cache of this session is complete (3 views)", (FEAT_DIR / "extract_done.json").exists()
        and core.read_json(FEAT_DIR / "cache_key.json", {}).get("order_hash") == ORDER_HASH_NOW
        and all((FEAT_DIR / f"{s}_{v}.npy").exists() for s in ("train", "val", "test")
                for v in ("feats", "crop", "sig")), str(FEAT_DIR), "extract_done + matching order")
V.check("cross-fitting folds present (Block 05)", (CFG.array_dir / "train_folds.npy").exists(),
        "train_folds.npy", "present")
V.check("micro-batch chosen by the GPU pre-flight", MICRO_BATCH in (16, 32) or TEST_MODE,
        f"{MICRO_BATCH} x {GRAD_ACCUM}", "32 x 2 or 16 x 4")
if TEST_MODE:
    MICRO_BATCH, GRAD_ACCUM = 8, 2
SLOTS = list(range(min(N_GPUS, 2))) if N_GPUS > 0 else ["cpu0", "cpu1"]
V.note("devices", ", ".join(f"GPU {s}" if isinstance(s, int) else s for s in SLOTS))
V.note("training deadline", time.strftime("%H:%M:%S", time.localtime(GUARD.training_deadline()))
       + f"  ({GUARD.training_remaining_s() / 3600:.2f} h of training time left)")
V.note("recipe", f"{CFG.train['lr_schedule']} LR over {EPOCH_LIMIT} epochs, early stop after "
                 f"{CFG.train['patience']} flat epochs (min {CFG.train['min_epochs']}), EMA {CFG.train['ema_decay']}, "
                 f"wd {CFG.train['weight_decay']}, length-bucketed batches, shared init")
for c in CONFIGS:
    V.note(f"  {c:<10s}", CONFIG_DESC[c])
V.note(f"  {TEACHER_CFG:<10s}", f"{len(TEACHERS)} runs = {KD_K} image-grouped folds × {KD_R} repeats")
V.note("jobs", f"{len(RUN_JOBS)} training runs ({len(CONFIGS)} configurations × {len(SEEDS)} seeds + "
               f"{len(TEACHERS)} teachers) + 4 soups")


def parse(name):
    """(kind, config, seed, repeat, fold) of a job name."""
    if name.startswith("SOUP_"):
        return "soup", name[5:], None, None, None
    if name.startswith(TEACHER_CFG + "_r"):
        r, k = name[len(TEACHER_CFG) + 2:].split("f")
        return "teacher", TEACHER_CFG, None, int(r), int(k)
    c, s = name.rsplit("_s", 1)
    return "run", c, int(s), None, None


def rdir(name):
    return CFG.runs_dir / name


def run_state(name):
    kind = parse(name)[0]
    d = rdir(name)
    if kind == "soup":
        sj = core.read_json(d / "soup.json", {}) or {}
        return {"done": "done", "failed": "failed"}.get(sj.get("status"), "new")
    res = core.read_json(d / "result.json", {}) or {}
    if res.get("status") == "done":
        if kind == "teacher" and not (d / "oof.npz").exists():
            return "unscored"
        if res.get("extendable") and (d / "last.pt").exists() and EPOCH_LIMIT > int(res.get("epochs_run", 0)):
            return "extendable"
        return "done"
    if (d / "finished.json").exists():
        return "finishing"
    if res.get("status") == "failed":
        return "failed"
    if (d / "last.pt").exists():
        return "paused"
    return "new"


def best_val(name):
    res = core.read_json(rdir(name) / "result.json", {}) or {}
    if isinstance(res.get("best"), dict):
        return res["best"]["val_tl_mAP"]
    b = (core.read_json(rdir(name) / "status.json", {}) or {}).get("best")
    return b if isinstance(b, (int, float)) else None


def weights_of(name):
    p = rdir(name) / "best.pt"
    if p.exists():
        return p
    a = ARCHIVE_DIR / f"{name}_best_fp16.pt"
    return a if a.exists() else None


def ready(name):
    kind, c, _, _, _ = parse(name)
    if kind == "soup":
        runs = [f"{c}_s{s}" for s in SEEDS]
        return all(run_state(n) == "done" for n in runs) and all(weights_of(n) for n in runs)
    if c == "KD_siglip":
        return KD_FILE.exists()
    return True


def epoch_minutes():
    vals = []
    for n in RUN_JOBS:
        h = rdir(n) / "history.csv"
        if h.exists():
            try:
                vals += [float(r["train_min"]) + float(r["eval_min"]) for r in csv.DictReader(open(h))]
            except Exception:
                pass
    if vals:
        return float(np.median(vals))
    return float(EPOCH_MIN_UPPER) * 0.8 if EPOCH_MIN_UPPER else 10.0


LAUNCHES = {}          # per job, in this execution of the block
MAX_LAUNCHES = 3       # a job that dies 3 times without saving is left alone
RETRIED = set()        # a crashed run gets ONE automatic retry per execution
# minutes of training time that must be left to start a job of each kind
MIN_LEFT = (dict(fresh=30, resume=12, extend=20, rescore=5, retry=45, soup=10) if not TEST_MODE
            else dict(fresh=1, resume=1, extend=1, rescore=1, retry=1, soup=1))


def next_job(active):
    """Pure: decides what to start next."""
    tr = GUARD.training_remaining_s() / 60.0
    order = [n for n in PLAN if n not in active and LAUNCHES.get(n, 0) < MAX_LAUNCHES]
    for n in order:                                    # 1) resume / extend / finalise / re-score
        st = run_state(n)
        if st in ("paused", "finishing") and tr > MIN_LEFT["resume"]:
            return n, "resume"
        if st == "extendable" and tr > MIN_LEFT["extend"]:
            return n, "extend"
        if st == "unscored" and tr > MIN_LEFT["rescore"] and (rdir(n) / "best.pt").exists():
            return n, "rescore"
    for n in order:                                    # 2) one retry of a crashed job
        if run_state(n) != "failed" or n in RETRIED or tr < MIN_LEFT["retry"]:
            continue
        if parse(n)[0] == "soup":
            return n, "retry"
        msg = str((core.read_json(rdir(n) / "result.json", {}) or {}).get("message", ""))
        if "floor check" not in msg and "signature mismatch" not in msg:
            return n, "retry"
    for n in order:                                    # 3) next job of the plan that is ready
        if run_state(n) == "new" and ready(n) and tr > MIN_LEFT["soup" if parse(n)[0] == "soup" else "fresh"]:
            return n, "fresh"
    return None


def make_room(need_gb):
    """Move fp16 weights of finished runs to scratch until need_gb is free (soup
    ingredients last: their weights are needed for the soups)."""
    if disk_free_gb(CFG.work_dir) >= need_gb:
        return True
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    souped = {n for n in RUN_JOBS if parse(n)[0] == "run"
              and run_state(f"SOUP_{parse(n)[1]}") == "done"}
    for n in sorted(RUN_JOBS, key=lambda x: (x not in souped, PLAN.index(x))):
        if disk_free_gb(CFG.work_dir) >= need_gb:
            break
        b = rdir(n) / "best.pt"
        if parse(n)[0] == "run" and run_state(n) == "done" and b.exists():
            shutil.move(str(b), str(ARCHIVE_DIR / f"{n}_best_fp16.pt"))
            print(f"{_Y}  disk: moved {n}/best.pt to {ARCHIVE_DIR} (scores stay in {rdir(n)}){_0}")
    return disk_free_gb(CFG.work_dir) >= need_gb


def assemble_kd():
    """All teachers scored → average their out-of-fold probabilities into KD_FILE (once)."""
    if KD_FILE.exists() or not all(run_state(n) == "done" for n in TEACHERS):
        return
    try:
        st = core.assemble_kd_targets([rdir(n) for n in TEACHERS], len(df_train), KD_FILE,
                                      train_labels=TRAIN_LABELS)
        st.update(teacher_val_tl={n: best_val(n) for n in TEACHERS}, written=time.strftime("%Y-%m-%d %H:%M:%S"),
                  alpha=CFG.kd_alpha)
        core.write_json(KD_INFO, st)
        print(f"\n{_G}■ soft targets assembled from {len(TEACHERS)} teachers ({st['repeats']} predictions per "
              f"train person): mean {st['mean']:.3f} on labelled positives {st['mean_on_positives']:.3f}, "
              f"on negatives {st['mean_on_negatives']:.3f}; out-of-fold mAP vs the single-annotator train "
              f"labels {st['oof_map_vs_train_labels']:.2f} → students can start{_0}", flush=True)
    except Exception as e:
        print(f"{_R}  soft targets could not be assembled: {type(e).__name__}: {e}{_0}")


# ──────────────────────────────────────────────────────────────────────────────
# 8.2  WORKERS
# ──────────────────────────────────────────────────────────────────────────────
class Slot:
    def __init__(self, name, dev, proc, logf, logp, offset, mode):
        self.name, self.dev, self.proc, self.logf, self.logp = name, dev, proc, logf, logp
        self.offset, self.mode, self.t0 = offset, mode, time.time()


def _paths(spec):
    spec.update(array_dir=str(CFG.array_dir), feat_dir=str(FEAT_DIR),
                hf_config_dir=str(FEAT_DIR / "hf_config"), init_path=str(FEAT_DIR / "qformer_init.pt"))
    return spec


def build_spec(name, mode):
    kind, c, seed, r, k = parse(name)
    d = rdir(name)
    if kind == "soup":
        runs = [f"{c}_s{s}" for s in SEEDS]
        spec = _paths({"name": name, "out_dir": str(d), "ingredients": [str(rdir(n)) for n in runs],
                       "archive": {n: str(weights_of(n)) for n in runs},
                       "num_workers": 0 if TEST_MODE else 2,
                       "eval_batch_size": 32 if TEST_MODE else int(CFG.train["eval_batch_size"])})
        return spec, "soup", d / "soup.log"
    old = core.read_json(d / "spec_resolved.json") if mode != "fresh" else None
    if old:
        if old.get("order_hash") not in (None, ORDER_HASH_NOW):
            raise RuntimeError(f"{name}: data order changed since this run started — cannot resume")
        spec = dict(old)                        # the run keeps its own trajectory settings …
        if not TEST_MODE:                       # … but takes the current stop / logging settings
            spec.update({k2: CFG.train[k2] for k2 in EXTENSIBLE_KEYS if k2 in CFG.train})
        if spec.get("lr_schedule") == "plateau":
            spec["max_epochs"] = EPOCH_LIMIT    # only plateau runs may change their length
    else:
        spec = dict(CFG.train)
        if kind == "teacher":
            spec.update(CONFIGS[KD_BASE])
            spec.update(name=name, config_name=TEACHER_CFG, seed=1000 + 10 * r + k, role="teacher",
                        fold_file=str(CFG.array_dir / "train_folds.npy"), fold_repeat=r - 1, fold=k - 1)
        else:
            spec.update(CONFIGS[c])
            spec.update(name=name, config_name=c, seed=seed)
            if c == "KD_siglip":
                spec.update(kd_path=str(KD_FILE), kd_alpha=float(CFG.kd_alpha))
        spec.update(phase="A_no_augmentation_v7", batch_size=MICRO_BATCH, grad_accum=GRAD_ACCUM,
                    order_hash=ORDER_HASH_NOW, max_epochs=EPOCH_LIMIT)
        if TEST_MODE:
            spec.update(min_epochs=1, patience=2, floor_min_lift=0.0, num_workers=0,
                        log_every=5, eval_batch_size=32, lr=1e-3)
    spec.update(run_dir=str(d), control_dir=str(CFG.control_dir), deadline=GUARD.training_deadline(),
                parent_pid=os.getpid(), test_inject_nonfinite_every=0, test_stop_after_steps=0,
                test_inject_oom_every=0)
    _paths(spec)
    if mode == "rescore":
        return spec, "evaluate", d / "evaluate.log"
    return spec, "train", d / "worker.log"


def launch(name, dev, mode):
    d = rdir(name)
    d.mkdir(parents=True, exist_ok=True)
    LAUNCHES[name] = LAUNCHES.get(name, 0) + 1
    if mode == "retry":
        RETRIED.add(name)
        for f in ("result.json", "soup.json"):
            if (d / f).exists():
                os.replace(d / f, d / f"{f[:-5]}_failed_{int(time.time())}.json")
    spec, cmd, logp = build_spec(name, mode)
    core.write_json(d / "spec.json", spec)
    (CFG.control_dir / f"STOP_{name}").unlink(missing_ok=True)
    offset = logp.stat().st_size if logp.exists() else 0
    logf = open(logp, "a")
    logf.write(f"\n##### {mode} {time.strftime('%Y-%m-%d %H:%M:%S')} on {dev} #####\n")
    logf.flush()
    env = dict(os.environ, PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               CUDA_VISIBLE_DEVICES=str(dev) if isinstance(dev, int) else "")
    proc = subprocess.Popen([sys.executable, str(CFG.core_path), cmd, "--spec", str(d / "spec.json")],
                            stdout=logf, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    tag = f"GPU {dev}" if isinstance(dev, int) else dev
    kind, c = parse(name)[:2]
    desc = CONFIG_DESC["SOUP"] if kind == "soup" else CONFIG_DESC.get(c, "")
    bsz = "" if kind == "soup" else f", micro-batch {spec.get('batch_size')} x {spec.get('grad_accum')}"
    print(f"\n{_B}▶ {mode.upper():<7s} {name}  on {tag}  (pid {proc.pid}{bsz})  — {desc}{_0}", flush=True)
    return Slot(name, dev, proc, logf, logp, offset, mode)


def stream(slot):
    try:
        with open(slot.logp, "rb") as f:
            f.seek(slot.offset)
            data = f.read()
    except FileNotFoundError:
        return
    if not data:
        return
    cut = data.rfind(b"\n")
    if cut < 0:
        return
    slot.offset += cut + 1
    for line in data[:cut].decode("utf-8", "replace").splitlines():
        if line.startswith("#####") or not line.strip():
            continue
        print(f"{_D}{slot.name[:16]:<16s}|{_0} {line}", flush=True)


def gpu_line():
    if N_GPUS == 0:
        return ""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout
        return "  ".join(f"GPU{r[0]} {r[1]}% {float(r[2]) / 1024:.1f}/{float(r[3]) / 1024:.0f}GB"
                         for r in (x.split(", ") for x in out.strip().splitlines()))
    except Exception:
        return ""


def colour_val(v, width=7):
    if v is None:
        return f"{'-':>{width}s}"
    c = _G if v >= TRACK_CCIM else (_Y if v >= TRACK_XENOS else _R)
    return f"{c}{v:{width}.2f}{_0}"


def _cell(n, active):
    st = "running" if n in active else run_state(n)
    v = best_val(n)
    if st == "done" and v is not None:
        return " " + colour_val(v, 8), v
    if st == "running":
        ep = (core.read_json(rdir(n) / "progress.json", {}) or {}).get("epoch", 0)
        return f"{f'▶ep{ep}/{EPOCH_LIMIT}':>9s}", None
    if st in ("paused", "finishing", "extendable", "unscored"):
        return f"{'‖' + (f'{v:.2f}' if v else ''):>9s}", None
    if st == "failed":
        return f"{'✗':>9s}", None
    return f"{'·':>9s}", None


def status_table(slots):
    active = {s.name for s in slots.values() if s}
    print(f"\n  {'─' * 78}\n  STATUS  {time.strftime('%H:%M:%S')}  |  {GUARD.footer()}  |  "
          f"training left {GUARD.training_remaining_s() / 3600:.2f} h")
    print("  best test-like val mAP per run  (▶ running  ‖ paused  ✗ failed  · waiting)")
    print(f"  {'config':<11s}" + "".join(f"{f's{s}':>9s}" for s in SEEDS) + f"{'mean (n)':>14s}   soup (uniform/greedy)")
    for c in ("C4_siglip", "KD_siglip", "C1_person", "C0_base"):
        cells, vals = zip(*[_cell(f"{c}_s{s}", active) for s in SEEDS])
        vals = [v for v in vals if v is not None]
        mean = f"{np.mean(vals):6.2f} ({len(vals)})" if vals else "-"
        sj = core.read_json(rdir(f"SOUP_{c}") / "soup.json", {}) or {}
        soup = (f"{sj['uniform']['val_tl_mAP']:.2f} / {sj['greedy']['val_tl_mAP']:.2f} "
                f"({len(sj['greedy']['members'])} of {len(sj['ingredients'])})" if sj.get("status") == "done"
                else ("▶" if f"SOUP_{c}" in active else ("✗" if sj.get("status") == "failed" else "·")))
        print(f"  {c:<11s}" + "".join(cells) + f"{mean:>14s}   {soup}")
    for r in range(1, KD_R + 1):
        cells = [_cell(f"{TEACHER_CFG}_r{r}f{k}", active)[0] for k in range(1, KD_K + 1)]
        print(f"  {f'teach r{r}':<11s}" + "".join(cells) + "   (folds 1–5; val-TL on 4/5 of train)")
    kd = core.read_json(KD_INFO, {}) or {}
    print(f"  soft targets: " + (f"ready (mean on negatives {kd.get('mean_on_negatives', 0):.3f})" if KD_FILE.exists()
                                 else f"waiting for {sum(run_state(n) != 'done' for n in TEACHERS)} teacher(s)"))
    em = epoch_minutes()
    left = [n for n in PLAN if run_state(n) != "done"]
    print(f"  epoch ≈ {em:.1f} min | {len(PLAN) - len(left)}/{len(PLAN)} jobs done | ≈ "
          f"{GUARD.training_remaining_s() / 60 / em * len(SLOTS):.0f} epochs of GPU time left this session")
    g = gpu_line()
    if g:
        print(f"  {g}")
    print(f"  colours: green ≥ {TRACK_CCIM:.1f} (CCIM track)  yellow ≥ {TRACK_XENOS:.1f} (Xenos track) — "
          f"val monitor only (test ≈ {CFG.val_to_test} × val-TL in v6) | /kaggle/working free "
          f"{disk_free_gb(CFG.work_dir):.1f} GB\n  {'─' * 78}\n", flush=True)


def worker_pids():
    pids = []
    try:
        import psutil
        for p in psutil.process_iter(["pid", "cmdline"]):
            cl = p.info.get("cmdline") or []
            if any(str(CFG.core_path) == a for a in cl) and any(c in cl for c in ("train", "soup", "evaluate")):
                pids.append(p.info["pid"])
    except Exception:
        pass
    return pids


def stop_all(slots, reason):
    alive = [s for s in slots.values() if s and s.proc.poll() is None]
    (CFG.control_dir / "STOP").write_text(reason)
    print(f"\n{_Y}■ STOP requested ({reason}). {len(alive)} worker(s) are saving an exact "
          f"resume point — usually < 1 min. Do not interrupt again.{_0}", flush=True)
    t0 = time.time()
    while any(s.proc.poll() is None for s in alive) and time.time() - t0 < 300:
        for s in alive:
            stream(s)
        time.sleep(3)
    for s in alive:
        if s.proc.poll() is None:
            s.proc.send_signal(signal.SIGTERM)
    t1 = time.time()
    while any(s.proc.poll() is None for s in alive) and time.time() - t1 < 90:
        time.sleep(3)
    for s in alive:
        if s.proc.poll() is None:
            s.proc.kill()
    for s in slots.values():
        if s:
            stream(s)
            s.logf.close()
    (CFG.control_dir / "STOP").unlink(missing_ok=True)
    for s in alive:
        st = run_state(s.name)
        c = _G if st in ("paused", "done", "finishing", "new") else _R
        print(f"  {c}{s.name}: {st}{_0}")
    for k in list(slots):
        slots[k] = None


# ──────────────────────────────────────────────────────────────────────────────
# 8.3  RUN
# ──────────────────────────────────────────────────────────────────────────────
print("\n8.3  RUN\n")
for _f in CFG.control_dir.glob("STOP*"):
    V.note("removed stale stop file", _f.name)
    _f.unlink()
_orphans = worker_pids()
if _orphans:
    V.note("workers from an earlier cell still alive", f"{_orphans} — asking them to save and exit")
    (CFG.control_dir / "STOP").write_text("orphan cleanup")
    _t = time.time()
    while worker_pids() and time.time() - _t < 300:
        time.sleep(5)
    (CFG.control_dir / "STOP").unlink(missing_ok=True)
V.check("no stray workers", not worker_pids(), worker_pids() or "none", "none")
V.check("/kaggle/working has room for two runs", make_room(8.0),
        f"{disk_free_gb(CFG.work_dir):.1f} GB", ">= 8 GB")
if KD_FILE.exists():
    _kdi = core.read_json(KD_INFO, {}) or {}
    V.check("soft targets from an earlier session are intact",
            core.array_sha(np.load(KD_FILE)) == _kdi.get("sha"), _kdi.get("sha"), "same content hash")
free_memory()

slots = {d: None for d in SLOTS}
EXITS = []
_t_table, _t_run0, _blocked = 0.0, time.time(), False
try:
    while True:
        for dev, s in slots.items():                       # reap
            if s is not None and s.proc.poll() is not None:
                stream(s)
                s.logf.close()
                rc = s.proc.returncode
                st = run_state(s.name)
                EXITS.append((s.name, rc, st))
                c = _G if st in ("done", "paused", "extendable") else _R
                bv = best_val(s.name) if parse(s.name)[0] != "soup" else None
                print(f"\n{c}■ {s.name} exited ({st}, rc {rc}) after {(time.time() - s.t0) / 60:.1f} min"
                      + (f" — best val-TL {bv:.2f}" if bv is not None else "") + f"{_0}", flush=True)
                if st == "failed":
                    print("\n".join(Path(s.logp).read_text().splitlines()[-30:]))
                slots[dev] = None
        assemble_kd()
        _blocked = False
        for dev in [d for d, s in slots.items() if s is None]:   # launch
            job = next_job({s.name for s in slots.values() if s})
            if job is None:
                break
            _need = 1.0 if (parse(job[0])[0] == "soup" or job[1] == "rescore") else \
                (4.5 if job[1] in ("resume", "extend") else 8.0)
            if not make_room(_need):
                print(f"{_R}  not launching {job[0]}: /kaggle/working has only "
                      f"{disk_free_gb(CFG.work_dir):.1f} GB free{_0}")
                _blocked = True
                break
            slots[dev] = launch(job[0], dev, job[1])
        for s in slots.values():                           # stream
            if s:
                stream(s)
        if time.time() - _t_table > 900:
            status_table(slots)
            _t_table = time.time()
        if all(s is None for s in slots.values()) and (next_job(set()) is None or _blocked):
            break
        time.sleep(5)
except KeyboardInterrupt:
    stop_all(slots, "notebook interrupt")

# ──────────────────────────────────────────────────────────────────────────────
# 8.4  SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print("\n8.4  SUMMARY\n")
status_table(slots)
free_memory()
_states = {n: run_state(n) for n in PLAN}
_failed = [n for n, s in _states.items() if s == "failed"]
_paused = [n for n, s in _states.items() if s in ("paused", "finishing", "extendable", "unscored")]
_done = [n for n, s in _states.items() if s == "done"]
V.note("time in this block", f"{(time.time() - _t_run0) / 3600:.2f} h")
V.check("no job failed", not _failed, _failed or "none", "none")
V.check("disk never blocked a launch", not _blocked, "blocked" if _blocked else "ok", "ok")
V.note("jobs done", f"{len(_done)}/{len(PLAN)}")
V.check("teachers finished and soft targets assembled", KD_FILE.exists(),
        f"{sum(_states[n] == 'done' for n in TEACHERS)}/{len(TEACHERS)} teachers", "all + soft targets", fatal=False)
for c in ("C4_siglip", "KD_siglip", "C1_person", "C0_base"):
    _v = [best_val(f"{c}_s{s}") for s in SEEDS if _states[f"{c}_s{s}"] == "done"]
    if _v:
        V.note(f"  {c}", f"val-TL {np.mean(_v):.2f} ± {np.std(_v):.2f} over {len(_v)} seed(s)")
_oom = 0
for n in _done + _paused:
    h = rdir(n) / "history.csv"
    if h.exists():
        _oom += sum(int(r.get("oom") or 0) for r in csv.DictReader(open(h)))
V.check("no CUDA OOM during training", _oom == 0, _oom, 0, fatal=False)
V.check("all planned jobs finished", len(_done) == len(PLAN), f"{len(_done)}/{len(PLAN)}", "all", fatal=False)
if _paused or len(_done) < len(PLAN):
    _msg = f"Not finished: {len(PLAN) - len(_done)} job(s)" + (f", paused: {', '.join(_paused)}" if _paused else "")
    print(f"\n  {_Y}{_msg}. Next session: Blocks 01–07, then this block — everything resumes where it stopped.\n"
          f"  Block 09 can be run now for a PROVISIONAL report (it marks it so); the final report needs all jobs.{_0}")
else:
    print(f"\n  {_G}All {len(PLAN)} jobs are done. Run Block 09.{_0}")
V.check("/kaggle/working still has free space", disk_free_gb(CFG.work_dir) >= 1.0,
        f"{disk_free_gb(CFG.work_dir):.1f} GB", ">= 1 GB")
BLOCK_08_OK = V.finish()

In [ ]:
# Block 09 — Test Evaluation & Final Report (Phase A v7: NO augmentation)
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Blocks 01–07 in this session; Block 08 finished its jobs
#                 (it runs on partial results too, but marks them PROVISIONAL)
#
# INTEGRITY RULES (fixed before any run started)
#   * Final configuration = highest MEAN test-like val mAP over its seeds,
#     among the single-network configurations C0, C1, C4, KD (the distilled KD
#     student is one network; its teachers are disclosed with the result).
#     Decided and saved to selection.json before any test number is computed.
#   * Headline = mean ± std of the final configuration's single models over
#     its seeds (not the best seed). Soups (one network made from N seeds) and
#     seed ensembles (N networks) are SEPARATE, labelled rows. The soup row is
#     the UNIFORM soup (plain average of all N seeds — no selection at all);
#     the greedy soup (seeds added only while test-like val improves) is shown
#     next to it as a diagnostic.
#   * Every configuration is reported (pre-declared ablation) with paired tests:
#     seed-matched paired t-test and a paired bootstrap over test persons.
#     Differences below ~0.3 mAP are called noise unless significant.
#   * Teachers never produced test scores. Run / soup test scores were written
#     once by their worker; every read of them here is appended to
#     test_access_log.jsonl. F1-type thresholds come from test-like val.
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from scipy import stats as sps

header("BLOCK 09 — Test Evaluation & Final Report (Phase A v7, no augmentation)  [v7]")
V = BlockVerdict(9, "Phase A results")
TEST_MODE = os.environ.get("EMOTIC_TEST_MODE") == "1"
RESULTS_DIR = CFG.work_dir / "results" / "phaseA_v7_no_augmentation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_ORDER = ["C0_base", "C1_person", "C4_siglip", "KD_siglip"]
CONFIG_LABEL = {"C0_base": "C0 base (Xenos architecture)", "C1_person": "C1 + EVA person crop/ROI/geometry",
                "C4_siglip": "C4 + SigLIP-2 person crop", "KD_siglip": "KD: C4 + cross-fitted self-distillation"}
SEEDS_PLANNED = [1, 2] if TEST_MODE else [1, 2, 3, 4, 5]
TARGET_MEAN = 39.60          # brief: 5-seed single-model mean ≥ 39.6 (≈ CCIM + 0.5)
if "FEAT_DIR" not in globals():
    FEAT_DIR = Path(core.read_json(CFG.cache_dir / "feat_dir.json", {})["feat_dir"])
KD_INFO_PATH = CFG.runs_dir / "kd_targets_C4_siglip.json"


def _parse(name):
    c, s = name.rsplit("_s", 1)
    return c, int(s)


# ──────────────────────────────────────────────────────────────────────────────
# 9.1  SELECTION — validation only
# ──────────────────────────────────────────────────────────────────────────────
print("9.1  SELECTION (test-like validation only)\n")
RUNS = {}
for _d in sorted(CFG.runs_dir.glob("*_s[0-9]*")):
    _res = core.read_json(_d / "result.json", {}) or {}
    if _d.is_dir() and _res.get("status") == "done" and _parse(_d.name)[0] in CONFIG_ORDER:
        RUNS[_d.name] = _res
V.check("finished runs found", len(RUNS) > 0, len(RUNS), ">= 1")
if len(RUNS) == 0:
    BLOCK_09_OK = V.finish()
    raise RuntimeError("No finished run yet — run Block 08 first.")
_seeds_by_cfg = {c: sorted(_parse(n)[1] for n in RUNS if _parse(n)[0] == c) for c in CONFIG_ORDER}
FULL = [c for c in CONFIG_ORDER if set(SEEDS_PLANNED) <= set(_seeds_by_cfg[c])]
if FULL:
    AVAIL, COMPLETE_SEEDS = FULL, list(SEEDS_PLANNED)
else:
    AVAIL = [c for c in CONFIG_ORDER if _seeds_by_cfg[c]]
    COMPLETE_SEEDS = sorted(set.intersection(*[set(_seeds_by_cfg[c]) for c in AVAIL]))
    if not COMPLETE_SEEDS:              # no seed shared by all: use the configuration with most seeds
        AVAIL = [max(AVAIL, key=lambda c: len(_seeds_by_cfg[c]))]
        COMPLETE_SEEDS = _seeds_by_cfg[AVAIL[0]]
_rows = []
for n, r in sorted(RUNS.items(), key=lambda kv: (CONFIG_ORDER.index(_parse(kv[0])[0]), _parse(kv[0])[1])):
    b = r["best"]
    _rows.append({"run": n, "config": r["config_name"], "seed": r["seed"], "best_epoch": b["epoch"] + 1,
                  "weights": b["variant"], "val_TL_mAP": b["val_tl_mAP"], "val_TL_lift": b["val_tl_lift"],
                  "epochs_run": r["epochs_run"], "stop": r.get("stop_reason", ""), "train_h": r["train_hours"]})
SEL_TABLE = pd.DataFrame(_rows)
print(SEL_TABLE.to_string(index=False, float_format="{:.3f}".format))
_missing_cfg = [c for c in CONFIG_ORDER if c not in AVAIL]
V.note("configurations with every planned seed", FULL or "none yet")
_prov = bool(_missing_cfg) or len(COMPLETE_SEEDS) < len(SEEDS_PLANNED)
V.check("all 4 configurations × planned seeds finished", not _prov,
        f"{len(AVAIL)} configs × {len(COMPLETE_SEEDS)} seeds", f"4 × {len(SEEDS_PLANNED)}", fatal=False)
if _prov:
    print(f"  {_Y}PROVISIONAL report: missing {', '.join(_missing_cfg) or 'seeds'} — finish Block 08 first "
          f"for the final one.{_0}")

# teachers (their val-TL is on 4/5 of the train data; they have no test scores)
_teach = []
for _d in sorted(CFG.runs_dir.glob("T_siglip_r*f*")):
    _r = core.read_json(_d / "result.json", {}) or {}
    _e = core.read_json(_d / "eval.json", {}) or {}
    if _r.get("status") == "done":
        _teach.append({"teacher": _d.name, "train_persons": _r.get("n_train"), "val_TL_mAP": _r["best"]["val_tl_mAP"],
                       "best_epoch": _r["best"]["epoch"] + 1, "held_out": _e.get("n_oof"),
                       "oof_mAP_vs_train_labels": _e.get("oof_map_vs_train_labels"),
                       "test_scores_written": (_d / "scores.npz").exists()})
TEACHERS_TABLE = pd.DataFrame(_teach)
KD_INFO = core.read_json(KD_INFO_PATH, {}) or {}
if len(TEACHERS_TABLE):
    print(f"\n  cross-fitting teachers ({len(TEACHERS_TABLE)}):")
    print("  " + TEACHERS_TABLE.to_string(index=False, float_format="{:.3f}".format).replace("\n", "\n  "))
    V.check("no teacher ever wrote test scores", not TEACHERS_TABLE["test_scores_written"].any(),
            int(TEACHERS_TABLE["test_scores_written"].sum()), 0)
if KD_INFO:
    V.note("soft targets", f"{KD_INFO.get('repeats')} out-of-fold predictions per train person, a = "
                           f"{KD_INFO.get('alpha')}; mean on labelled positives {KD_INFO.get('mean_on_positives', 0):.3f}, "
                           f"on negatives {KD_INFO.get('mean_on_negatives', 0):.3f}")

VAL_MEAN = {c: float(np.mean([RUNS[f"{c}_s{s}"]["best"]["val_tl_mAP"] for s in COMPLETE_SEEDS])) for c in AVAIL}
FINAL_CONFIG = max(VAL_MEAN, key=VAL_MEAN.get)
FINAL_RUNS = [f"{FINAL_CONFIG}_s{s}" for s in COMPLETE_SEEDS]
_vals_final = {n: RUNS[n]["best"]["val_tl_mAP"] for n in FINAL_RUNS}
DISPLAY_RUN = sorted(_vals_final, key=_vals_final.get)[len(_vals_final) // 2]     # median val seed
SOUPS = {}
for c in AVAIL:
    sj = core.read_json(CFG.runs_dir / f"SOUP_{c}" / "soup.json", {}) or {}
    if sj.get("status") == "done" and sorted(sj["ingredients"]) == sorted(f"{c}_s{s}" for s in COMPLETE_SEEDS):
        SOUPS[c] = sj
print()
for c in AVAIL:
    mark = f"  {_G}← final (highest mean val-TL){_0}" if c == FINAL_CONFIG else ""
    soup = (f" | soup val-TL {SOUPS[c]['uniform']['val_tl_mAP']:.2f} (greedy "
            f"{SOUPS[c]['greedy']['val_tl_mAP']:.2f} with {len(SOUPS[c]['greedy']['members'])} of "
            f"{len(SOUPS[c]['ingredients'])})" if c in SOUPS else " | soup not available")
    print(f"  {c:<10s} mean val-TL {VAL_MEAN[c]:6.2f} over seeds {COMPLETE_SEEDS}{soup}{mark}")
print(f"\n  headline = mean ± std of {FINAL_CONFIG} over {len(FINAL_RUNS)} seeds; its uniform soup (one network) "
      f"and its {len(FINAL_RUNS)}-seed ensemble are separate rows")
SELECTION = {"final_config": FINAL_CONFIG, "final_runs": FINAL_RUNS, "display_run": DISPLAY_RUN,
             "complete_seeds": COMPLETE_SEEDS, "configs": AVAIL, "provisional": _prov, "val_tl_mean": VAL_MEAN,
             "soup_rule": "soup row = uniform soup of all seeds (no selection); greedy soup (Wortsman et al. "
                          "2022, test-like val) shown alongside as a diagnostic",
             "soups_val_tl": {c: {"uniform": s["uniform"]["val_tl_mAP"], "greedy": s["greedy"]["val_tl_mAP"],
                                  "greedy_members": s["greedy"]["members"]} for c, s in SOUPS.items()},
             "decided_at": time.strftime("%Y-%m-%d %H:%M:%S"),
             "rule": "final config = argmax mean test-like val mAP over its seeds among single-network "
                     "configurations; headline = its seed mean; test not read before this file was written"}
core.write_json(RESULTS_DIR / "selection.json", SELECTION)
V.check("selection saved before any test metric", (RESULTS_DIR / "selection.json").exists(), "saved", "saved")

# ──────────────────────────────────────────────────────────────────────────────
# 9.2  SCORES
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.2  SCORES\n")
_USE = [f"{c}_s{s}" for c in AVAIL for s in COMPLETE_SEEDS]


def _needs_scoring(n):
    ev = core.read_json(CFG.runs_dir / n / "eval.json", {}) or {}
    return not (CFG.runs_dir / n / "scores.npz").exists() or "error" in ev or not RUNS[n].get("scored", False)


_todo = [n for n in _USE if _needs_scoring(n)]
V.note("runs scored by their worker", f"{len(_USE) - len(_todo)}/{len(_USE)}")
if _todo:
    V.note("scoring now", ", ".join(_todo))
    _arch = FEAT_DIR.parent / "emotic_weights_archive_v7"
    _devs = list(range(min(N_GPUS, 2))) or ["cpu"]
    _queue, _live = list(_todo), {}
    while _queue or _live:
        for dv in _devs:
            if dv not in _live and _queue:
                n = _queue.pop(0)
                d = CFG.runs_dir / n
                if not (d / "best.pt").exists() and (_arch / f"{n}_best_fp16.pt").exists():
                    shutil.copy(_arch / f"{n}_best_fp16.pt", d / "best.pt")
                spec = core.read_json(d / "spec_resolved.json", {})
                spec.update(run_dir=str(d), array_dir=str(CFG.array_dir), feat_dir=str(FEAT_DIR),
                            hf_config_dir=str(FEAT_DIR / "hf_config"),
                            init_path=str(FEAT_DIR / "qformer_init.pt"), num_workers=2)
                core.write_json(d / "eval_spec.json", spec)
                env = dict(os.environ, PYTHONUNBUFFERED="1",
                           CUDA_VISIBLE_DEVICES=str(dv) if isinstance(dv, int) else "")
                _lf = open(d / "evaluate.log", "w")
                _live[dv] = (n, subprocess.Popen([sys.executable, str(CFG.core_path), "evaluate", "--spec",
                                                  str(d / "eval_spec.json")], env=env, stdout=_lf,
                                                 stderr=subprocess.STDOUT), _lf)
        time.sleep(2)
        for dv in list(_live):
            n, p, _lf = _live[dv]
            if p.poll() is not None:
                _lf.close()
                print("  " + "\n  ".join((CFG.runs_dir / n / "evaluate.log").read_text().strip().splitlines()[-2:]))
                del _live[dv]

SCORES, EVALS = {}, {}
for n in _USE:
    d = CFG.runs_dir / n
    ev = core.read_json(d / "eval.json", {}) or {}
    if not (d / "scores.npz").exists() or "error" in ev:
        V.check(f"[{n}] scores available", False, ev.get("error", "missing"), "scores.npz")
        continue
    z = np.load(d / "scores.npz")
    SCORES[n] = {k: z[k] for k in z.files}
    EVALS[n] = ev
SOUP_SCORES = {}
for c in SOUPS:
    z = np.load(CFG.runs_dir / f"SOUP_{c}" / "soup_scores.npz")
    SOUP_SCORES[c] = {k: z[k] for k in z.files}
_ok_order = len(SCORES) > 0 and all(np.array_equal(s["test_labels"], TEST_LABELS)
                                    and np.array_equal(s["val_labels_tl"], Y_TESTLIKE_VAL)
                                    for s in list(SCORES.values()) + list(SOUP_SCORES.values()))
V.check("scores align with this session's test and val labels (same order)", _ok_order,
        "aligned" if _ok_order else "MISALIGNED", "aligned")
_drift = {n: abs(e["val_tl_mAP_recomputed"] - e["val_tl_mAP_at_selection"]) for n, e in EVALS.items()}
V.check("re-scored val-TL matches the value used for selection",
        all(_drift[n] < core.SCORE_TOLERANCE[EVALS[n].get("weights_dtype", "float32")] for n in _drift),
        f"max |Δ| {max(_drift.values()) if _drift else 0:.3f}", "within tolerance")
if SOUPS:
    _sd = {c: abs(s["first_ingredient_recomputed"] - s["own_val_tl"][s["greedy"]["steps"][0]["add"]])
           for c, s in SOUPS.items()}
    V.check("soups: first ingredient reproduces its own val-TL", all(v < 0.30 for v in _sd.values()),
            f"max |Δ| {max(_sd.values()):.3f}", "< 0.30 (fp16 weights)")
_missing = [n for n in _USE if n not in SCORES]
V.check("every run of the reported seeds has scores", not _missing, _missing or "all", "all")
if _missing:
    BLOCK_09_OK = V.finish()
    raise RuntimeError(f"Missing scores for {_missing}; see the scoring output above.")

# ──────────────────────────────────────────────────────────────────────────────
# 9.3  TEST METRICS  (the only place test labels meet predictions)
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.3  TEST METRICS\n")
with open(RESULTS_DIR / "test_access_log.jsonl", "a") as _f:
    _f.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "block": "09",
                         "runs_read": len(_USE), "soups_read": sorted(SOUPS), "final_config": FINAL_CONFIG,
                         "complete_seeds": COMPLETE_SEEDS, "provisional": _prov}) + "\n")
_n_access = sum(1 for _ in open(RESULTS_DIR / "test_access_log.jsonl"))
V.note("test-set reads logged", f"{_n_access} execution(s) of this block so far "
                                f"(see {RESULTS_DIR / 'test_access_log.jsonl'})")
FLOOR_TEST = core.prevalence_floor(TEST_LABELS)
METRICS = {}
for n in _USE:
    th = np.asarray(EVALS[n]["thresholds"], np.float32)
    METRICS[n] = {"test": core.evaluate_scores(SCORES[n]["test_scores"], TEST_LABELS, thresholds=th),
                  "val_tl": core.evaluate_scores(SCORES[n]["val_scores"], Y_TESTLIKE_VAL, thresholds=th,
                                                 ranking=False)}
ENS = {}
for c in AVAIL:
    runs = [f"{c}_s{s}" for s in COMPLETE_SEEDS]
    ev_ = np.mean([SCORES[n]["val_scores"] for n in runs], 0)
    et_ = np.mean([SCORES[n]["test_scores"] for n in runs], 0)
    th_ = core.optimal_thresholds(ev_, Y_TESTLIKE_VAL)
    ENS[c] = {"runs": runs, "val_scores": ev_, "test_scores": et_, "thresholds": th_,
              "test": core.evaluate_scores(et_, TEST_LABELS, thresholds=th_),
              "val_tl": core.evaluate_scores(ev_, Y_TESTLIKE_VAL, thresholds=th_, ranking=False)}
SOUP_M = {}
for c, sc in SOUP_SCORES.items():
    SOUP_M[c] = {}
    for kind in ("greedy", "uniform"):
        th_ = np.asarray(SOUPS[c][kind]["thresholds"], np.float32)
        SOUP_M[c][kind] = {"test": core.evaluate_scores(sc[f"{kind}_test"], TEST_LABELS, thresholds=th_),
                           "val_tl_mAP": SOUPS[c][kind]["val_tl_mAP"], "members": SOUPS[c][kind]["members"],
                           "test_scores": sc[f"{kind}_test"], "thresholds": th_}
_chk = core.exact_map(SCORES[FINAL_RUNS[0]]["test_scores"], TEST_LABELS)
V.check("bootstrap AP code reproduces sklearn mAP on the real test scores",
        abs(_chk - METRICS[FINAL_RUNS[0]]["test"]["mAP"]) < 1e-6,
        f"{_chk:.4f} vs {METRICS[FINAL_RUNS[0]]['test']['mAP']:.4f}", "equal")
V.note("test set", f"{len(TEST_LABELS):,} persons, random-ranker floor {FLOOR_TEST:.2f} "
                   f"(Xenos et al. report 7,203 persons; published numbers are compared as reported)")


def seed_maps(c):
    return np.array([METRICS[f"{c}_s{s}"]["test"]["mAP"] for s in COMPLETE_SEEDS])


def tci(x):
    """95% t-interval of a seed mean."""
    if len(x) < 2:
        return (float("nan"), float("nan"))
    h = sps.t.ppf(0.975, len(x) - 1) * np.std(x, ddof=1) / np.sqrt(len(x))
    return float(np.mean(x) - h), float(np.mean(x) + h)


_B_REPS = 200 if TEST_MODE else 1000
BOOT_ENS = {c: core.bootstrap_map(ENS[c]["test_scores"], TEST_LABELS, B=_B_REPS, seed=0) for c in AVAIL}
BOOT_SOUP = {c: core.bootstrap_map(SOUP_M[c]["uniform"]["test_scores"], TEST_LABELS, B=_B_REPS, seed=0)
             for c in SOUP_M}


def _ci(b):
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


# ──────────────────────────────────────────────────────────────────────────────
# 9.4  MAIN RESULT
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.4  MAIN RESULT — EMOTIC test mAP, no augmentation" + ("   [PROVISIONAL]" if _prov else "") + "\n")
MAIN = [("Xenos et al. 2024 (InstructBLIP + LLaVA text)", CFG.xenos_test_map, "published, single model", False),
        ("EmotiCon + CCIM, Yang et al. CVPR 2023 (previous SOTA)", CFG.ccim_test_map, "published, single model", False)]
for c in AVAIL:
    x = seed_maps(c)
    fin = " ◀ final" if c == FINAL_CONFIG else ""
    MAIN.append((f"{c} — single model, mean of {len(x)} seeds{fin}", float(np.mean(x)),
                 f"± {np.std(x, ddof=1) if len(x) > 1 else 0:.2f} std; 95% CI of mean [{tci(x)[0]:.2f}, {tci(x)[1]:.2f}]",
                 True))
    if c in SOUP_M:
        u, g = SOUP_M[c]["uniform"], SOUP_M[c]["greedy"]
        MAIN.append((f"{c} — SOUP: 1 network = average of {len(u['members'])} seeds (separate row)", u["test"]["mAP"],
                     f"bootstrap 95% CI [{_ci(BOOT_SOUP[c])[0]:.2f}, {_ci(BOOT_SOUP[c])[1]:.2f}]; greedy soup "
                     f"({len(g['members'])} of {len(u['members'])} seed{'s' if len(u['members']) > 1 else ''}): "
                     f"{g['test']['mAP']:.2f}", True))
    MAIN.append((f"{c} — {len(x)}-seed ENSEMBLE: {len(x)} networks (separate row)", ENS[c]["test"]["mAP"],
                 f"bootstrap 95% CI [{_ci(BOOT_ENS[c])[0]:.2f}, {_ci(BOOT_ENS[c])[1]:.2f}]", True))
print(f"  {'method':<72s}{'test mAP':>9s}{'lift':>8s}{'vs CCIM':>9s}   note")
print(f"  {'─' * 124}")
for name, m, note, ours in MAIN:
    c = (_G if m > CFG.ccim_test_map else (_Y if m > CFG.xenos_test_map else _R)) if ours else ""
    print(f"  {c}{name:<72s}{m:9.2f}{m / FLOOR_TEST:7.3f}x{m - CFG.ccim_test_map:+9.2f}{_0 if ours else ''}   {note}")
print(f"\n  lift = mAP ÷ random-ranker floor ({FLOOR_TEST:.2f}); published rows use our test floor.")
if "KD_siglip" in AVAIL and KD_INFO:
    print(f"  KD_siglip training used {len(KD_INFO.get('teachers', []))} teacher runs of the same architecture "
          f"(cross-fitted on train only); inference is one network.")
_fm = seed_maps(FINAL_CONFIG)
_mean = float(np.mean(_fm))
V.check(f"final single-model mean beats Xenos et al. (38.52) [{FINAL_CONFIG}]", _mean > CFG.xenos_test_map,
        f"{_mean:.2f}", "> 38.52", fatal=False)
V.check(f"final single-model mean beats EmotiCon+CCIM (39.13) [{FINAL_CONFIG}]", _mean > CFG.ccim_test_map,
        f"{_mean:.2f}", "> 39.13", fatal=False)
V.check(f"final single-model mean reaches the paper target ({TARGET_MEAN})", _mean >= TARGET_MEAN,
        f"{_mean:.2f}", f">= {TARGET_MEAN}", fatal=False)
V.check("lower 95% bound of the final seed mean is above CCIM", tci(_fm)[0] > CFG.ccim_test_map,
        f"{tci(_fm)[0]:.2f}", "> 39.13", fatal=False)
if FINAL_CONFIG in SOUP_M:
    _sm = SOUP_M[FINAL_CONFIG]["uniform"]["test"]["mAP"]
    V.check("final soup (one network, separate row) beats CCIM", _sm > CFG.ccim_test_map,
            f"{_sm:.2f}", "> 39.13", fatal=False)
V.check("final seed ensemble (separate row) beats CCIM", ENS[FINAL_CONFIG]["test"]["mAP"] > CFG.ccim_test_map,
        f"{ENS[FINAL_CONFIG]['test']['mAP']:.2f}", "> 39.13", fatal=False)

# ──────────────────────────────────────────────────────────────────────────────
# 9.5  ABLATION (paired)
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.5  ABLATION — mean ± std over the same seeds; Δ vs the previous rung\n")
ABL, PAIRED = [], {}
for i, c in enumerate(AVAIL):
    x = seed_maps(c)
    v = np.array([RUNS[f"{c}_s{s}"]["best"]["val_tl_mAP"] for s in COMPLETE_SEEDS])
    row = {"config": c, "what": CONFIG_LABEL[c], "val_TL_mean": v.mean(), "test_mean": x.mean(),
           "test_std": x.std(ddof=1) if len(x) > 1 else 0.0,
           "test_soup": SOUP_M[c]["uniform"]["test"]["mAP"] if c in SOUP_M else np.nan,
           "test_ensemble": ENS[c]["test"]["mAP"],
           "macro_F1_mean": np.mean([METRICS[f"{c}_s{s}"]["test"]["macro_f1"] for s in COMPLETE_SEEDS])}
    if i > 0:
        p_ = AVAIL[i - 1]
        xp = seed_maps(p_)
        d_seed = x - xp
        t_p = float(sps.ttest_rel(x, xp).pvalue) if len(x) > 1 and np.std(d_seed) > 0 else float("nan")
        db = BOOT_ENS[c] - BOOT_ENS[p_]
        PAIRED[c] = {"vs": p_, "delta_seed_mean": float(d_seed.mean()), "paired_t_p": t_p,
                     "wins": int((d_seed > 0).sum()), "n": int(len(d_seed)),
                     "delta_ensemble": float(ENS[c]["test"]["mAP"] - ENS[p_]["test"]["mAP"]),
                     "boot_ci": [float(np.percentile(db, 2.5)), float(np.percentile(db, 97.5))],
                     "boot_p_le_0": float((db <= 0).mean())}
        row.update(delta=d_seed.mean(), p_t=t_p, wins=f"{int((d_seed > 0).sum())}/{len(d_seed)}")
    ABL.append(row)
ABL = pd.DataFrame(ABL)
print(ABL.to_string(index=False, float_format="{:.3f}".format))
print()
for c, pr in PAIRED.items():
    sig = (pr["paired_t_p"] < 0.05 if np.isfinite(pr["paired_t_p"]) else False) and pr["boot_ci"][0] > 0
    col = _G if sig and pr["delta_seed_mean"] > 0 else (_R if pr["delta_seed_mean"] < 0 else _Y)
    print(f"  {col}{c} vs {pr['vs']}: Δ seed-mean {pr['delta_seed_mean']:+.2f} (paired t p = {pr['paired_t_p']:.3f}, "
          f"better in {pr['wins']}/{pr['n']} seeds) | Δ ensemble {pr['delta_ensemble']:+.2f}, person-bootstrap "
          f"95% CI [{pr['boot_ci'][0]:+.2f}, {pr['boot_ci'][1]:+.2f}]"
          f"{'  → significant' if sig else '  → not significant'}{_0}")
if "KD_siglip" in AVAIL and "C4_siglip" in AVAIL:
    _kd_single = seed_maps("KD_siglip").mean()
    print(f"\n  compute-matched view: KD single model (mean) {_kd_single:.2f} vs C4 {len(COMPLETE_SEEDS)}-seed "
          f"ensemble {ENS['C4_siglip']['test']['mAP']:.2f} (KD needs 1 network at test time)")
    if "KD_siglip" in SOUP_M and "C4_siglip" in SOUP_M:
        _dbs = BOOT_SOUP["KD_siglip"] - BOOT_SOUP["C4_siglip"]
        print(f"  KD soup vs C4 soup: Δ {SOUP_M['KD_siglip']['uniform']['test']['mAP'] - SOUP_M['C4_siglip']['uniform']['test']['mAP']:+.2f}, "
              f"person-bootstrap 95% CI [{np.percentile(_dbs, 2.5):+.2f}, {np.percentile(_dbs, 97.5):+.2f}]")

# ──────────────────────────────────────────────────────────────────────────────
# 9.6  FULL METRIC PANEL
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n9.6  FULL METRIC PANEL — {FINAL_CONFIG}: seed mean, uniform soup, ensemble; thresholds from test-like val\n")
_panel_keys = [("mAP", "mAP"), ("AUC", "mean AUC"), ("micro_f1", "micro-F1"), ("macro_f1", "macro-F1"),
               ("macro_prec", "macro-precision"), ("macro_rec", "macro-recall"), ("jaccard", "Jaccard"),
               ("hamming", "Hamming loss ↓"), ("P@1", "P@1"), ("P@3", "P@3"), ("P@5", "P@5"),
               ("R@3", "R@3"), ("one_error", "one-error ↓"), ("ranking_loss", "ranking loss ↓"),
               ("coverage", "coverage ↓")]
_panel = {"seed mean": [np.mean([METRICS[n]["test"][k] for n in FINAL_RUNS]) for k, _ in _panel_keys],
          "seed std": [np.std([METRICS[n]["test"][k] for n in FINAL_RUNS]) for k, _ in _panel_keys]}
if FINAL_CONFIG in SOUP_M:
    _panel["soup"] = [SOUP_M[FINAL_CONFIG]["uniform"]["test"][k] for k, _ in _panel_keys]
_panel["ensemble"] = [ENS[FINAL_CONFIG]["test"][k] for k, _ in _panel_keys]
PANEL = pd.DataFrame(_panel, index=[lab for _, lab in _panel_keys])
print(PANEL.to_string(float_format="{:.4f}".format))

# ──────────────────────────────────────────────────────────────────────────────
# 9.7  PER-CLASS ANALYSIS
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.7  PER-CLASS AP (test, seed mean) — and where self-distillation helps\n")
_ap = {c: np.nanmean([METRICS[f"{c}_s{s}"]["test"]["per_class_ap"] for s in COMPLETE_SEEDS], 0) * 100 for c in AVAIL}
_apv = {c: np.nanmean([METRICS[f"{c}_s{s}"]["val_tl"]["per_class_ap"] for s in COMPLETE_SEEDS], 0) * 100
        for c in AVAIL}
PER_CLASS = pd.DataFrame({"emotion": EMOTION_LABELS, "test_prev": TEST_LABELS.mean(0), "rho": PROTOCOL_RHO,
                          **{f"AP {c}": _ap[c] for c in AVAIL}})
SPEARMAN = {}
if "KD_siglip" in AVAIL and "C4_siglip" in AVAIL:
    PER_CLASS["KD gain val-TL"] = _apv["KD_siglip"] - _apv["C4_siglip"]
    PER_CLASS["KD gain test"] = _ap["KD_siglip"] - _ap["C4_siglip"]
    for k, col in (("val", "KD gain val-TL"), ("test", "KD gain test")):
        g = PER_CLASS[col].to_numpy()
        ok = np.isfinite(g)
        SPEARMAN[k] = float(sps.spearmanr(PROTOCOL_RHO[ok], g[ok]).correlation) if ok.sum() > 2 else float("nan")
if "C4_siglip" in AVAIL and "C1_person" in AVAIL:
    PER_CLASS["SigLIP gain test"] = _ap["C4_siglip"] - _ap["C1_person"]
print(PER_CLASS.sort_values("rho", ascending=False).to_string(index=False, float_format="{:.3f}".format))
if SPEARMAN:
    V.note("Spearman(rho, KD per-class gain)", f"val-TL {SPEARMAN['val']:+.3f}   test {SPEARMAN['test']:+.3f}   "
                                               "(positive = gains follow the protocol gap)")
_weak = PER_CLASS.nsmallest(8, f"AP {FINAL_CONFIG}")[["emotion", f"AP {FINAL_CONFIG}"]]
V.note("weakest classes", ", ".join(f"{r[0]} {r[1]:.1f}" for r in _weak.itertuples(index=False)))

# ──────────────────────────────────────────────────────────────────────────────
# 9.8  VAL → TEST CONSISTENCY
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.8  VAL → TEST CONSISTENCY\n")
_vt = pd.DataFrame([{"run": n, "val_TL_mAP": RUNS[n]["best"]["val_tl_mAP"], "test_mAP": METRICS[n]["test"]["mAP"]}
                    for n in _USE])
_vt["test/val"] = _vt["test_mAP"] / _vt["val_TL_mAP"]
if len(_vt) >= 3:
    V.note("corr(val-TL, test) across all runs", f"{np.corrcoef(_vt['val_TL_mAP'], _vt['test_mAP'])[0, 1]:+.3f}")
    _cm = pd.DataFrame({"val": pd.Series(VAL_MEAN), "test": pd.Series({c: seed_maps(c).mean() for c in AVAIL})})
    V.note("config ranking by val mean vs by test mean",
           f"{list(_cm['val'].sort_values(ascending=False).index)} vs {list(_cm['test'].sort_values(ascending=False).index)}")
V.note("test / val-TL ratio", f"{_vt['test/val'].mean():.3f} ± {_vt['test/val'].std():.3f}")

# ──────────────────────────────────────────────────────────────────────────────
# 9.9  PERSIST
# ──────────────────────────────────────────────────────────────────────────────
print("\n9.9  PERSIST\n")
PHASE_A = {"phase": "A_no_augmentation_v7", "selection": SELECTION, "test_floor": FLOOR_TEST,
           "n_test": int(len(TEST_LABELS)), "provisional": _prov,
           "headline": {"config": FINAL_CONFIG, "seeds": COMPLETE_SEEDS, "test_mAP_per_seed": _fm.tolist(),
                        "mean": float(_fm.mean()), "std": float(_fm.std(ddof=1)) if len(_fm) > 1 else 0.0,
                        "ci95_mean": list(tci(_fm))},
           "median_val_seed": {"run": DISPLAY_RUN, **core.scalar_metrics(METRICS[DISPLAY_RUN]["test"])},
           "soups": {c: {k: {"members": SOUP_M[c][k]["members"], "val_tl_mAP": SOUP_M[c][k]["val_tl_mAP"],
                             **core.scalar_metrics(SOUP_M[c][k]["test"])} for k in ("greedy", "uniform")}
                     for c in SOUP_M},
           "ensembles": {c: {"runs": ENS[c]["runs"], **core.scalar_metrics(ENS[c]["test"]),
                             "ci95": list(_ci(BOOT_ENS[c]))} for c in AVAIL},
           "ablation": ABL.to_dict("records"), "paired": PAIRED, "spearman_rho_vs_kd_gain": SPEARMAN,
           "kd": KD_INFO, "teachers": TEACHERS_TABLE.to_dict("records") if len(TEACHERS_TABLE) else [],
           "published": {"xenos_2024": CFG.xenos_test_map, "emoticon_ccim_2023": CFG.ccim_test_map},
           "all_runs": {n: {"val_tl_mAP": RUNS[n]["best"]["val_tl_mAP"], **core.scalar_metrics(METRICS[n]["test"])}
                        for n in _USE},
           "test_reads": _n_access, "core_version": core.CORE_VERSION,
           "written": time.strftime("%Y-%m-%d %H:%M:%S")}
core.write_json(RESULTS_DIR / "final_results.json", PHASE_A)
SEL_TABLE.to_csv(RESULTS_DIR / "runs.csv", index=False)
ABL.to_csv(RESULTS_DIR / "ablation.csv", index=False)
PANEL.to_csv(RESULTS_DIR / "metric_panel.csv")
PER_CLASS.to_csv(RESULTS_DIR / "per_class_ap.csv", index=False)
_vt.to_csv(RESULTS_DIR / "val_vs_test.csv", index=False)
if len(TEACHERS_TABLE):
    TEACHERS_TABLE.to_csv(RESULTS_DIR / "teachers.csv", index=False)
np.savez(RESULTS_DIR / "final_scores.npz", display_test=SCORES[DISPLAY_RUN]["test_scores"],
         ensemble_test=ENS[FINAL_CONFIG]["test_scores"],
         soup_test=(SOUP_M[FINAL_CONFIG]["uniform"]["test_scores"] if FINAL_CONFIG in SOUP_M
                    else np.zeros((0, NUM_CLASSES))))
V.check("results written", (RESULTS_DIR / "final_results.json").exists(), str(RESULTS_DIR), "written")
HEADLINE_RUN = DISPLAY_RUN
HEADLINE_TEST_SCORES = SCORES[DISPLAY_RUN]["test_scores"]
ENSEMBLE_TEST_SCORES = ENS[FINAL_CONFIG]["test_scores"]
ENSEMBLE_RUNS = ENS[FINAL_CONFIG]["runs"]
WINNER = FINAL_CONFIG
SOUP_TEST_SCORES = SOUP_M[FINAL_CONFIG]["uniform"]["test_scores"] if FINAL_CONFIG in SOUP_M else None
free_memory()
BLOCK_09_OK = V.finish()

In [ ]:
# Block 10 — Qualitative Results: 6 Test Samples (Phase A v7: NO augmentation)
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES : Block 09 in this session
#
# Models shown (fixed by Block 09's pre-declared rule, not by test results):
#   * the final configuration's MEDIAN-VAL seed (a single model),
#   * the final configuration's soup (one network = average of its seeds; separate row), and
#   * the final configuration's seed ensemble (a separate row in Block 09).
#
# Six test persons per model, chosen by rule, not by eye: the persons whose
# per-sample AP sits at the 1st, 15th, 40th, 60th, 85th and 99th percentile of
# the model's test distribution — from clear failures to clear successes.
# Shown live below and saved to /kaggle/working/figures/phaseA_v7_no_augmentation/.
#
# In each panel: the image with the person's red box, the ground-truth labels
# (3-annotator union), and the model's top-5 emotions with probabilities
# (green ✓ = in the ground truth, red ✗ = not).
# ══════════════════════════════════════════════════════════════════════════════

import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

header("BLOCK 10 — Qualitative Results: 6 Test Samples (Phase A v7, no augmentation)  [v7]")
V = BlockVerdict(10, "Qualitative")
QUAL_DIR = CFG.viz_dir / "phaseA_v7_no_augmentation"
QUAL_DIR.mkdir(parents=True, exist_ok=True)
QUANTILES = [1, 15, 40, 60, 85, 99]


def per_sample_ap(scores, labels):
    out = np.full(len(labels), np.nan)
    for i in range(len(labels)):
        y = labels[i]
        if y.sum() == 0:
            continue
        o = np.argsort(-scores[i], kind="mergesort")
        hit = y[o]
        ranks = np.arange(1, len(o) + 1)
        out[i] = float(((np.cumsum(hit) / ranks) * hit).sum() / hit.sum())
    return out


def pick(ap):
    chosen = []
    for q in QUANTILES:
        target = np.nanpercentile(ap, q)
        order = np.argsort(np.abs(np.nan_to_num(ap, nan=9.0) - target), kind="mergesort")
        i = next(int(j) for j in order if int(j) not in chosen)
        chosen.append(i)
    return chosen


def panel(tag, title, scores, thresholds):
    ap = per_sample_ap(scores, TEST_LABELS)
    idx = pick(ap)
    fig, axes = plt.subplots(2, 3, figsize=(18, 14))
    rows = []
    for k, (ax, i) in enumerate(zip(axes.flat, idx)):
        r = df_test.iloc[i]
        try:
            img = Image.open(r.img_path).convert("RGB")
            ok = True
        except Exception:
            img, ok = Image.new("RGB", (320, 240), (128, 128, 128)), False
        W, H = img.size
        s = min(1.0, 900.0 / max(W, H))
        if s < 1.0:
            img = img.resize((max(1, int(W * s)), max(1, int(H * s))))
        ax.imshow(img)
        ax.add_patch(mpatches.Rectangle((r.xmin * s, r.ymin * s), (r.xmax - r.xmin) * s, (r.ymax - r.ymin) * s,
                                        fill=False, edgecolor="red", linewidth=2.5))
        ax.set_xticks([]); ax.set_yticks([])
        gt = [EMOTION_LABELS[c] for c in np.where(TEST_LABELS[i] > 0)[0]]
        top = np.argsort(-scores[i])[:5]
        ax.set_title(f"test #{i}   per-sample AP {ap[i]:.2f}   (percentile {QUANTILES[k]})",
                     fontsize=11, fontweight="bold")
        # captions in points below the image, so spacing is identical for every aspect ratio
        lines = [(("GT: " if j == 0 else "    ") + l, "#1f3b73")
                 for j, l in enumerate(textwrap.wrap(", ".join(gt), 62))] or [("GT: -", "#1f3b73")]
        for c in top:
            good = TEST_LABELS[i, c] > 0
            lines.append((f"{'✓' if good else '✗'} {EMOTION_LABELS[c]:<16s} {scores[i, c]:.2f}"
                          + ("  ▲" if scores[i, c] >= thresholds[c] else ""),
                          "#138a36" if good else "#c0392b"))
        for j, (txt, col) in enumerate(lines):
            ax.annotate(txt, xy=(0, 0), xycoords="axes fraction", xytext=(0, -6 - 13 * j),
                        textcoords="offset points", fontsize=9.5, va="top", ha="left", color=col,
                        family="DejaVu Sans Mono" if j >= len(lines) - len(top) else None)
        rows.append({"model": tag, "percentile": QUANTILES[k], "test_index": i, "image": str(r.img_path),
                     "box": [int(r.xmin), int(r.ymin), int(r.xmax), int(r.ymax)], "sample_AP": float(ap[i]),
                     "ground_truth": gt,
                     "top5": [(EMOTION_LABELS[c], round(float(scores[i, c]), 4)) for c in top],
                     "image_loaded": ok})
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.995)
    fig.text(0.5, 0.005, "✓ in ground truth   ✗ not in ground truth   ▲ above the class threshold tuned on "
             "test-like val", ha="center", fontsize=10, color="#555555")
    fig.tight_layout(rect=(0, 0.03, 1, 0.97), h_pad=9.0)
    png = QUAL_DIR / f"qualitative_{tag}.png"
    fig.savefig(png, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    pd.DataFrame(rows).to_csv(QUAL_DIR / f"qualitative_{tag}.csv", index=False)
    return png, rows, ap


# ──────────────────────────────────────────────────────────────────────────────
# 10.1  HEADLINE SINGLE MODEL
# ──────────────────────────────────────────────────────────────────────────────
print("10.1  SINGLE MODEL (final configuration, median-val seed)\n")
_th = np.asarray(EVALS[HEADLINE_RUN]["thresholds"], np.float32)
_m = METRICS[HEADLINE_RUN]["test"]["mAP"]
_png, _rows, _ap = panel(HEADLINE_RUN, f"Phase A v7 (no augmentation) — single model {HEADLINE_RUN} "
                                       f"(median-val seed of {WINNER}) — test mAP {_m:.2f}",
                         HEADLINE_TEST_SCORES, _th)
V.check("6 samples chosen by percentile rule", len(_rows) == 6 and len({r["test_index"] for r in _rows}) == 6,
        len(_rows), 6)
V.check("all 6 images loaded", all(r["image_loaded"] for r in _rows), sum(r["image_loaded"] for r in _rows), 6)
V.check("figure saved", _png.exists(), _png.name, "png")
V.note("median per-sample AP (test)", f"{np.nanmedian(_ap):.3f}")
QUAL_OUT = {"single": [str(_png)]}

# ──────────────────────────────────────────────────────────────────────────────
# 10.2  SOUP (one network = average of the seeds; separate row of Block 09)
# ──────────────────────────────────────────────────────────────────────────────
print("\n10.2  SOUP\n")
if globals().get("SOUP_TEST_SCORES") is not None and WINNER in SOUP_M:
    _g = SOUP_M[WINNER]["uniform"]
    _png3, _rows3, _ = panel("soup", f"Phase A v7 (no augmentation) — soup of {WINNER} (1 network = average of "
                                     f"{len(_g['members'])} seeds, separate row) — test mAP {_g['test']['mAP']:.2f}",
                             SOUP_TEST_SCORES, np.asarray(_g["thresholds"], np.float32))
    V.check("soup figure saved", _png3.exists(), _png3.name, "png")
    QUAL_OUT["soup"] = [str(_png3)]
else:
    V.note("soup", "not available yet (needs all seeds of the final configuration)")

# ──────────────────────────────────────────────────────────────────────────────
# 10.3  SEED ENSEMBLE (separate row of Block 09)
# ──────────────────────────────────────────────────────────────────────────────
print("\n10.3  SEED ENSEMBLE\n")
if ENSEMBLE_TEST_SCORES is not None and len(ENSEMBLE_RUNS) >= 2:
    _m = ENS[WINNER]["test"]["mAP"]
    _png2, _rows2, _ = panel("ensemble", f"Phase A v7 (no augmentation) — {len(ENSEMBLE_RUNS)}-seed ensemble of "
                                         f"{WINNER} (separate row) — test mAP {_m:.2f}", ENSEMBLE_TEST_SCORES,
                             np.asarray(ENS[WINNER]["thresholds"], np.float32))
    V.check("ensemble figure saved", _png2.exists(), _png2.name, "png")
    QUAL_OUT["ensemble"] = [str(_png2)]
else:
    V.note("ensemble", "not available (fewer than 2 finished seeds)")

core.write_json(QUAL_DIR / "index.json", {"phase": "A_no_augmentation_v7", "figures": QUAL_OUT,
                                          "percentiles": QUANTILES, "headline": HEADLINE_RUN})
print(f"\n  {_B}Saved to {QUAL_DIR}{_0}")
for _f in sorted(QUAL_DIR.glob("*")):
    print(f"    {_f.name}")
BLOCK_10_OK = V.finish()